In [ ]:
!pip -q install -U pandas pillow numpy tqdm transformers accelerate
!pip -q install git+https://github.com/openai/CLIP.git

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, json, hashlib, time, gc, platform
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

PROJECT_ROOT = Path('/content/drive/MyDrive/TwinCheck')
GENIMAGE_ROOT = PROJECT_ROOT / 'data/processed/vision/genimage'
F1_MANIFEST_DIR = GENIMAGE_ROOT
F2_OUT = PROJECT_ROOT / 'features/vision/genimage_f1_normalized_v2'
F2_OUT.mkdir(parents=True, exist_ok=True)

SPLIT_MANIFESTS = {
    'train': F1_MANIFEST_DIR / 'genimage_train_manifest.csv',
    'validation': F1_MANIFEST_DIR / 'genimage_validation_manifest.csv',
    'test': F1_MANIFEST_DIR / 'genimage_test_manifest.csv',
}

print('Project:', PROJECT_ROOT)
print('F1 manifest folder:', F1_MANIFEST_DIR)
print('Revised F2 output:', F2_OUT)
print('Project folder exists:', PROJECT_ROOT.exists())

In [ ]:
from zipfile import ZipFile

missing_manifest_files = [p for p in SPLIT_MANIFESTS.values() if not p.is_file()]
if missing_manifest_files:
    print('F1 split manifests missing from:', F1_MANIFEST_DIR)
    from google.colab import files
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError('No F1 results ZIP uploaded; cannot continue.')
    uploaded_name = next(iter(uploaded))
    zip_path = Path('/content') / uploaded_name
    with ZipFile(zip_path) as zf:
        targets = [
            n for n in zf.namelist()
            if n.endswith('.csv') and 'GenImage_Artifacts/' in n and 'manifest' in Path(n).name
        ]
        if not targets:
            raise RuntimeError('ZIP contains no GenImage manifest CSVs.')
        F1_MANIFEST_DIR.mkdir(parents=True, exist_ok=True)
        for member in targets:
            dest = F1_MANIFEST_DIR / Path(member).name
            dest.write_bytes(zf.read(member))
            print('Extracted manifest:', dest)

missing_manifest_files = [p for p in SPLIT_MANIFESTS.values() if not p.is_file()]
if missing_manifest_files:
    raise FileNotFoundError('Required split manifests still missing: ' + ', '.join(map(str, missing_manifest_files)))
print('All three F1 split manifests found.')

In [ ]:
EXPECTED_COUNTS = {
    'train': {'real': 4000, 'fake': 5000},
    'validation': {'real': 1000, 'fake': 1000},
    'test': {'real': 1000, 'fake': 1000},
}
REQUIRED = {
    'md5', 'label', 'class_name', 'generator', 'split',
    'normalized_export_path', 'normalized_format',
    'normalized_width', 'normalized_height', 'normalized_mode', 'normalized_md5'
}
frames=[]
for split, path in SPLIT_MANIFESTS.items():
    df=pd.read_csv(path)
    missing=REQUIRED-set(df.columns)
    if missing: raise ValueError(f'{split} manifest missing columns: {sorted(missing)}')
    if not (df['split'].astype(str)==split).all():
        raise ValueError(f'Split field mismatch in {path}')
    frames.append(df)
manifest=pd.concat(frames, ignore_index=True)

assert len(manifest)==13000, f'Expected 13,000 rows, got {len(manifest)}'
assert manifest['md5'].nunique()==13000, 'Original source MD5s are not unique in selected set'
assert manifest['normalized_export_path'].notna().all()
assert manifest['normalized_md5'].notna().all()
assert manifest['class_name'].isin(['real','fake']).all()

observed_counts = {}
for split, expected in EXPECTED_COUNTS.items():
    sub=manifest[manifest.split==split]
    counts={str(k):int(v) for k,v in sub.class_name.value_counts().to_dict().items()}
    observed_counts[split]=counts
    assert counts==expected, f'{split} class distribution mismatch: {counts}'

trainval_fake=set(manifest[manifest.split.isin(['train','validation']) & (manifest.class_name=='fake')]['generator'].astype(str))
test_fake=set(manifest[(manifest.split=='test') & (manifest.class_name=='fake')]['generator'].astype(str))
print('Fake generators in train/validation:', trainval_fake)
print('Fake generators in test:', test_fake)
assert trainval_fake=={'BigGAN'}, f'Unexpected train/validation fake generator(s): {trainval_fake}'
assert test_fake=={'Stable Diffusion v1.4'}, f'Unexpected test fake generator(s): {test_fake}'

standardized_metadata_ok=(
    (manifest.normalized_format.astype(str).str.upper()=='JPEG').all() and
    (manifest.normalized_width==256).all() and
    (manifest.normalized_height==256).all() and
    (manifest.normalized_mode.astype(str)=='RGB').all()
)
assert standardized_metadata_ok, 'Manifest metadata does not show all normalized inputs as JPEG 256x256 RGB.'

print('Manifest row counts:', observed_counts)
print('Total rows:',len(manifest),'| unique normalized paths:',manifest.normalized_export_path.nunique())
print('Normalized metadata standardization:', standardized_metadata_ok)
print('F1 manifest audit: PASS')

In [ ]:
def md5_file(path, chunk_size=1024*1024):
    h=hashlib.md5()
    with open(path,'rb') as f:
        for chunk in iter(lambda:f.read(chunk_size), b''):
            h.update(chunk)
    return h.hexdigest()

missing_paths=[]
metadata_errors=[]
hash_mismatches=[]
actual_formats=[]
actual_sizes=[]
actual_modes=[]
for row in tqdm(manifest.itertuples(index=False), total=len(manifest), desc='F1 normalized file audit'):
    path=Path(row.normalized_export_path)
    if not path.is_file():
        missing_paths.append(str(path)); continue
    try:
        with Image.open(path) as im:
            actual_formats.append((im.format or '').upper())
            actual_sizes.append(im.size)
            actual_modes.append(im.mode)
            im.verify()
        with Image.open(path) as im:
            if im.size!=(256,256) or im.mode!='RGB' or (im.format or '').upper()!='JPEG':
                metadata_errors.append((str(path), im.format, im.size, im.mode))
        observed_md5=md5_file(path)
        if observed_md5 != str(row.normalized_md5):
            hash_mismatches.append((str(path), str(row.normalized_md5), observed_md5))
    except Exception as e:
        metadata_errors.append((str(path), repr(e)))

print('Missing normalized files:',len(missing_paths))
print('Format/dimension/mode/read errors:',len(metadata_errors))
print('Normalized MD5 mismatches:',len(hash_mismatches))
if missing_paths[:5]: print('Missing examples:',missing_paths[:5])
if metadata_errors[:5]: print('Error examples:',metadata_errors[:5])
if hash_mismatches[:5]: print('Hash mismatch examples:',hash_mismatches[:5])
assert not missing_paths and not metadata_errors and not hash_mismatches, 'F1 normalized image audit failed; stop before model inference.'
print('All 13,000 normalized images verified: PASS')

In [ ]:
source_format_summary=(manifest.groupby(['split','class_name','original_format'], dropna=False).size().reset_index(name='count'))
source_dimension_summary=(manifest.groupby(['split','class_name','original_width','original_height'], dropna=False).size().reset_index(name='count'))
normalized_summary=(manifest.groupby(['split','class_name','normalized_format','normalized_width','normalized_height','normalized_mode'],dropna=False).size().reset_index(name='count'))
source_format_summary.to_csv(F2_OUT/'source_format_by_split_class.csv',index=False)
source_dimension_summary.to_csv(F2_OUT/'source_dimensions_by_split_class.csv',index=False)
normalized_summary.to_csv(F2_OUT/'normalized_spec_by_split_class.csv',index=False)

print('Original formats (descriptive):')
display(source_format_summary)
print('Normalized specifications:')
display(normalized_summary)
unique_specs=manifest[['normalized_format','normalized_width','normalized_height','normalized_mode']].drop_duplicates()
assert len(unique_specs)==1 and tuple(unique_specs.iloc[0])==('JPEG',256,256,'RGB')
print('Shortcut mitigation input specification: PASS (limited claim)')

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import hashlib
import json

PROJECT_ROOT = Path("/content/drive/MyDrive/TwinCheck")

GENIMAGE_MANIFEST_PATH = (
    PROJECT_ROOT
    / "data/processed/vision/genimage/genimage_manifest.csv"
)

F2_OUT = (
    PROJECT_ROOT
    / "features/vision/genimage"
)

F2_OUT.mkdir(parents=True, exist_ok=True)

print("=" * 80)
print("GENIMAGE STEP 7 — DETERMINISTIC EMBEDDING MANIFEST")
print("=" * 80)

if not GENIMAGE_MANIFEST_PATH.exists():
    raise FileNotFoundError(
        f"GenImage manifest not found:\n{GENIMAGE_MANIFEST_PATH}"
    )

genimage_manifest = pd.read_csv(GENIMAGE_MANIFEST_PATH)

print("\nLoaded manifest:")
print(GENIMAGE_MANIFEST_PATH)

print("\nRows:", len(genimage_manifest))
print("Columns:", genimage_manifest.columns.tolist())

REQUIRED_COLUMNS = [
    "split",
    "class_name",
    "generator",
    "normalized_export_path",
    "normalized_md5",
    "md5"
]

missing_columns = [
    col for col in REQUIRED_COLUMNS
    if col not in genimage_manifest.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required GenImage columns: {missing_columns}"
    )

print("\nRequired column check: PASS")

print("\nSplit distribution:")
print(genimage_manifest["split"].value_counts(dropna=False))

print("\nClass distribution:")
print(genimage_manifest["class_name"].value_counts(dropna=False))

print("\nSplit/class distribution:")
print(
    genimage_manifest
    .groupby(["split", "class_name"], dropna=False)
    .size()
)

EXPECTED_SPLITS = {"train", "validation", "test"}
EXPECTED_CLASSES = {"real", "fake"}

actual_splits = set(genimage_manifest["split"].dropna().unique())
actual_classes = set(genimage_manifest["class_name"].dropna().unique())

if actual_splits != EXPECTED_SPLITS:
    raise ValueError(
        f"Unexpected splits.\n"
        f"Expected: {EXPECTED_SPLITS}\n"
        f"Actual: {actual_splits}"
    )

if actual_classes != EXPECTED_CLASSES:
    raise ValueError(
        f"Unexpected classes.\n"
        f"Expected: {EXPECTED_CLASSES}\n"
        f"Actual: {actual_classes}"
    )

print("\nSplit and class names: PASS")

missing_summary = (
    genimage_manifest[REQUIRED_COLUMNS]
    .isna()
    .sum()
)

print("\nMissing values:")
print(missing_summary)

if missing_summary.sum() > 0:
    raise ValueError(
        "Missing values detected in required columns. "
        "Resolve them before embedding."
    )

print("\nMissing required values: PASS")

EXPECTED_COUNTS = {
    ("train", "real"): 4000,
    ("train", "fake"): 5000,
    ("validation", "real"): 1000,
    ("validation", "fake"): 1000,
    ("test", "real"): 1000,
    ("test", "fake"): 1000
}

actual_counts = (
    genimage_manifest
    .groupby(["split", "class_name"])
    .size()
    .to_dict()
)

print("\nExpected vs actual counts:")

count_errors = []

for key, expected in EXPECTED_COUNTS.items():

    actual = actual_counts.get(key, 0)

    print(f"{key}: expected={expected:,}, actual={actual:,}")

    if actual != expected:
        count_errors.append(
            {
                "split": key[0],
                "class_name": key[1],
                "expected": expected,
                "actual": actual
            }
        )

if count_errors:
    print("\nCOUNT MISMATCHES:")
    print(count_errors)

    raise ValueError(
        "GenImage split/class counts do not match expected 13K distribution."
    )

if len(genimage_manifest) != 13000:
    raise ValueError(
        f"Expected 13000 rows, found {len(genimage_manifest)}"
    )

print("\n13,000-row distribution: PASS")

print("\nChecking normalized image files...")

normalized_paths = genimage_manifest["normalized_export_path"]

path_exists = normalized_paths.map(
    lambda p: Path(str(p)).is_file()
)

missing_paths = genimage_manifest.loc[
    ~path_exists,
    ["split", "class_name", "normalized_export_path"]
]

print("Existing normalized files:", int(path_exists.sum()))
print("Missing normalized files:", len(missing_paths))

if len(missing_paths) > 0:
    display(missing_paths.head(20))

    raise FileNotFoundError(
        "Some normalized GenImage files are missing."
    )

print("\nNormalized files existence: PASS")

for hash_col in ["md5", "normalized_md5"]:

    if genimage_manifest[hash_col].astype(str).str.strip().eq("").any():
        raise ValueError(f"Empty hashes found in {hash_col}")

print("\nHash fields non-empty: PASS")

duplicate_normalized_hashes = (
    genimage_manifest["normalized_md5"]
    .duplicated(keep=False)
)

duplicate_count = int(duplicate_normalized_hashes.sum())

print(
    "\nRows involved in duplicate normalized MD5:",
    duplicate_count
)

if duplicate_count > 0:

    duplicate_df = genimage_manifest.loc[
        duplicate_normalized_hashes,
        [
            "split",
            "class_name",
            "generator",
            "normalized_export_path",
            "normalized_md5"
        ]
    ]

    display(duplicate_df.head(20))

    print(
        "WARNING: Duplicate normalized hashes exist. "
        "They will be reported but not automatically removed."
    )

SPLIT_ORDER = {
    "train": 0,
    "validation": 1,
    "test": 2
}

CLASS_ORDER = {
    "real": 0,
    "fake": 1
}

genimage_manifest["split_order"] = (
    genimage_manifest["split"].map(SPLIT_ORDER)
)

genimage_manifest["class_order"] = (
    genimage_manifest["class_name"].map(CLASS_ORDER)
)

if genimage_manifest["split_order"].isna().any():
    raise ValueError("Unknown split values found.")

if genimage_manifest["class_order"].isna().any():
    raise ValueError("Unknown class values found.")

genimage_manifest = (
    genimage_manifest
    .sort_values(
        ["split_order", "class_order", "md5"],
        ascending=True
    )
    .reset_index(drop=True)
)

genimage_manifest = genimage_manifest.drop(
    columns=["split_order", "class_order"]
)

genimage_manifest.insert(
    0,
    "embedding_row",
    np.arange(
        len(genimage_manifest),
        dtype=np.int64
    )
)

expected_rows = np.arange(
    len(genimage_manifest),
    dtype=np.int64
)

if not np.array_equal(
    genimage_manifest["embedding_row"].to_numpy(),
    expected_rows
):
    raise ValueError("Embedding row index is not contiguous.")

print("\nEmbedding row numbering: PASS")

order_payload = "\n".join(
    genimage_manifest["normalized_md5"]
    .astype(str)
    .tolist()
).encode("utf-8")

INPUT_ORDER_SHA256 = hashlib.sha256(
    order_payload
).hexdigest()

OUTPUT_MANIFEST = (
    F2_OUT
    / "f1_genimage_13000_embedding_input_manifest.csv"
)

genimage_manifest.to_csv(
    OUTPUT_MANIFEST,
    index=False
)

ORDER_METADATA = {
    "dataset": "GenImage",
    "total_rows": int(len(genimage_manifest)),
    "ordering": [
        "split",
        "class_name",
        "md5"
    ],
    "split_order": SPLIT_ORDER,
    "class_order": CLASS_ORDER,
    "fingerprint_method": (
        "SHA256 of newline-joined ordered normalized_md5 values"
    ),
    "input_order_sha256": INPUT_ORDER_SHA256,
    "manifest_path": str(OUTPUT_MANIFEST),
    "normalized_duplicate_hash_rows": duplicate_count
}

METADATA_PATH = (
    F2_OUT
    / "genimage_embedding_order_metadata.json"
)

with open(
    METADATA_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        ORDER_METADATA,
        f,
        indent=2,
        ensure_ascii=False
    )

print("\n" + "=" * 80)
print("GENIMAGE STEP 7 SUMMARY")
print("=" * 80)

print("Rows:", len(genimage_manifest))
print("Fingerprint:", INPUT_ORDER_SHA256)

print("\nFinal distribution:")
print(
    genimage_manifest
    .groupby(["split", "class_name"])
    .size()
)

print("\nManifest saved:")
print(OUTPUT_MANIFEST)

print("\nMetadata saved:")
print(METADATA_PATH)

print("\nPreview:")
display(
    genimage_manifest[
        [
            "embedding_row",
            "split",
            "class_name",
            "generator",
            "normalized_export_path",
            "normalized_md5"
        ]
    ].head(10)
)

print("\n" + "=" * 80)
print("STEP 7 COMPLETED — GENIMAGE MANIFEST")
print("=" * 80)

In [ ]:
import os
import gc
import json
import time
import numpy as np
import pandas as pd
import torch

from PIL import Image
from tqdm.auto import tqdm
from transformers import AutoProcessor, AutoModel

PROJECT_ROOT = "/content/drive/MyDrive/TwinCheck"

INPUT_MANIFEST_PATH = os.path.join(
    PROJECT_ROOT,
    "features/vision/genimage/"
    "f1_genimage_13000_embedding_input_manifest.csv"
)

ORDER_METADATA_PATH = os.path.join(
    PROJECT_ROOT,
    "features/vision/genimage/"
    "genimage_embedding_order_metadata.json"
)

OUTPUT_ROOT = os.path.join(
    PROJECT_ROOT,
    "features/vision/genimage"
)

SIGLIP_MODEL_NAME = (
    "google/siglip2-so400m-patch14-384"
)

CLIP_MODEL_NAME = (
    "openai/clip-vit-large-patch14"
)

EXPECTED_ROWS = 13000

SIGLIP_DIM = 1152
CLIP_DIM = 768

SIGLIP_BATCH_SIZE = 4
CLIP_BATCH_SIZE = 8

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("=" * 80)
print("GENIMAGE STEP 10 — FULL EMBEDDING EXTRACTION")
print("=" * 80)

print("Device:", DEVICE)

if torch.cuda.is_available():

    print("GPU:", torch.cuda.get_device_name(0))

    print(
        "GPU memory:",
        round(
            torch.cuda.get_device_properties(0).total_memory
            / (1024**3),
            2
        ),
        "GB"
    )

genimage_step10_manifest = pd.read_csv(
    INPUT_MANIFEST_PATH
)

with open(
    ORDER_METADATA_PATH,
    "r"
) as f:

    genimage_order_metadata = json.load(f)

print("\nManifest loaded.")

print(
    "Number of rows:",
    len(genimage_step10_manifest)
)

required_columns = [
    "embedding_row",
    "md5",
    "label",
    "class_name",
    "split",
    "normalized_export_path",
    "normalized_md5"
]

missing_columns = [
    column
    for column in required_columns
    if column not in genimage_step10_manifest.columns
]

assert not missing_columns, (
    f"Missing columns: {missing_columns}"
)

assert len(genimage_step10_manifest) == EXPECTED_ROWS

assert (
    genimage_step10_manifest["embedding_row"].tolist()
    == list(range(EXPECTED_ROWS))
)

assert (
    genimage_step10_manifest[
        "normalized_export_path"
    ].isna().sum()
    == 0
)

assert (
    genimage_step10_manifest[
        "normalized_md5"
    ].isna().sum()
    == 0
)

print("Manifest validation: PASS")

saved_fingerprint = (
    genimage_order_metadata.get(
        "input_order_sha256"
    )
)

assert saved_fingerprint is not None

EXPECTED_FINGERPRINT = (
    "fa180d44d1005122"
    "ee3b57cc47edecb9137abee953764b3d"
    "7d57bd32d5708282"
)

assert saved_fingerprint == EXPECTED_FINGERPRINT, (
    "Manifest fingerprint does not match "
    "the validated Step 7 fingerprint."
)

print(
    "Fingerprint verification: PASS"
)

print(
    "Fingerprint:",
    saved_fingerprint
)

genimage_step10_image_paths = (
    genimage_step10_manifest[
        "normalized_export_path"
    ].tolist()
)

missing_images = [
    path
    for path in genimage_step10_image_paths
    if not os.path.isfile(path)
]

assert len(missing_images) == 0, (
    f"{len(missing_images)} images are missing."
)

print(
    "All 13,000 normalized images exist: PASS"
)

SIGLIP_OUTPUT_DIR = os.path.join(
    OUTPUT_ROOT,
    "siglip2_so400m"
)

CLIP_OUTPUT_DIR = os.path.join(
    OUTPUT_ROOT,
    "clip_vitl14"
)

os.makedirs(
    SIGLIP_OUTPUT_DIR,
    exist_ok=True
)

os.makedirs(
    CLIP_OUTPUT_DIR,
    exist_ok=True
)

def extract_embedding_tensor(model_output):

    if isinstance(model_output, torch.Tensor):

        return model_output

    if (
        hasattr(model_output, "image_embeds")
        and model_output.image_embeds is not None
    ):

        return model_output.image_embeds

    if (
        hasattr(model_output, "pooler_output")
        and model_output.pooler_output is not None
    ):

        return model_output.pooler_output

    if (
        hasattr(model_output, "last_hidden_state")
        and model_output.last_hidden_state is not None
    ):

        return model_output.last_hidden_state[:, 0, :]

    raise ValueError(
        "Could not extract embedding tensor from model output."
    )

def run_full_embedding_extraction(
    model_name,
    model_key,
    output_dir,
    expected_dimension,
    initial_batch_size
):

    print("\n")
    print("=" * 80)
    print("STARTING:", model_key)
    print("=" * 80)

    final_raw_path = os.path.join(
        output_dir,
        "embeddings_raw.npy"
    )

    final_normalized_path = os.path.join(
        output_dir,
        "embeddings_normalized.npy"
    )

    final_metadata_path = os.path.join(
        output_dir,
        "extraction_metadata.json"
    )

    partial_raw_path = os.path.join(
        output_dir,
        "embeddings_raw_partial.npy"
    )

    partial_normalized_path = os.path.join(
        output_dir,
        "embeddings_normalized_partial.npy"
    )

    checkpoint_path = os.path.join(
        output_dir,
        "extraction_checkpoint.json"
    )

    if (
        os.path.exists(final_raw_path)
        and os.path.exists(final_normalized_path)
        and os.path.exists(final_metadata_path)
    ):

        with open(
            final_metadata_path,
            "r"
        ) as f:

            existing_metadata = json.load(f)

        if (
            existing_metadata.get(
                "input_order_sha256"
            ) == saved_fingerprint

            and existing_metadata.get(
                "num_images"
            ) == EXPECTED_ROWS

            and existing_metadata.get(
                "embedding_dimension"
            ) == expected_dimension

        ):

            existing_raw = np.load(
                final_raw_path,
                mmap_mode="r"
            )

            existing_normalized = np.load(
                final_normalized_path,
                mmap_mode="r"
            )

            if (
                existing_raw.shape
                == (
                    EXPECTED_ROWS,
                    expected_dimension
                )

                and existing_normalized.shape
                == (
                    EXPECTED_ROWS,
                    expected_dimension
                )
            ):

                print(
                    "Compatible completed embeddings found."
                )

                print(
                    "Skipping:",
                    model_key
                )

                del existing_raw
                del existing_normalized

                return existing_metadata

            del existing_raw
            del existing_normalized

        print(
            "Existing outputs are incompatible. "
            "A new extraction will be performed."
        )

    next_row = 0

    if os.path.exists(checkpoint_path):

        with open(
            checkpoint_path,
            "r"
        ) as f:

            checkpoint = json.load(f)

        checkpoint_is_valid = (

            checkpoint.get(
                "input_order_sha256"
            ) == saved_fingerprint

            and checkpoint.get(
                "model_name"
            ) == model_name

            and os.path.exists(
                partial_raw_path
            )

            and os.path.exists(
                partial_normalized_path
            )
        )

        if checkpoint_is_valid:

            next_row = int(
                checkpoint.get(
                    "next_row",
                    0
                )
            )

            assert 0 <= next_row <= EXPECTED_ROWS

            print(
                "Valid checkpoint found."
            )

            print(
                "Resume row:",
                next_row
            )

        else:

            print(
                "Checkpoint invalid or incompatible."
            )

            print(
                "Starting extraction from row 0."
            )

            next_row = 0

    if next_row == 0:

        raw_array = np.lib.format.open_memmap(
            partial_raw_path,
            mode="w+",
            dtype=np.float32,
            shape=(
                EXPECTED_ROWS,
                expected_dimension
            )
        )

        normalized_array = np.lib.format.open_memmap(
            partial_normalized_path,
            mode="w+",
            dtype=np.float32,
            shape=(
                EXPECTED_ROWS,
                expected_dimension
            )
        )

    else:

        raw_array = np.lib.format.open_memmap(
            partial_raw_path,
            mode="r+"
        )

        normalized_array = np.lib.format.open_memmap(
            partial_normalized_path,
            mode="r+"
        )

        assert raw_array.shape == (
            EXPECTED_ROWS,
            expected_dimension
        )

        assert normalized_array.shape == (
            EXPECTED_ROWS,
            expected_dimension
        )

    print(
        "\nLoading processor:",
        model_name
    )

    processor = AutoProcessor.from_pretrained(
        model_name
    )

    print(
        "Loading model:",
        model_name
    )

    model = AutoModel.from_pretrained(
        model_name
    )

    model = model.to(DEVICE)

    model.eval()

    current_batch_size = initial_batch_size

    start_time = time.time()

    progress_bar = tqdm(
        total=EXPECTED_ROWS,
        initial=next_row,
        desc=model_key
    )

    while next_row < EXPECTED_ROWS:

        batch_end = min(
            next_row + current_batch_size,
            EXPECTED_ROWS
        )

        batch_paths = (
            genimage_step10_image_paths[
                next_row:batch_end
            ]
        )

        batch_images = []

        for image_path in batch_paths:

            with Image.open(image_path) as img:

                batch_images.append(
                    img.convert("RGB").copy()
                )

        inputs = processor(
            images=batch_images,
            return_tensors="pt"
        )

        inputs = {
            key: value.to(DEVICE)
            for key, value in inputs.items()
        }

        try:

            with torch.no_grad():

                if DEVICE.type == "cuda":

                    with torch.autocast(
                        device_type="cuda",
                        dtype=torch.float16
                    ):

                        model_output = (
                            model.get_image_features(
                                **inputs
                            )
                        )

                else:

                    model_output = (
                        model.get_image_features(
                            **inputs
                        )
                    )

            features = extract_embedding_tensor(
                model_output
            )

            features = (
                features.float()
                .cpu()
                .numpy()
            )

        except torch.cuda.OutOfMemoryError:

            print(
                "\nCUDA OUT OF MEMORY."
            )

            print(
                "Reducing batch size:",
                current_batch_size,
                "->",
                max(
                    1,
                    current_batch_size // 2
                )
            )

            del batch_images

            del inputs

            if "model_output" in locals():

                del model_output

            if "features" in locals():

                del features

            torch.cuda.empty_cache()

            gc.collect()

            current_batch_size = max(
                1,
                current_batch_size // 2
            )

            continue

        assert features.ndim == 2

        assert features.shape == (
            len(batch_paths),
            expected_dimension
        ), (
            f"Unexpected feature shape: "
            f"{features.shape}"
        )

        assert np.isfinite(features).all(), (
            "NaN or Inf detected in embeddings."
        )

        norms = np.linalg.norm(
            features,
            axis=1,
            keepdims=True
        )

        assert np.all(norms > 0), (
            "Zero embedding detected."
        )

        normalized_features = (
            features / norms
        ).astype(
            np.float32
        )

        raw_array[
            next_row:batch_end
        ] = features.astype(
            np.float32
        )

        normalized_array[
            next_row:batch_end
        ] = normalized_features

        raw_array.flush()

        normalized_array.flush()

        next_row = batch_end

        checkpoint = {

            "model_name": model_name,

            "model_key": model_key,

            "next_row": next_row,

            "num_images": EXPECTED_ROWS,

            "embedding_dimension": expected_dimension,

            "input_order_sha256": saved_fingerprint,

            "status": "in_progress",

            "timestamp": time.strftime(
                "%Y-%m-%d %H:%M:%S"
            )
        }

        with open(
            checkpoint_path,
            "w"
        ) as f:

            json.dump(
                checkpoint,
                f,
                indent=2
            )

        progress_bar.update(
            len(batch_paths)
        )

        elapsed = (
            time.time() - start_time
        )

        print(
            f"\nProcessed {next_row:,}/{EXPECTED_ROWS:,}"
            f" | Batch size = {current_batch_size}"
            f" | Elapsed = {elapsed:.1f} sec"
        )

        del batch_images
        del inputs
        del model_output
        del features
        del normalized_features
        del norms

        gc.collect()

        if DEVICE.type == "cuda":

            torch.cuda.empty_cache()

    progress_bar.close()

    raw_array.flush()

    normalized_array.flush()

    del raw_array

    del normalized_array

    del model

    del processor

    gc.collect()

    if torch.cuda.is_available():

        torch.cuda.empty_cache()

    os.replace(
        partial_raw_path,
        final_raw_path
    )

    os.replace(
        partial_normalized_path,
        final_normalized_path
    )

    print(
        "\nPerforming final audit..."
    )

    raw_final = np.load(
        final_raw_path,
        mmap_mode="r"
    )

    normalized_final = np.load(
        final_normalized_path,
        mmap_mode="r"
    )

    assert raw_final.shape == (
        EXPECTED_ROWS,
        expected_dimension
    )

    assert normalized_final.shape == (
        EXPECTED_ROWS,
        expected_dimension
    )

    audit_chunk_size = 500

    for audit_start in range(
        0,
        EXPECTED_ROWS,
        audit_chunk_size
    ):

        audit_end = min(
            audit_start + audit_chunk_size,
            EXPECTED_ROWS
        )

        assert np.isfinite(
            raw_final[
                audit_start:audit_end
            ]
        ).all()

        assert np.isfinite(
            normalized_final[
                audit_start:audit_end
            ]
        ).all()

    final_norms = np.linalg.norm(
        normalized_final,
        axis=1
    )

    assert np.allclose(
        final_norms,
        1.0,
        atol=1e-5
    )

    elapsed_total = (
        time.time() - start_time
    )

    final_metadata = {

        "model_name": model_name,

        "model_key": model_key,

        "dataset": "GenImage",

        "dataset_version": "F1_normalized",

        "num_images": EXPECTED_ROWS,

        "embedding_dimension": expected_dimension,

        "raw_embedding_dtype": "float32",

        "normalized_embedding_dtype": "float32",

        "normalization": "L2",

        "input_manifest": INPUT_MANIFEST_PATH,

        "input_order_sha256": saved_fingerprint,

        "raw_embeddings_file": final_raw_path,

        "normalized_embeddings_file": final_normalized_path,

        "extraction_device": str(DEVICE),

        "gpu_name": (
            torch.cuda.get_device_name(0)
            if torch.cuda.is_available()
            else None
        ),

        "elapsed_seconds": elapsed_total,

        "status": "completed",

        "timestamp": time.strftime(
            "%Y-%m-%d %H:%M:%S"
        )
    }

    with open(
        final_metadata_path,
        "w"
    ) as f:

        json.dump(
            final_metadata,
            f,
            indent=2
        )

    checkpoint["status"] = "completed"

    checkpoint["next_row"] = EXPECTED_ROWS

    with open(
        checkpoint_path,
        "w"
    ) as f:

        json.dump(
            checkpoint,
            f,
            indent=2
        )

    del raw_final

    del normalized_final

    del final_norms

    gc.collect()

    print("\n")
    print("=" * 80)
    print(model_key, "— EXTRACTION COMPLETE")
    print("=" * 80)

    print(
        "Raw embeddings:",
        final_raw_path
    )

    print(
        "Normalized embeddings:",
        final_normalized_path
    )

    print(
        "Number of images:",
        EXPECTED_ROWS
    )

    print(
        "Embedding dimension:",
        expected_dimension
    )

    print(
        "Elapsed seconds:",
        round(
            elapsed_total,
            2
        )
    )

    print(
        "Final audit: PASS"
    )

    return final_metadata

siglip2_genimage_metadata = (
    run_full_embedding_extraction(

        model_name=SIGLIP_MODEL_NAME,

        model_key="SigLIP2",

        output_dir=SIGLIP_OUTPUT_DIR,

        expected_dimension=SIGLIP_DIM,

        initial_batch_size=SIGLIP_BATCH_SIZE
    )
)

clip_genimage_metadata = (
    run_full_embedding_extraction(

        model_name=CLIP_MODEL_NAME,

        model_key="CLIP_ViT_L14",

        output_dir=CLIP_OUTPUT_DIR,

        expected_dimension=CLIP_DIM,

        initial_batch_size=CLIP_BATCH_SIZE
    )
)

print("\n")
print("=" * 80)
print("GENIMAGE STEP 10 — FINAL SUMMARY")
print("=" * 80)

print("\nSIGLIP 2")

print(
    "Rows:",
    siglip2_genimage_metadata["num_images"]
)

print(
    "Dimension:",
    siglip2_genimage_metadata["embedding_dimension"]
)

print(
    "Status:",
    siglip2_genimage_metadata["status"]
)

print("\nCLIP")

print(
    "Rows:",
    clip_genimage_metadata["num_images"]
)

print(
    "Dimension:",
    clip_genimage_metadata["embedding_dimension"]
)

print(
    "Status:",
    clip_genimage_metadata["status"]
)

assert (
    siglip2_genimage_metadata["num_images"]
    == EXPECTED_ROWS
)

assert (
    clip_genimage_metadata["num_images"]
    == EXPECTED_ROWS
)

assert (
    siglip2_genimage_metadata[
        "input_order_sha256"
    ]
    == clip_genimage_metadata[
        "input_order_sha256"
    ]
)

print(
    "\nBoth models have matching input-order fingerprints."
)

print(
    "\nGENIMAGE STEP 10: COMPLETE"
)

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

ROOT = Path("/content/drive/MyDrive/TwinCheck")
GI = ROOT / "features/vision/genimage"

manifest_path = GI / "f1_genimage_13000_embedding_input_manifest.csv"
metadata_path = GI / "genimage_embedding_order_metadata.json"

FILES = {
    "SigLIP2 raw": GI / "siglip2_so400m/embeddings_raw.npy",
    "SigLIP2 normalized": GI / "siglip2_so400m/embeddings_normalized.npy",
    "CLIP raw": GI / "clip_vitl14/embeddings_raw.npy",
    "CLIP normalized": GI / "clip_vitl14/embeddings_normalized.npy",
}

EXPECTED_SHAPES = {
    "SigLIP2 raw": (13000, 1152),
    "SigLIP2 normalized": (13000, 1152),
    "CLIP raw": (13000, 768),
    "CLIP normalized": (13000, 768),
}

print("=" * 90)
print("F2 — GENIMAGE EMBEDDING VERIFICATION")
print("=" * 90)

manifest = pd.read_csv(manifest_path)

assert len(manifest) == 13000
assert manifest["embedding_row"].is_unique
assert np.array_equal(
    manifest["embedding_row"].to_numpy(),
    np.arange(13000)
)

print("\n1. Manifest row integrity: PASS")
print("Rows:", len(manifest))
print("Row order: 0 to 12,999")
print("Unique rows:", manifest["embedding_row"].nunique())

metadata = json.loads(metadata_path.read_text())

print("\n2. Embedding order metadata:")
print(json.dumps(metadata, indent=2))

arrays = {}

for name, path in FILES.items():

    print("\n" + "-" * 70)
    print(name)

    assert path.is_file(), f"MISSING FILE: {path}"

    arr = np.load(path, mmap_mode="r")

    assert arr.shape == EXPECTED_SHAPES[name], (
        f"Unexpected shape for {name}: {arr.shape}"
    )

    assert np.isfinite(arr).all(), (
        f"Non-finite values detected in {name}"
    )

    arrays[name] = arr

    print("File:", path)
    print("Shape:", arr.shape)
    print("Dtype:", arr.dtype)
    print("Size MB:", round(path.stat().st_size / 1e6, 2))
    print("Finite values: PASS")

    if "normalized" in name.lower():

        norms = np.linalg.norm(
            arr.astype(np.float32),
            axis=1
        )

        print("Norm mean:", float(norms.mean()))
        print("Norm minimum:", float(norms.min()))
        print("Norm maximum:", float(norms.max()))

        assert np.allclose(norms, 1.0, atol=1e-4), (
            f"Normalization failure in {name}"
        )

        print("Unit norm validation: PASS")

print("\n" + "=" * 90)
print("4. RAW vs NORMALIZED CONSISTENCY")
print("=" * 90)

for model in ["SigLIP2", "CLIP"]:

    raw = arrays[f"{model} raw"].astype(np.float32)
    normalized = arrays[f"{model} normalized"].astype(np.float32)

    calculated = raw / np.maximum(
        np.linalg.norm(raw, axis=1, keepdims=True),
        1e-12
    )

    cosine = np.sum(
        calculated * normalized,
        axis=1
    )

    print(f"\n{model}")
    print("Mean cosine:", float(cosine.mean()))
    print("Minimum cosine:", float(cosine.min()))

    assert np.allclose(
        calculated,
        normalized,
        atol=1e-4
    ), f"{model}: raw and normalized arrays inconsistent"

    print("Raw-normalized consistency: PASS")

print("\n" + "=" * 90)
print("GENIMAGE EMBEDDING VERIFICATION")
print("=" * 90)

print("13,000 rows: PASS")
print("Expected dimensions: PASS")
print("Finite values: PASS")
print("Normalized embeddings: PASS")
print("Raw-normalized consistency: PASS")

print("\nVERIFICATION COMPLETE")

In [ ]:
import os
import json
import hashlib
import zipfile
import warnings

from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime

import pandas as pd
from PIL import Image

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/TwinCheck"
)

DATASET_DIR = (
    PROJECT_ROOT
    / "data/raw/vision/verite"
)

AUDIT_DIR = (
    PROJECT_ROOT
    / "audit/verite"
)

AUDIT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

if not DATASET_DIR.exists():

    raise FileNotFoundError(
        f"VERITE directory not found:\n{DATASET_DIR}"
    )

print("=" * 80)
print("VERITE DATASET VERIFICATION — CORRECTED")
print("=" * 80)

print("Dataset:", DATASET_DIR)

all_files = [
    p for p in DATASET_DIR.rglob("*")
    if p.is_file()
]

inventory_records = []

for path in all_files:

    inventory_records.append({
        "relative_path": str(
            path.relative_to(DATASET_DIR)
        ),
        "absolute_path": str(path),
        "filename": path.name,
        "extension": path.suffix.lower(),
        "size_bytes": path.stat().st_size,
    })

inventory_df = pd.DataFrame(
    inventory_records
)

inventory_df.to_csv(
    AUDIT_DIR / "verite_file_inventory.csv",
    index=False
)

extension_counts = (
    inventory_df["extension"]
    .value_counts()
    .to_dict()
)

print("\nTotal files:", len(all_files))

print("\nExtension counts:")

for ext, count in extension_counts.items():

    print(f"{ext or '[no extension]'}: {count}")

IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".tif",
    ".tiff",
    ".webp",
    ".gif",
}

image_files = [
    p for p in all_files
    if p.suffix.lower() in IMAGE_EXTENSIONS
]

print("\nImage files:", len(image_files))

image_records = []

corrupt_images = []

dimension_counts = Counter()

mode_counts = Counter()

format_counts = Counter()

hash_to_paths = defaultdict(list)

hash_errors = []

filename_to_paths = defaultdict(list)

for path in image_files:

    filename_to_paths[
        path.name.lower()
    ].append(str(path))

    record = {
        "filename": path.name,
        "relative_path": str(
            path.relative_to(DATASET_DIR)
        ),
        "absolute_path": str(path),
        "extension": path.suffix.lower(),
        "size_bytes": path.stat().st_size,
        "readable": False,
        "image_format": None,
        "width": None,
        "height": None,
        "mode": None,
        "sha256": None,
        "error": None,
    }

    try:

        with Image.open(path) as img:

            img.verify()

        with Image.open(path) as img:

            img.load()

            record["readable"] = True

            record["image_format"] = img.format

            record["width"] = img.width

            record["height"] = img.height

            record["mode"] = img.mode

            dimension_counts[
                (img.width, img.height)
            ] += 1

            mode_counts[
                img.mode
            ] += 1

            format_counts[
                img.format
            ] += 1

    except Exception as e:

        record["error"] = str(e)

        corrupt_images.append({
            "path": str(path),
            "error": str(e),
        })

    try:

        sha = hashlib.sha256()

        with open(path, "rb") as f:

            for chunk in iter(
                lambda: f.read(1024 * 1024),
                b""
            ):

                sha.update(chunk)

        record["sha256"] = sha.hexdigest()

        hash_to_paths[
            record["sha256"]
        ].append(str(path))

    except Exception as e:

        hash_errors.append({
            "path": str(path),
            "error": str(e),
        })

    image_records.append(record)

    if len(image_records) % 100 == 0:

        print(
            f"Audited {len(image_records)} / "
            f"{len(image_files)} images"
        )

image_audit_df = pd.DataFrame(
    image_records
)

image_audit_df.to_csv(
    AUDIT_DIR / "verite_image_audit.csv",
    index=False
)

duplicate_content_results = []

for sha, paths in hash_to_paths.items():

    if len(paths) > 1:

        duplicate_content_results.append({
            "sha256": sha,
            "duplicate_count": len(paths),
            "paths": " | ".join(paths),
        })

duplicate_content_df = pd.DataFrame(
    duplicate_content_results
)

duplicate_content_df.to_csv(
    AUDIT_DIR / "verite_duplicate_content.csv",
    index=False
)

duplicate_filename_results = []

for filename, paths in filename_to_paths.items():

    if len(paths) > 1:

        duplicate_filename_results.append({
            "filename": filename,
            "duplicate_count": len(paths),
            "paths": " | ".join(paths),
        })

duplicate_filename_df = pd.DataFrame(
    duplicate_filename_results
)

duplicate_filename_df.to_csv(
    AUDIT_DIR / "verite_duplicate_filenames.csv",
    index=False
)

csv_results = []

for path in all_files:

    if path.suffix.lower() != ".csv":

        continue

    record = {
        "path": str(path),
        "filename": path.name,
        "readable": False,
        "rows": None,
        "columns": None,
        "column_names": None,
        "error": None,
    }

    try:

        temp_df = pd.read_csv(
            path,
            low_memory=False
        )

        record["readable"] = True

        record["rows"] = len(temp_df)

        record["columns"] = len(
            temp_df.columns
        )

        record["column_names"] = " | ".join(
            map(str, temp_df.columns)
        )

        print("\nCSV:", path)

        print("Rows:", len(temp_df))

        print("Columns:", list(temp_df.columns))

        print("\nPreview:")

        display(
            temp_df.head(5)
        )

    except Exception as e:

        record["error"] = str(e)

    csv_results.append(record)

csv_audit_df = pd.DataFrame(
    csv_results
)

csv_audit_df.to_csv(
    AUDIT_DIR / "verite_csv_metadata_audit.csv",
    index=False
)

json_results = []

for path in all_files:

    ext = path.suffix.lower()

    if ext not in {
        ".json",
        ".jsonl",
        ".ndjson"
    }:

        continue

    record = {
        "path": str(path),
        "filename": path.name,
        "extension": ext,
        "readable": False,
        "records": None,
        "top_level_type": None,
        "top_level_keys": None,
        "error": None,
    }

    try:

        if ext == ".json":

            with open(
                path,
                "r",
                encoding="utf-8"
            ) as f:

                data = json.load(f)

            record["readable"] = True

            record["top_level_type"] = (
                type(data).__name__
            )

            if isinstance(data, dict):

                record["top_level_keys"] = (
                    " | ".join(
                        map(str, data.keys())
                    )
                )

                print("\nJSON:", path)

                print(
                    "Keys:",
                    list(data.keys())
                )

            elif isinstance(data, list):

                record["records"] = len(data)

                print("\nJSON:", path)

                print(
                    "Records:",
                    len(data)
                )

                print("Preview:", data[:3])

        else:

            jsonl_records = []

            with open(
                path,
                "r",
                encoding="utf-8"
            ) as f:

                for line in f:

                    line = line.strip()

                    if line:

                        jsonl_records.append(
                            json.loads(line)
                        )

            record["readable"] = True

            record["records"] = len(
                jsonl_records
            )

            record["top_level_type"] = "JSONL"

            print("\nJSONL:", path)

            print(
                "Records:",
                len(jsonl_records)
            )

            print(
                "Preview:",
                jsonl_records[:3]
            )

    except Exception as e:

        record["error"] = str(e)

    json_results.append(record)

json_audit_df = pd.DataFrame(
    json_results
)

json_audit_df.to_csv(
    AUDIT_DIR / "verite_json_metadata_audit.csv",
    index=False
)

zip_results = []

for path in all_files:

    if path.suffix.lower() != ".zip":

        continue

    record = {
        "path": str(path),
        "filename": path.name,
        "zip_readable": False,
        "integrity_test": None,
        "member_count": None,
        "image_members": None,
        "metadata_members": None,
        "error": None,
    }

    try:

        with zipfile.ZipFile(
            path,
            "r"
        ) as zf:

            members = zf.namelist()

            bad_member = zf.testzip()

            record["zip_readable"] = True

            record["integrity_test"] = (
                "PASS"
                if bad_member is None
                else f"FAIL: {bad_member}"
            )

            record["member_count"] = len(
                members
            )

            record["image_members"] = sum(
                Path(m).suffix.lower()
                in IMAGE_EXTENSIONS
                for m in members
            )

            record["metadata_members"] = sum(
                Path(m).suffix.lower()
                in {
                    ".csv",
                    ".json",
                    ".jsonl",
                    ".ndjson",
                    ".txt",
                }
                for m in members
            )

            print("\nZIP:", path)

            print(
                "Members:",
                record["member_count"]
            )

            print(
                "Image members:",
                record["image_members"]
            )

            print(
                "Integrity:",
                record["integrity_test"]
            )

    except Exception as e:

        record["error"] = str(e)

    zip_results.append(record)

zip_audit_df = pd.DataFrame(
    zip_results
)

zip_audit_df.to_csv(
    AUDIT_DIR / "verite_zip_audit.csv",
    index=False
)

readable_count = int(
    image_audit_df["readable"].sum()
)

unreadable_count = (
    len(image_files)
    - readable_count
)

duplicate_content_instances = sum(
    len(paths) - 1
    for paths in hash_to_paths.values()
    if len(paths) > 1
)

duplicate_filename_instances = sum(
    len(paths) - 1
    for paths in filename_to_paths.values()
    if len(paths) > 1
)

summary = {
    "dataset": "VERITE",
    "audit_timestamp": datetime.now().isoformat(),
    "dataset_directory": str(DATASET_DIR),
    "audit_directory": str(AUDIT_DIR),

    "total_files": int(len(all_files)),

    "extension_counts": {
        str(k): int(v)
        for k, v in extension_counts.items()
    },

    "image_count": int(len(image_files)),

    "readable_images": int(readable_count),

    "unreadable_images": int(unreadable_count),

    "image_formats": {
        str(k): int(v)
        for k, v in format_counts.items()
    },

    "image_modes": {
        str(k): int(v)
        for k, v in mode_counts.items()
    },

    "unique_dimensions": int(
        len(dimension_counts)
    ),

    "dimension_counts": {
        f"{w}x{h}": int(count)
        for (w, h), count in dimension_counts.items()
    },

    "duplicate_content_groups": int(
        len(duplicate_content_results)
    ),

    "duplicate_content_instances": int(
        duplicate_content_instances
    ),

    "duplicate_filename_groups": int(
        len(duplicate_filename_results)
    ),

    "duplicate_filename_instances": int(
        duplicate_filename_instances
    ),

    "sha256_errors": int(
        len(hash_errors)
    ),

    "csv_files": int(
        len(csv_results)
    ),

    "json_files": int(
        len(json_results)
    ),

    "zip_files": int(
        len(zip_results)
    ),

    "corrupt_images": corrupt_images,

    "sha256_error_details": hash_errors,

    "status": (
        "PASS"
        if (
            unreadable_count == 0
            and len(hash_errors) == 0
        )
        else "REVIEW_REQUIRED"
    ),
}

summary_path = (
    AUDIT_DIR
    / "verite_dataset_verification_summary.json"
)

with open(
    summary_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        summary,
        f,
        indent=4,
        ensure_ascii=False
    )

print("\n")
print("=" * 80)
print("VERITE VERIFICATION SUMMARY")
print("=" * 80)

print("Total files:", len(all_files))

print("Images:", len(image_files))

print("Readable:", readable_count)

print("Unreadable:", unreadable_count)

print("\nFormats:", dict(format_counts))

print("Modes:", dict(mode_counts))

print("Unique dimensions:", len(dimension_counts))

print("\nDuplicate content groups:",
      len(duplicate_content_results))

print("Duplicate content instances:",
      duplicate_content_instances)

print("\nDuplicate filename groups:",
      len(duplicate_filename_results))

print("Duplicate filename instances:",
      duplicate_filename_instances)

print("SHA256 errors:", len(hash_errors))

print("CSV files:", len(csv_results))

print("JSON files:", len(json_results))

print("ZIP files:", len(zip_results))

print("\nOverall status:", summary["status"])

print("\nAudit files saved:")

print(
    AUDIT_DIR
    / "verite_file_inventory.csv"
)

print(
    AUDIT_DIR
    / "verite_image_audit.csv"
)

print(
    AUDIT_DIR
    / "verite_duplicate_content.csv"
)

print(
    AUDIT_DIR
    / "verite_duplicate_filenames.csv"
)

print(
    AUDIT_DIR
    / "verite_csv_metadata_audit.csv"
)

print(
    AUDIT_DIR
    / "verite_json_metadata_audit.csv"
)

print(
    AUDIT_DIR
    / "verite_zip_audit.csv"
)

print(summary_path)

print("\n" + "=" * 80)
print("VERITE VERIFICATION COMPLETED")
print("=" * 80)

In [ ]:
import os
import json
import time
import hashlib
import gc
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image, ImageFile

import torch
import torch.nn.functional as F
from transformers import AutoProcessor, AutoModel

ImageFile.LOAD_TRUNCATED_IMAGES = False

ROOT = Path("/content/drive/MyDrive/TwinCheck")

VERITE_IMAGE_ROOT = ROOT / "data/raw/vision/verite"

OUTPUT_ROOT = ROOT / "features/vision/verite"

SIGLIP_OUT = OUTPUT_ROOT / "siglip2_so400m"
CLIP_OUT = OUTPUT_ROOT / "clip_vitl14"

AUDIT_ROOT = ROOT / "audit/verite"

SIGLIP_OUT.mkdir(parents=True, exist_ok=True)
CLIP_OUT.mkdir(parents=True, exist_ok=True)
AUDIT_ROOT.mkdir(parents=True, exist_ok=True)

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

BATCH_SIZE = 16

SIGLIP_MODEL_NAME = "google/siglip2-so400m-patch14-384"
CLIP_MODEL_NAME = "openai/clip-vit-large-patch14"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

USE_FP16 = DEVICE == "cuda"

print("=" * 70)
print("VERITE EMBEDDING EXTRACTION")
print("=" * 70)
print("Device:", DEVICE)
print("FP16 inference:", USE_FP16)

if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA capability:", torch.cuda.get_device_capability())

if not VERITE_IMAGE_ROOT.exists():
    raise FileNotFoundError(
        f"VERITE image folder not found:\n{VERITE_IMAGE_ROOT}"
    )

image_paths = sorted(
    [
        p for p in VERITE_IMAGE_ROOT.rglob("*")
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
    ],
    key=lambda p: str(p.relative_to(VERITE_IMAGE_ROOT)).lower()
)

print("\nImages discovered:", len(image_paths))

if len(image_paths) == 0:
    raise RuntimeError("No images found in VERITE directory.")

def calculate_md5(file_path, chunk_size=1024 * 1024):
    h = hashlib.md5()

    with open(file_path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()

manifest_rows = []
failed_images = []

for i, image_path in enumerate(image_paths):

    try:
        with Image.open(image_path) as im:
            im.verify()

        with Image.open(image_path) as im:
            width, height = im.size
            original_mode = im.mode

            rgb_image = im.convert("RGB")

            rgb_image.load()

        image_id = calculate_md5(image_path)

        manifest_rows.append({
            "row_index": len(manifest_rows),
            "image_id": image_id,
            "relative_path": str(
                image_path.relative_to(VERITE_IMAGE_ROOT)
            ),
            "absolute_path": str(image_path),
            "width": width,
            "height": height,
            "original_mode": original_mode,
            "model_mode": "RGB",
            "md5": image_id
        })

    except Exception as e:

        failed_images.append({
            "path": str(image_path),
            "error": str(e)
        })

    if (i + 1) % 100 == 0:
        print(f"Validated {i+1}/{len(image_paths)} images")

manifest = pd.DataFrame(manifest_rows)

print("\nVALIDATION SUMMARY")
print("----------------")
print("Discovered:", len(image_paths))
print("Valid:", len(manifest))
print("Failed:", len(failed_images))

if len(manifest) == 0:
    raise RuntimeError("No valid images available.")

manifest = manifest.sort_values(
    "relative_path",
    key=lambda s: s.str.lower()
).reset_index(drop=True)

manifest["row_index"] = np.arange(len(manifest))

MANIFEST_PATH = OUTPUT_ROOT / "verite_embedding_manifest.csv"

manifest.to_csv(MANIFEST_PATH, index=False)

FAILED_PATH = AUDIT_ROOT / "verite_failed_images.json"

with open(FAILED_PATH, "w") as f:
    json.dump(failed_images, f, indent=2)

print("Manifest saved:", MANIFEST_PATH)

fingerprint_string = "\n".join(
    manifest["image_id"].astype(str).tolist()
)

INPUT_FINGERPRINT = hashlib.sha256(
    fingerprint_string.encode("utf-8")
).hexdigest()

print("\nInput fingerprint:")
print(INPUT_FINGERPRINT)

def load_rgb_images(paths):

    images = []

    for path in paths:

        with Image.open(path) as im:
            images.append(im.convert("RGB"))

    return images

def extract_features(model, processor, images):

    inputs = processor(
        images=images,
        return_tensors="pt"
    )

    inputs = {
        key: value.to(DEVICE)
        for key, value in inputs.items()
    }

    with torch.inference_mode():

        if USE_FP16:

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16
            ):

                outputs = model.get_image_features(
                    **inputs
                )

        else:

            outputs = model.get_image_features(
                **inputs
            )

    if isinstance(outputs, tuple):
        outputs = outputs[0]

    if hasattr(outputs, "pooler_output"):
        outputs = outputs.pooler_output

    if hasattr(outputs, "image_embeds"):
        outputs = outputs.image_embeds

    if not isinstance(outputs, torch.Tensor):
        raise TypeError(
            f"Unexpected feature output type: {type(outputs)}"
        )

    features = outputs.float()

    raw_features = features.cpu().numpy()

    normalized_features = F.normalize(
        features,
        p=2,
        dim=1
    ).cpu().numpy()

    return raw_features, normalized_features

def run_model_extraction(
    model_name,
    output_dir,
    manifest,
    batch_size=16
):

    print("\n" + "=" * 70)
    print("MODEL:", model_name)
    print("=" * 70)

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    raw_path = output_dir / "embeddings_raw.npy"
    norm_path = output_dir / "embeddings_normalized.npy"

    checkpoint_raw = output_dir / "checkpoint_raw.npy"
    checkpoint_norm = output_dir / "checkpoint_normalized.npy"
    checkpoint_json = output_dir / "checkpoint_metadata.json"

    start_index = 0
    raw_chunks = []
    norm_chunks = []

    if checkpoint_json.exists():

        with open(checkpoint_json, "r") as f:
            checkpoint_info = json.load(f)

        if checkpoint_info.get("fingerprint") == INPUT_FINGERPRINT:

            start_index = checkpoint_info["completed_rows"]

            if start_index > 0:

                raw_chunks = [
                    np.load(checkpoint_raw)
                ]

                norm_chunks = [
                    np.load(checkpoint_norm)
                ]

                print(
                    f"Resuming from row {start_index}"
                )

        else:

            print(
                "Checkpoint fingerprint mismatch. "
                "Starting fresh."
            )

            start_index = 0

    print("Loading processor...")

    processor = AutoProcessor.from_pretrained(
        model_name
    )

    print("Loading model...")

    model = AutoModel.from_pretrained(
        model_name
    )

    model = model.to(DEVICE)
    model.eval()

    total_rows = len(manifest)

    start_time = time.time()

    for start in range(start_index, total_rows, batch_size):

        end = min(start + batch_size, total_rows)

        batch_manifest = manifest.iloc[start:end]

        batch_paths = batch_manifest[
            "absolute_path"
        ].tolist()

        images = load_rgb_images(batch_paths)

        raw_batch, norm_batch = extract_features(
            model,
            processor,
            images
        )

        raw_chunks.append(raw_batch)
        norm_chunks.append(norm_batch)

        current_raw = np.concatenate(
            raw_chunks,
            axis=0
        )

        current_norm = np.concatenate(
            norm_chunks,
            axis=0
        )

        np.save(
            checkpoint_raw,
            current_raw
        )

        np.save(
            checkpoint_norm,
            current_norm
        )

        checkpoint_info = {
            "model": model_name,
            "completed_rows": end,
            "total_rows": total_rows,
            "fingerprint": INPUT_FINGERPRINT,
            "batch_size": batch_size,
            "inference_precision": (
                "FP16" if USE_FP16 else "FP32"
            ),
            "embedding_storage": "FP32",
            "timestamp": time.strftime(
                "%Y-%m-%d %H:%M:%S"
            )
        }

        with open(checkpoint_json, "w") as f:
            json.dump(checkpoint_info, f, indent=2)

        elapsed = time.time() - start_time

        rows_completed = end

        rate = rows_completed / max(elapsed, 1e-6)

        remaining = total_rows - rows_completed

        eta_seconds = remaining / max(rate, 1e-6)

        print(
            f"{end}/{total_rows} | "
            f"{rate:.2f} images/sec | "
            f"ETA {eta_seconds/60:.1f} min"
        )

        del images
        del raw_batch
        del norm_batch

        if DEVICE == "cuda":
            torch.cuda.empty_cache()

    raw_embeddings = np.concatenate(
        raw_chunks,
        axis=0
    )

    normalized_embeddings = np.concatenate(
        norm_chunks,
        axis=0
    )

    np.save(raw_path, raw_embeddings)
    np.save(norm_path, normalized_embeddings)

    audit = {
        "model": model_name,
        "rows": int(raw_embeddings.shape[0]),
        "raw_shape": list(raw_embeddings.shape),
        "normalized_shape": list(
            normalized_embeddings.shape
        ),
        "raw_dtype": str(raw_embeddings.dtype),
        "normalized_dtype": str(
            normalized_embeddings.dtype
        ),
        "raw_finite": bool(
            np.isfinite(raw_embeddings).all()
        ),
        "normalized_finite": bool(
            np.isfinite(normalized_embeddings).all()
        ),
        "normalized_norms_mean": float(
            np.linalg.norm(
                normalized_embeddings,
                axis=1
            ).mean()
        ),
        "normalized_norms_min": float(
            np.linalg.norm(
                normalized_embeddings,
                axis=1
            ).min()
        ),
        "normalized_norms_max": float(
            np.linalg.norm(
                normalized_embeddings,
                axis=1
            ).max()
        ),
        "fingerprint": INPUT_FINGERPRINT,
        "inference_precision": (
            "FP16" if USE_FP16 else "FP32"
        ),
        "storage_precision": "FP32",
        "batch_size": batch_size,
        "elapsed_seconds": time.time() - start_time
    }

    audit_path = output_dir / "final_audit.json"

    with open(audit_path, "w") as f:
        json.dump(audit, f, indent=2)

    print("\nFINAL AUDIT")
    print("-----------")
    print(json.dumps(audit, indent=2))

    if checkpoint_raw.exists():
        checkpoint_raw.unlink()

    if checkpoint_norm.exists():
        checkpoint_norm.unlink()

    if checkpoint_json.exists():
        checkpoint_json.unlink()

    del model
    del processor

    gc.collect()

    if DEVICE == "cuda":
        torch.cuda.empty_cache()

    return raw_embeddings, normalized_embeddings, audit

siglip_raw, siglip_norm, siglip_audit = (
    run_model_extraction(
        model_name=SIGLIP_MODEL_NAME,
        output_dir=SIGLIP_OUT,
        manifest=manifest,
        batch_size=BATCH_SIZE
    )
)

clip_raw, clip_norm, clip_audit = (
    run_model_extraction(
        model_name=CLIP_MODEL_NAME,
        output_dir=CLIP_OUT,
        manifest=manifest,
        batch_size=BATCH_SIZE
    )
)

print("\n" + "=" * 70)
print("VERITE BOTH-MODEL FINAL AUDIT")
print("=" * 70)

assert len(manifest) == siglip_raw.shape[0]
assert len(manifest) == siglip_norm.shape[0]

assert len(manifest) == clip_raw.shape[0]
assert len(manifest) == clip_norm.shape[0]

assert siglip_raw.shape[1] == 1152
assert siglip_norm.shape[1] == 1152

assert clip_raw.shape[1] == 768
assert clip_norm.shape[1] == 768

assert np.isfinite(siglip_raw).all()
assert np.isfinite(siglip_norm).all()

assert np.isfinite(clip_raw).all()
assert np.isfinite(clip_norm).all()

siglip_norm_values = np.linalg.norm(
    siglip_norm,
    axis=1
)

clip_norm_values = np.linalg.norm(
    clip_norm,
    axis=1
)

assert np.allclose(
    siglip_norm_values,
    1.0,
    atol=1e-4
)

assert np.allclose(
    clip_norm_values,
    1.0,
    atol=1e-4
)

assert (
    siglip_audit["fingerprint"]
    == clip_audit["fingerprint"]
    == INPUT_FINGERPRINT
)

final_report = {
    "dataset": "VERITE",
    "image_count": int(len(manifest)),
    "input_fingerprint": INPUT_FINGERPRINT,
    "manifest_path": str(MANIFEST_PATH),
    "failed_image_count": int(len(failed_images)),
    "siglip2": siglip_audit,
    "clip": clip_audit,
    "cross_model_alignment": True,
    "cross_model_audit_pass": True
}

FINAL_REPORT_PATH = (
    AUDIT_ROOT / "verite_both_models_final_report.json"
)

with open(FINAL_REPORT_PATH, "w") as f:
    json.dump(final_report, f, indent=2)

print("\n" + "=" * 70)
print("VERITE BOTH MODELS COMPLETE")
print("=" * 70)

print("Images:", len(manifest))
print("SigLIP 2 shape:", siglip_raw.shape)
print("CLIP shape:", clip_raw.shape)
print("Input fingerprint:", INPUT_FINGERPRINT)
print("Final report:", FINAL_REPORT_PATH)

print("\nALL FINAL ASSERTIONS PASSED.")

In [ ]:
from pathlib import Path
import pandas as pd
import requests
import os

ROOT = Path("/content/drive/MyDrive/TwinCheck")

VERITE_ROOT = ROOT / "data/raw/vision/verite"
IMAGE_DIR = VERITE_ROOT / "images_extracted/images"

METADATA_DIR = ROOT / "data/raw/vision/verite/metadata"
METADATA_DIR.mkdir(parents=True, exist_ok=True)

CSV_PATH = METADATA_DIR / "VERITE_official.csv"

CSV_URL = (
    "https://raw.githubusercontent.com/"
    "stevejpapad/image-text-verification/master/VERITE/VERITE.csv"
)

print("=" * 70)
print("STEP 1 — DOWNLOAD OFFICIAL VERITE CAPTION FILE")
print("=" * 70)

response = requests.get(CSV_URL, timeout=60)
response.raise_for_status()

CSV_PATH.write_bytes(response.content)

print("Download successful!")
print("Saved to:", CSV_PATH)
print("File size:", f"{CSV_PATH.stat().st_size:,}", "bytes")

df = pd.read_csv(CSV_PATH)

print("\n" + "=" * 70)
print("STEP 2 — DATASET STRUCTURE")
print("=" * 70)

print("Rows:", len(df))
print("Columns:", list(df.columns))

print("\nFirst 5 rows:")
display(df.head())

required_columns = ["caption", "image_path", "label"]

missing_columns = [
    column for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

print("\nRequired columns confirmed.")

df["caption"] = df["caption"].fillna("").astype(str)

empty_caption_count = (
    df["caption"].str.strip() == ""
).sum()

print("\n" + "=" * 70)
print("STEP 3 — CAPTION QUALITY CHECK")
print("=" * 70)

print("Total captions:", len(df))
print("Empty captions:", empty_caption_count)
print("Non-empty captions:", len(df) - empty_caption_count)

df["official_image_filename"] = df["image_path"].apply(
    lambda x: Path(str(x)).name
)

df["local_image_path"] = df["official_image_filename"].apply(
    lambda filename: str(IMAGE_DIR / filename)
)

df["image_exists_locally"] = df["local_image_path"].apply(
    lambda path: Path(path).exists()
)

missing_images = df[~df["image_exists_locally"]].copy()

print("\n" + "=" * 70)
print("STEP 4 — IMAGE-CAPTION MAPPING CHECK")
print("=" * 70)

print("CSV rows:", len(df))
print("Rows with local image:", df["image_exists_locally"].sum())
print("Rows with missing local image:", len(missing_images))

if len(missing_images) > 0:
    print("\nExamples of missing image mappings:")
    display(
        missing_images[
            ["caption", "image_path", "label"]
        ].head(20)
    )

print("\n" + "=" * 70)
print("STEP 5 — LABEL DISTRIBUTION")
print("=" * 70)

print(df["label"].value_counts(dropna=False))

duplicate_pairs = df.duplicated(
    subset=["image_path", "caption", "label"]
).sum()

print("\nDuplicate image-caption-label rows:", duplicate_pairs)

print("\n" + "=" * 70)
print("VERITE CAPTION DISCOVERY SUMMARY")
print("=" * 70)

print("Official CSV downloaded: YES")
print("Rows:", len(df))
print("Empty captions:", empty_caption_count)
print("Local image mappings found:", df["image_exists_locally"].sum())
print("Missing local images:", len(missing_images))
print("Duplicate rows:", duplicate_pairs)

VALIDATION_PATH = METADATA_DIR / "VERITE_caption_mapping_audit.csv"

df.to_csv(VALIDATION_PATH, index=False)

print("\nAudit CSV saved to:")
print(VALIDATION_PATH)

print("\n" + "=" * 70)
print("STEP COMPLETE — REVIEW THE OUTPUT BEFORE EMBEDDING")
print("=" * 70)

In [ ]:
import os
import json
import hashlib
import time
import numpy as np
import pandas as pd
import torch

from pathlib import Path
from tqdm.auto import tqdm

ROOT = Path("/content/drive/MyDrive/TwinCheck")

VERITE_ROOT = ROOT / "data/raw/vision/verite"
CSV_PATH = VERITE_ROOT / "metadata/VERITE_official.csv"

OUTPUT_DIR = ROOT / "features/vision/verite/siglip2_text"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

AUDIT_DIR = ROOT / "audit/verite"
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(CSV_PATH)

df["caption"] = df["caption"].fillna("").astype(str)
df["image_path"] = df["image_path"].astype(str)
df["label"] = df["label"].astype(str)

df["caption_record_index"] = np.arange(len(df))

df["image_filename"] = df["image_path"].apply(
    lambda x: Path(x).name
)

CAPTIONS = df["caption"].tolist()

print("Caption records:", len(CAPTIONS))
print("Unique images:", df["image_filename"].nunique())

order_string = "\n".join(
    df["caption_record_index"].astype(str)
    + "|"
    + df["image_path"]
    + "|"
    + df["label"]
    + "|"
    + df["caption"]
)

order_fingerprint = hashlib.sha256(
    order_string.encode("utf-8")
).hexdigest()

print("Fingerprint:", order_fingerprint)

MODEL_NAME = "google/siglip2-so400m-patch14-384"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

model.eval()

BATCH_SIZE = 16

all_embeddings = []

start_time = time.time()

with torch.inference_mode():

    for start in tqdm(
        range(0, len(CAPTIONS), BATCH_SIZE),
        desc="Encoding VERITE captions"
    ):

        batch_captions = CAPTIONS[
            start:start + BATCH_SIZE
        ]

        inputs = processor(
            text=batch_captions,
            padding="max_length",
            truncation=True,
            max_length=64,
            return_tensors="pt"
        )

        inputs = {
            key: value.to(DEVICE)
            for key, value in inputs.items()
        }

        result = model.get_text_features(**inputs)

        if hasattr(result, "pooler_output"):
            text_features = result.pooler_output

        elif isinstance(result, torch.Tensor):
            text_features = result

        else:
            raise TypeError(
                f"Unexpected output type: {type(result)}"
            )

        text_features = text_features.float()

        all_embeddings.append(
            text_features.cpu().numpy()
        )

embeddings_raw = np.concatenate(
    all_embeddings,
    axis=0
).astype(np.float32)

elapsed_time = time.time() - start_time

norms = np.linalg.norm(
    embeddings_raw,
    axis=1,
    keepdims=True
)

assert np.all(norms > 0), "Zero-norm embedding detected"

embeddings_normalized = (
    embeddings_raw / norms
).astype(np.float32)

print("\n" + "=" * 70)
print("VALIDATION")
print("=" * 70)

print("Raw shape:", embeddings_raw.shape)
print("Normalized shape:", embeddings_normalized.shape)

print("Raw finite:", np.isfinite(embeddings_raw).all())
print("Normalized finite:", np.isfinite(embeddings_normalized).all())

normalized_norms = np.linalg.norm(
    embeddings_normalized,
    axis=1
)

print("Norm minimum:", normalized_norms.min())
print("Norm maximum:", normalized_norms.max())

assert embeddings_raw.shape == (1001, 1152)
assert embeddings_normalized.shape == (1001, 1152)

assert np.isfinite(embeddings_raw).all()
assert np.isfinite(embeddings_normalized).all()

assert np.allclose(
    normalized_norms,
    1.0,
    atol=1e-4
)

print("Embedding validation: PASS")

RAW_PATH = OUTPUT_DIR / "embeddings_raw.npy"

NORMALIZED_PATH = OUTPUT_DIR / "embeddings_normalized.npy"

np.save(RAW_PATH, embeddings_raw)

np.save(
    NORMALIZED_PATH,
    embeddings_normalized
)

manifest = df.copy()

manifest["embedding_index"] = np.arange(len(manifest))

manifest["local_image_path"] = manifest[
    "image_filename"
].apply(
    lambda filename: str(
        VERITE_ROOT / "images_extracted/images" / filename
    )
)

manifest["image_exists"] = manifest[
    "local_image_path"
].apply(
    lambda path: Path(path).exists()
)

MANIFEST_PATH = OUTPUT_DIR / "text_embedding_manifest.csv"

manifest.to_csv(
    MANIFEST_PATH,
    index=False
)

audit = {
    "dataset": "VERITE",
    "model": MODEL_NAME,
    "task": "caption_text_embedding",
    "total_caption_records": int(len(df)),
    "unique_images": int(df["image_filename"].nunique()),
    "embedding_shape": list(embeddings_raw.shape),
    "normalized_embedding_shape": list(
        embeddings_normalized.shape
    ),
    "dtype": str(embeddings_raw.dtype),
    "device": DEVICE,
    "batch_size": BATCH_SIZE,
    "elapsed_seconds": round(elapsed_time, 2),
    "caption_order_fingerprint": order_fingerprint,
    "all_embeddings_finite": bool(
        np.isfinite(embeddings_raw).all()
    ),
    "all_normalized_embeddings_finite": bool(
        np.isfinite(embeddings_normalized).all()
    ),
    "unit_norm_validation": bool(
        np.allclose(normalized_norms, 1.0, atol=1e-4)
    ),
    "all_local_images_exist": bool(
        manifest["image_exists"].all()
    ),
    "label_distribution": {
        str(k): int(v)
        for k, v in df["label"].value_counts().items()
    },
    "raw_embedding_path": str(RAW_PATH),
    "normalized_embedding_path": str(NORMALIZED_PATH),
    "manifest_path": str(MANIFEST_PATH)
}

AUDIT_PATH = OUTPUT_DIR / "final_audit.json"

with open(AUDIT_PATH, "w") as f:
    json.dump(audit, f, indent=2)

print("\n" + "=" * 70)
print("FINAL REPORT")
print("=" * 70)

print(json.dumps(audit, indent=2))

print("\nSaved files:")
print(RAW_PATH)
print(NORMALIZED_PATH)
print(MANIFEST_PATH)
print(AUDIT_PATH)

print("\nVERITE SIGLIP2 TEXT EMBEDDINGS COMPLETE")

In [ ]:
from pathlib import Path
import pandas as pd
import json

ROOT = Path("/content/drive/MyDrive/TwinCheck")

VERITE_ROOT = ROOT / "data/raw/vision/verite"
IMAGE_DIR = VERITE_ROOT / "images_extracted/images"

CAPTION_CSV = VERITE_ROOT / "metadata/VERITE_official.csv"

IMAGE_MANIFEST = (
    ROOT / "features/vision/verite/verite_embedding_manifest.csv"
)

AUDIT_DIR = ROOT / "audit/verite"
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

local_images = sorted([
    p.name for p in IMAGE_DIR.iterdir()
    if p.is_file()
])

local_image_set = set(local_images)

print("=" * 70)
print("STEP 1 — LOCAL IMAGE COLLECTION")
print("=" * 70)

print("Local image files:", len(local_images))
print("Unique local filenames:", len(local_image_set))

captions = pd.read_csv(CAPTION_CSV)

captions["image_filename"] = captions["image_path"].apply(
    lambda x: Path(str(x)).name
)

caption_image_set = set(
    captions["image_filename"].unique()
)

print("\n" + "=" * 70)
print("STEP 2 — CAPTION DATASET")
print("=" * 70)

print("Caption records:", len(captions))
print("Unique caption-linked images:", len(caption_image_set))

images_without_captions = sorted(
    local_image_set - caption_image_set
)

captions_without_local_images = sorted(
    caption_image_set - local_image_set
)

images_with_captions = sorted(
    local_image_set & caption_image_set
)

print("\n" + "=" * 70)
print("STEP 3 — ALIGNMENT")
print("=" * 70)

print("Images with caption records:", len(images_with_captions))
print("Local images without captions:", len(images_without_captions))
print("Caption references without local image:",
      len(captions_without_local_images))

print("\nImages without caption records:")
for name in images_without_captions:
    print("  ", name)

print("\nCaption references missing locally:")
for name in captions_without_local_images:
    print("  ", name)

print("\n" + "=" * 70)
print("STEP 4 — IMAGE EMBEDDING MANIFEST")
print("=" * 70)

if IMAGE_MANIFEST.exists():

    image_manifest = pd.read_csv(IMAGE_MANIFEST)

    print("Manifest found:", IMAGE_MANIFEST)
    print("Manifest rows:", len(image_manifest))
    print("Manifest columns:", list(image_manifest.columns))

    possible_columns = [
        "image_filename",
        "filename",
        "file_name",
        "image_path",
        "path",
        "local_path"
    ]

    filename_column = next(
        (
            col for col in possible_columns
            if col in image_manifest.columns
        ),
        None
    )

    if filename_column is not None:

        manifest_filenames = set(
            image_manifest[filename_column]
            .astype(str)
            .apply(lambda x: Path(x).name)
        )

        print("Filename column:", filename_column)
        print("Unique filenames in manifest:",
              len(manifest_filenames))

        print(
            "Local images missing from manifest:",
            len(local_image_set - manifest_filenames)
        )

        print(
            "Manifest images missing locally:",
            len(manifest_filenames - local_image_set)
        )

    else:

        print(
            "Could not automatically identify filename column."
        )

else:

    print("Image manifest not found at expected path.")
    print("Please check the path.")

audit = {
    "dataset": "VERITE",
    "local_image_count": len(local_image_set),
    "caption_record_count": len(captions),
    "unique_caption_linked_images": len(caption_image_set),
    "images_with_captions": len(images_with_captions),
    "images_without_captions": images_without_captions,
    "caption_references_without_local_images":
        captions_without_local_images,
    "alignment_status": (
        "PASS"
        if len(captions_without_local_images) == 0
        else "FAIL"
    )
}

AUDIT_JSON = AUDIT_DIR / "verite_image_caption_alignment_audit.json"

with open(AUDIT_JSON, "w") as f:
    json.dump(audit, f, indent=2)

print("\n" + "=" * 70)
print("FINAL ALIGNMENT AUDIT")
print("=" * 70)

print(json.dumps(audit, indent=2))

print("\nAudit saved to:")
print(AUDIT_JSON)

print("\nALIGNMENT AUDIT COMPLETE")

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import json

ROOT = Path("/content/drive/MyDrive/TwinCheck")

VERITE_ROOT = ROOT / "data/raw/vision/verite"

IMAGE_MANIFEST_PATH = (
    ROOT / "features/vision/verite/verite_embedding_manifest.csv"
)

TEXT_MANIFEST_PATH = (
    ROOT / "features/vision/verite/siglip2_text/text_embedding_manifest.csv"
)

SIGLIP_IMAGE_PATH = (
    ROOT / "features/vision/verite/siglip2_so400m/embeddings_normalized.npy"
)

SIGLIP_TEXT_PATH = (
    ROOT / "features/vision/verite/siglip2_text/embeddings_normalized.npy"
)

OUTPUT_DIR = ROOT / "features/vision/verite/alignment"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

AUDIT_DIR = ROOT / "audit/verite"
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

image_manifest = pd.read_csv(IMAGE_MANIFEST_PATH)
text_manifest = pd.read_csv(TEXT_MANIFEST_PATH)

image_embeddings = np.load(SIGLIP_IMAGE_PATH)
text_embeddings = np.load(SIGLIP_TEXT_PATH)

print("=" * 70)
print("STEP 1 — LOAD DATA")
print("=" * 70)

print("Image manifest rows:", len(image_manifest))
print("Text manifest rows:", len(text_manifest))
print("Image embedding shape:", image_embeddings.shape)
print("Text embedding shape:", text_embeddings.shape)

image_manifest["image_filename"] = (
    image_manifest["relative_path"]
    .astype(str)
    .apply(lambda x: Path(x).name)
)

text_manifest["image_filename"] = (
    text_manifest["image_filename"]
    .astype(str)
    .apply(lambda x: Path(x).name)
)

assert "row_index" in image_manifest.columns
assert "embedding_index" in text_manifest.columns

assert image_manifest["row_index"].is_unique
assert text_manifest["embedding_index"].is_unique

assert len(image_manifest) == image_embeddings.shape[0]
assert len(text_manifest) == text_embeddings.shape[0]

assert sorted(image_manifest["row_index"].tolist()) == list(
    range(len(image_manifest))
)

assert sorted(text_manifest["embedding_index"].tolist()) == list(
    range(len(text_manifest))
)

print("\nEmbedding row-index validation: PASS")

duplicate_image_names = image_manifest[
    image_manifest["image_filename"].duplicated(keep=False)
]

print("\nDuplicate image filenames in image manifest:",
      len(duplicate_image_names))

assert len(duplicate_image_names) == 0, (
    "Duplicate filenames found; basename matching is unsafe."
)

image_lookup = image_manifest[
    ["image_filename", "row_index", "image_id", "relative_path", "md5"]
].copy()

image_lookup = image_lookup.rename(
    columns={
        "row_index": "image_embedding_index",
        "image_id": "image_id_from_image_manifest",
        "relative_path": "image_relative_path",
        "md5": "image_md5"
    }
)

alignment = text_manifest.merge(
    image_lookup,
    on="image_filename",
    how="left",
    validate="many_to_one"
)

alignment["image_embedding_found"] = (
    alignment["image_embedding_index"].notna()
)

alignment["image_embedding_index"] = (
    alignment["image_embedding_index"].astype("Int64")
)

unmatched = alignment[
    ~alignment["image_embedding_found"]
]

print("\n" + "=" * 70)
print("STEP 2 — CAPTION TO IMAGE EMBEDDING MATCH")
print("=" * 70)

print("Caption records:", len(alignment))
print("Matched records:", alignment["image_embedding_found"].sum())
print("Unmatched records:", len(unmatched))

if len(unmatched) > 0:
    print("\nUnmatched captions:")
    display(unmatched)

assert len(unmatched) == 0, (
    "Some captions could not be mapped to image embeddings."
)

caption_image_names = set(
    text_manifest["image_filename"]
)

image_names = set(
    image_manifest["image_filename"]
)

uncaptioned_images = sorted(
    image_names - caption_image_names
)

print("\nImages without caption records:")
print(uncaptioned_images)

print("\n" + "=" * 70)
print("STEP 3 — FEATURE DIMENSION CHECK")
print("=" * 70)

print("Image embeddings:", image_embeddings.shape)
print("Text embeddings:", text_embeddings.shape)

assert image_embeddings.shape[1] == 1152
assert text_embeddings.shape[1] == 1152

print("Feature dimensions compatible: PASS")

ALIGNMENT_PATH = OUTPUT_DIR / "verite_caption_to_image_embedding_alignment.csv"

alignment.to_csv(
    ALIGNMENT_PATH,
    index=False
)

audit = {
    "dataset": "VERITE",
    "image_manifest_rows": int(len(image_manifest)),
    "text_manifest_rows": int(len(text_manifest)),
    "image_embedding_shape": list(image_embeddings.shape),
    "text_embedding_shape": list(text_embeddings.shape),
    "caption_records_matched": int(
        alignment["image_embedding_found"].sum()
    ),
    "caption_records_unmatched": int(len(unmatched)),
    "uncaptioned_images": uncaptioned_images,
    "duplicate_image_filenames": int(
        len(duplicate_image_names)
    ),
    "embedding_dimensions_compatible": bool(
        image_embeddings.shape[1] == text_embeddings.shape[1]
    ),
    "alignment_status": "PASS" if (
        len(unmatched) == 0
        and len(duplicate_image_names) == 0
        and image_embeddings.shape[1] == text_embeddings.shape[1]
    ) else "FAIL",
    "alignment_csv": str(ALIGNMENT_PATH)
}

AUDIT_PATH = AUDIT_DIR / "verite_definitive_embedding_alignment_audit.json"

with open(AUDIT_PATH, "w") as f:
    json.dump(audit, f, indent=2)

print("\n" + "=" * 70)
print("FINAL ALIGNMENT REPORT")
print("=" * 70)

print(json.dumps(audit, indent=2))

print("\nAlignment table saved:")
print(ALIGNMENT_PATH)

print("\nAudit report saved:")
print(AUDIT_PATH)

print("\nDEFINITIVE VERITE ALIGNMENT AUDIT COMPLETE")

In [ ]:
import os, gc, json, time
from pathlib import Path

import torch
import numpy as np
import pandas as pd
from PIL import Image
from diffusers import AutoencoderKL

ROOT = Path("/content/drive/MyDrive/TwinCheck")

IMAGE_DIR = ROOT / "data/raw/vision/verite/images_extracted/images"

OUTPUT_DIR = ROOT / "data/processed/vision/verite/vae_reconstructed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

AUDIT_DIR = ROOT / "audit/verite"
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

MANIFEST_PATH = AUDIT_DIR / "verite_vae_reconstruction_manifest.csv"
REPORT_PATH = AUDIT_DIR / "verite_vae_reconstruction_report.json"

MODEL_NAME = "stabilityai/sd-vae-ft-mse"
MAX_SIZE = 512
CHECKPOINT_EVERY = 20

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32

print("Device:", DEVICE)

if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    print("Free GPU memory (GB):",
          round(torch.cuda.mem_get_info()[0] / 1e9, 2))

if MANIFEST_PATH.exists():
    old_df = pd.read_csv(MANIFEST_PATH)

    completed = set(
        old_df.loc[
            old_df["status"] == "SUCCESS",
            "image_filename"
        ].tolist()
    )

    records = old_df.to_dict("records")

    print("Existing manifest found.")
    print("Previously successful images:", len(completed))

else:
    records = []
    completed = set()

    print("Starting fresh.")

gc.collect()

if DEVICE == "cuda":
    torch.cuda.empty_cache()

vae = AutoencoderKL.from_pretrained(
    MODEL_NAME,
    torch_dtype=DTYPE
).to(DEVICE)

vae.eval()

print("VAE loaded.")
print("Scaling factor:", vae.config.scaling_factor)

def resize_image(image, max_size=512):

    image = image.convert("RGB")

    original_width, original_height = image.size

    scale = min(
        max_size / max(original_width, original_height),
        1.0
    )

    new_width = max(
        8,
        int(original_width * scale) // 8 * 8
    )

    new_height = max(
        8,
        int(original_height * scale) // 8 * 8
    )

    image = image.resize(
        (new_width, new_height),
        Image.Resampling.LANCZOS
    )

    return image

def save_checkpoint():

    df = pd.DataFrame(records)

    df.to_csv(MANIFEST_PATH, index=False)

    successful = int(
        (df["status"] == "SUCCESS").sum()
    )

    failed = int(
        (df["status"] == "FAILED").sum()
    )

    report = {
        "dataset": "VERITE",
        "model": MODEL_NAME,
        "device": DEVICE,
        "precision": str(DTYPE),
        "max_image_size": MAX_SIZE,
        "total_records_in_manifest": len(df),
        "successful": successful,
        "failed": failed,
        "manifest_path": str(MANIFEST_PATH),
        "output_directory": str(OUTPUT_DIR)
    }

    with open(REPORT_PATH, "w") as f:
        json.dump(report, f, indent=2)

image_files = sorted([
    p for p in IMAGE_DIR.iterdir()
    if p.suffix.lower() in {
        ".jpg", ".jpeg", ".png", ".webp", ".bmp"
    }
])

assert len(image_files) == 664, (
    f"Expected 664 original images; found {len(image_files)}"
)

print("Total source images:", len(image_files))

start_time = time.time()

processed_this_run = 0
failed_this_run = 0

for idx, image_path in enumerate(image_files, 1):

    if image_path.name in completed:

        output_path = OUTPUT_DIR / (
            image_path.stem + "_vae_twin.jpg"
        )

        if output_path.exists():
            continue
        else:
            completed.remove(image_path.name)

    print(f"\n[{idx}/{len(image_files)}] {image_path.name}")

    output_path = OUTPUT_DIR / (
        image_path.stem + "_vae_twin.jpg"
    )

    try:

        original = Image.open(image_path).convert("RGB")

        original_width, original_height = original.size

        processed = resize_image(original, MAX_SIZE)

        processed_width, processed_height = processed.size

        image_array = (
            np.array(processed).astype(np.float32) / 255.0
        )

        tensor = torch.from_numpy(image_array)
        tensor = tensor.permute(2, 0, 1).unsqueeze(0)

        tensor = tensor * 2.0 - 1.0

        tensor = tensor.to(
            device=DEVICE,
            dtype=DTYPE
        )

        with torch.inference_mode():

            encoded = vae.encode(tensor)

            latent = encoded.latent_dist.mode()

            latent = latent * vae.config.scaling_factor

            decoded = vae.decode(
                latent / vae.config.scaling_factor
            ).sample

        decoded = (
            decoded / 2.0 + 0.5
        ).clamp(0, 1)

        decoded_array = (
            decoded[0]
            .permute(1, 2, 0)
            .float()
            .cpu()
            .numpy()
        )

        decoded_array = (
            decoded_array * 255
        ).round().astype(np.uint8)

        reconstructed = Image.fromarray(decoded_array)

        reconstructed.save(
            output_path,
            format="JPEG",
            quality=95
        )

        original_array = np.array(
            processed
        ).astype(np.float32)

        twin_array = np.array(
            reconstructed
        ).astype(np.float32)

        mse = float(
            np.mean((original_array - twin_array) ** 2)
        )

        psnr = (
            float("inf")
            if mse == 0
            else float(10 * np.log10((255 ** 2) / mse))
        )

        record = {
            "image_filename": image_path.name,
            "original_path": str(image_path),
            "reconstructed_path": str(output_path),
            "original_width": original_width,
            "original_height": original_height,
            "reconstruction_width": processed_width,
            "reconstruction_height": processed_height,
            "mse": mse,
            "psnr_db": psnr,
            "status": "SUCCESS",
            "error": ""
        }

        records = [
            r for r in records
            if r.get("image_filename") != image_path.name
        ]

        records.append(record)

        completed.add(image_path.name)

        processed_this_run += 1

        print("SUCCESS | PSNR:", round(psnr, 3), "dB")

    except Exception as e:

        failed_this_run += 1

        print("FAILED:", repr(e))

        record = {
            "image_filename": image_path.name,
            "original_path": str(image_path),
            "reconstructed_path": str(output_path),
            "original_width": None,
            "original_height": None,
            "reconstruction_width": None,
            "reconstruction_height": None,
            "mse": None,
            "psnr_db": None,
            "status": "FAILED",
            "error": repr(e)
        }

        records = [
            r for r in records
            if r.get("image_filename") != image_path.name
        ]

        records.append(record)

    finally:

        for var in [
            "original", "processed", "tensor",
            "encoded", "latent", "decoded",
            "decoded_array", "reconstructed",
            "original_array", "twin_array"
        ]:

            if var in globals():
                try:
                    del globals()[var]
                except:
                    pass

        gc.collect()

        if DEVICE == "cuda":
            torch.cuda.empty_cache()

    if processed_this_run % CHECKPOINT_EVERY == 0:

        save_checkpoint()

        print(
            f"Checkpoint saved | "
            f"Successful this run: {processed_this_run}"
        )

save_checkpoint()

final_df = pd.read_csv(MANIFEST_PATH)

successful_df = final_df[
    final_df["status"] == "SUCCESS"
]

failed_df = final_df[
    final_df["status"] == "FAILED"
]

print("\n" + "=" * 60)
print("FULL VAE RECONSTRUCTION COMPLETE")
print("=" * 60)

print("Source images:", len(image_files))
print("Manifest records:", len(final_df))
print("Successful reconstructions:", len(successful_df))
print("Failed reconstructions:", len(failed_df))
print("Processed during this run:", processed_this_run)

if len(successful_df) > 0:

    print("\nPSNR statistics:")
    print(successful_df["psnr_db"].describe())

print("\nManifest:", MANIFEST_PATH)
print("Report:", REPORT_PATH)
print("Twin folder:", OUTPUT_DIR)

print("\nElapsed seconds:", round(time.time() - start_time, 2))

display(final_df.head())

In [ ]:
import pandas as pd

recon_manifest = pd.read_csv("/content/drive/MyDrive/TwinCheck/audit/verite/verite_vae_reconstruction_manifest.csv")
embedding_rows = pd.read_csv("/content/drive/MyDrive/TwinCheck/features/vision/verite/verite_embedding_manifest.csv")[["row_index", "absolute_path"]]
twin_records = (
    recon_manifest.merge(embedding_rows, left_on="original_path", right_on="absolute_path", how="inner", validate="one_to_one")
    .rename(columns={"row_index": "embedding_index", "reconstructed_path": "twin_path"})
    .sort_values("embedding_index")
    .to_dict("records")
)
assert len(twin_records) == 664

In [ ]:
import os
import torch
import numpy as np
from PIL import Image
from diffusers import AutoencoderKL
from tqdm.auto import tqdm

VAE_MODEL_ID = "stabilityai/sd-vae-ft-mse"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

OUTPUT_DIR = (
    "/content/drive/MyDrive/TwinCheck/data/processed/vision/"
    "verite/vae_reconstructed"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

affected_records = [
    record for record in twin_records
    if record["image_filename"] in [
        "false_179.jpeg",
        "false_179.jpg"
    ]
]

assert len(affected_records) == 2

print("Affected records:", len(affected_records))

print("\nLoading VAE...")

vae_repair = AutoencoderKL.from_pretrained(
    VAE_MODEL_ID
)

vae_repair = vae_repair.to(DEVICE)

if DEVICE.type == "cuda":
    vae_repair = vae_repair.to(dtype=torch.float16)

vae_repair.eval()

print("VAE loaded successfully")
print("Device:", DEVICE)

def preprocess_for_vae_repair(image_path):

    image = Image.open(image_path).convert("RGB")

    original_width, original_height = image.size

    scale = 512 / max(original_width, original_height)

    new_width = original_width * scale
    new_height = original_height * scale

    new_width = max(
        8,
        round(new_width / 8) * 8
    )

    new_height = max(
        8,
        round(new_height / 8) * 8
    )

    image = image.resize(
        (new_width, new_height),
        Image.Resampling.LANCZOS
    )

    image_array = np.asarray(
        image,
        dtype=np.float32
    ) / 255.0

    image_array = image_array * 2.0 - 1.0

    tensor = torch.from_numpy(
        image_array
    ).permute(2, 0, 1).unsqueeze(0)

    tensor = tensor.to(DEVICE)

    if DEVICE.type == "cuda":
        tensor = tensor.to(torch.float16)

    return tensor, image.size

def reconstruct_single_image(
    original_path,
    output_path
):

    image_tensor, resized_dimensions = (
        preprocess_for_vae_repair(original_path)
    )

    with torch.inference_mode():

        encoded = vae_repair.encode(
            image_tensor
        )

        latent = encoded.latent_dist.mode()

        latent = latent * vae_repair.config.scaling_factor

        decoded = vae_repair.decode(
            latent / vae_repair.config.scaling_factor
        ).sample

    decoded = decoded.float()

    decoded = (
        decoded / 2.0 + 0.5
    ).clamp(0, 1)

    decoded = decoded.squeeze(0)

    decoded = decoded.permute(
        1, 2, 0
    ).cpu().numpy()

    decoded = (
        decoded * 255
    ).round().astype(np.uint8)

    reconstructed = Image.fromarray(
        decoded,
        mode="RGB"
    )

    reconstructed.save(
        output_path,
        format="JPEG",
        quality=95
    )

    return {
        "output_path": output_path,
        "resized_dimensions": resized_dimensions,
        "output_dimensions": reconstructed.size
    }

repair_results = []

for record in affected_records:

    original_filename = record["image_filename"]

    if original_filename.endswith(".jpeg"):

        unique_name = "false_179_jpeg_vae_twin.jpg"

    elif original_filename.endswith(".jpg"):

        unique_name = "false_179_jpg_vae_twin.jpg"

    else:

        raise ValueError(
            f"Unexpected filename: {original_filename}"
        )

    new_twin_path = os.path.join(
        OUTPUT_DIR,
        unique_name
    )

    result = reconstruct_single_image(
        record["original_path"],
        new_twin_path
    )

    record["twin_path"] = new_twin_path

    repair_results.append({
        "image_filename": original_filename,
        "original_path": record["original_path"],
        "twin_path": new_twin_path,
        "embedding_index": record["embedding_index"],
        "resized_dimensions": result["resized_dimensions"],
        "output_dimensions": result["output_dimensions"]
    })

    print("\nReconstructed:", original_filename)
    print("New twin:", new_twin_path)
    print("Dimensions:", result["output_dimensions"])

print("\n" + "="*60)
print("REPAIR VALIDATION")
print("="*60)

for record in affected_records:

    print("\nOriginal:", record["image_filename"])
    print("Twin:", record["twin_path"])

    assert os.path.exists(
        record["twin_path"]
    )

    print(
        "Twin MD5:",
        calculate_md5(record["twin_path"])
    )

all_twin_paths = [
    record["twin_path"]
    for record in twin_records
]

assert len(all_twin_paths) == 664

assert len(set(all_twin_paths)) == 664, (
    "There are still duplicate twin paths!"
)

print("\nPASS: 664 records now have 664 unique twin paths.")

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import torch
from PIL import Image
from tqdm.auto import tqdm

BATCH_SIZE = 4
CHECKPOINT_EVERY = 20

OUTPUT_DIR = (
    "/content/drive/MyDrive/TwinCheck/features/vision/verite/"
    "siglip2_vae_twins"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

FEATURES_PATH = os.path.join(
    OUTPUT_DIR, "embeddings_normalized.npy"
)

CHECKPOINT_PATH = os.path.join(
    OUTPUT_DIR, "embedding_checkpoint.npy"
)

MANIFEST_PATH = os.path.join(
    OUTPUT_DIR, "twin_embedding_manifest.csv"
)

AUDIT_PATH = os.path.join(
    OUTPUT_DIR, "embedding_extraction_audit.json"
)

twin_records_sorted = sorted(
    twin_records,
    key=lambda record: record["embedding_index"]
)

assert len(twin_records_sorted) == 664

image_paths = [
    record["twin_path"]
    for record in twin_records_sorted
]

assert len(set(image_paths)) == 664, (
    "Duplicate twin paths detected!"
)

assert all(os.path.exists(path) for path in image_paths), (
    "Some twin files are missing!"
)

print("Total records:", len(twin_records_sorted))
print("Unique twin paths:", len(set(image_paths)))
print("All files exist: True")

def extract_siglip_embeddings(image_batch):

    images = []

    for path in image_batch:

        with Image.open(path) as image:

            images.append(
                image.convert("RGB")
            )

    inputs = processor(
        images=images,
        return_tensors="pt"
    )

    inputs = {
        key: value.to(DEVICE)
        for key, value in inputs.items()
    }

    with torch.inference_mode():

        outputs = model.get_image_features(
            pixel_values=inputs["pixel_values"]
        )

        if hasattr(outputs, "pooler_output"):

            features = outputs.pooler_output

        elif isinstance(outputs, torch.Tensor):

            features = outputs

        else:

            raise TypeError(
                f"Unexpected output type: {type(outputs)}"
            )

        features = features.float()

        features = torch.nn.functional.normalize(
            features,
            p=2,
            dim=1
        )

    return features.cpu().numpy()

all_embeddings = []
failed_images = []

for start in tqdm(
    range(0, len(image_paths), BATCH_SIZE),
    desc="SigLIP2 twin embedding extraction"
):

    end = min(start + BATCH_SIZE, len(image_paths))

    batch_paths = image_paths[start:end]

    try:

        batch_features = extract_siglip_embeddings(
            batch_paths
        )

        assert batch_features.shape == (
            len(batch_paths),
            1152
        )

        assert np.isfinite(batch_features).all()

        all_embeddings.extend(batch_features)

    except Exception as batch_error:

        print(
            f"\nBatch failure ({start}:{end}): "
            f"{batch_error}"
        )

        for path in batch_paths:

            try:

                individual_features = (
                    extract_siglip_embeddings([path])
                )

                assert individual_features.shape == (
                    1, 1152
                )

                assert np.isfinite(
                    individual_features
                ).all()

                all_embeddings.append(
                    individual_features[0]
                )

            except Exception as individual_error:

                print(
                    "Image failed:",
                    os.path.basename(path)
                )

                failed_images.append({
                    "twin_path": path,
                    "error": str(individual_error)
                })

                all_embeddings.append(
                    np.full(
                        1152,
                        np.nan,
                        dtype=np.float32
                    )
                )

    if (
        end % CHECKPOINT_EVERY == 0
        or end == len(image_paths)
    ):

        checkpoint_array = np.asarray(
            all_embeddings,
            dtype=np.float32
        )

        np.save(
            CHECKPOINT_PATH,
            checkpoint_array
        )

        print(
            f"\nCheckpoint: {len(all_embeddings)}/664"
        )

twin_embeddings = np.asarray(
    all_embeddings,
    dtype=np.float32
)

print("\nEmbedding shape:", twin_embeddings.shape)

assert twin_embeddings.shape == (664, 1152)

valid_rows = np.isfinite(
    twin_embeddings
).all(axis=1)

valid_embeddings = twin_embeddings[valid_rows]

norms = np.linalg.norm(
    valid_embeddings,
    axis=1
)

print("\nValid embeddings:", int(valid_rows.sum()))
print("Failed embeddings:", int((~valid_rows).sum()))

if len(norms) > 0:

    print("Mean norm:", norms.mean())
    print("Minimum norm:", norms.min())
    print("Maximum norm:", norms.max())

    assert np.allclose(
        norms,
        1.0,
        atol=1e-4
    )

np.save(
    FEATURES_PATH,
    twin_embeddings
)

print("\nSaved embeddings:")
print(FEATURES_PATH)

manifest = pd.DataFrame({

    "embedding_row": np.arange(664),

    "original_embedding_index": [
        record["embedding_index"]
        for record in twin_records_sorted
    ],

    "original_filename": [
        record["image_filename"]
        for record in twin_records_sorted
    ],

    "original_path": [
        record["original_path"]
        for record in twin_records_sorted
    ],

    "twin_path": [
        record["twin_path"]
        for record in twin_records_sorted
    ],

    "embedding_valid": valid_rows

})

manifest.to_csv(
    MANIFEST_PATH,
    index=False
)

print("\nManifest saved:")
print(MANIFEST_PATH)

audit = {

    "dataset": "VERITE VAE reconstructed twins",

    "model": "google/siglip2-so400m-patch14-384",

    "embedding_dimension": 1152,

    "total_images": 664,

    "successful_embeddings": int(valid_rows.sum()),

    "failed_embeddings": int((~valid_rows).sum()),

    "batch_size": BATCH_SIZE,

    "normalization": "L2",

    "mean_embedding_norm": (
        float(norms.mean()) if len(norms) else None
    ),

    "minimum_embedding_norm": (
        float(norms.min()) if len(norms) else None
    ),

    "maximum_embedding_norm": (
        float(norms.max()) if len(norms) else None
    ),

    "failed_images": failed_images,

    "embedding_file": FEATURES_PATH,

    "manifest_file": MANIFEST_PATH

}

with open(AUDIT_PATH, "w") as f:

    json.dump(
        audit,
        f,
        indent=4
    )

print("\nAudit saved:")
print(AUDIT_PATH)

if valid_rows.all():

    print("\n" + "=" * 60)
    print("SUCCESS: ALL 664 TWIN EMBEDDINGS EXTRACTED")
    print("=" * 60)

else:

    print("\nWARNING: Some embeddings failed.")
    print("Review failed_images and audit report.")

In [ ]:
import os
import json
import numpy as np
import pandas as pd

ROOT = (
    "/content/drive/MyDrive/TwinCheck"
)

ORIGINAL_EMBEDDINGS_PATH = (
    ROOT + "/features/vision/verite/"
    "siglip2_so400m/embeddings_normalized.npy"
)

ORIGINAL_MANIFEST_PATH = (
    ROOT + "/features/vision/verite/"
    "verite_embedding_manifest.csv"
)

TWIN_EMBEDDINGS_PATH = (
    ROOT + "/features/vision/verite/"
    "siglip2_vae_twins/embeddings_normalized.npy"
)

TWIN_MANIFEST_PATH = (
    ROOT + "/features/vision/verite/"
    "siglip2_vae_twins/twin_embedding_manifest.csv"
)

AUDIT_DIR = (
    ROOT + "/audit/verite/"
    "vae_twin_similarity"
)

os.makedirs(AUDIT_DIR, exist_ok=True)

SIMILARITY_CSV = (
    AUDIT_DIR + "/verite_twin_cosine_similarity.csv"
)

SUMMARY_JSON = (
    AUDIT_DIR + "/verite_twin_similarity_summary.json"
)

original_embeddings = np.load(
    ORIGINAL_EMBEDDINGS_PATH
)

twin_embeddings = np.load(
    TWIN_EMBEDDINGS_PATH
)

original_manifest = pd.read_csv(
    ORIGINAL_MANIFEST_PATH
)

twin_manifest = pd.read_csv(
    TWIN_MANIFEST_PATH
)

print("Original embeddings:", original_embeddings.shape)
print("Twin embeddings:", twin_embeddings.shape)

print("Original manifest:", original_manifest.shape)
print("Twin manifest:", twin_manifest.shape)

assert original_embeddings.shape == (664, 1152)

assert twin_embeddings.shape == (664, 1152)

assert len(original_manifest) == 664

assert len(twin_manifest) == 664

assert np.isfinite(original_embeddings).all()

assert np.isfinite(twin_embeddings).all()

print("\nBasic validation passed.")

assert "row_index" in original_manifest.columns

assert "absolute_path" in original_manifest.columns

assert "original_embedding_index" in twin_manifest.columns

assert "original_path" in twin_manifest.columns

assert "twin_path" in twin_manifest.columns

original_indices = twin_manifest[
    "original_embedding_index"
].to_numpy()

assert np.all(original_indices >= 0)

assert np.all(original_indices < 664)

assert len(np.unique(original_indices)) == 664

print("Original embedding indices validated.")

original_path_lookup = dict(
    zip(
        original_manifest["row_index"],
        original_manifest["absolute_path"]
    )
)

alignment_errors = []

for _, row in twin_manifest.iterrows():

    index = int(
        row["original_embedding_index"]
    )

    twin_original_path = row["original_path"]

    manifest_original_path = original_path_lookup[index]

    if os.path.normpath(twin_original_path) != (
        os.path.normpath(manifest_original_path)
    ):

        alignment_errors.append({
            "embedding_index": index,
            "twin_original_path": twin_original_path,
            "manifest_original_path": manifest_original_path
        })

print("Alignment errors:", len(alignment_errors))

if alignment_errors:

    print("\nAlignment error examples:")

    for error in alignment_errors[:10]:

        print(error)

assert len(alignment_errors) == 0, (
    "Original image embedding alignment failed!"
)

print("PASS — exact original image alignment verified.")

correct_original_embeddings = original_embeddings[
    original_indices
]

cosine_similarities = np.sum(
    correct_original_embeddings * twin_embeddings,
    axis=1
)

cosine_similarities = np.clip(
    cosine_similarities,
    -1.0,
    1.0
)

THRESHOLD = 0.92

passed = cosine_similarities >= THRESHOLD

manual_review = cosine_similarities < THRESHOLD

print("\n" + "=" * 60)
print("COSINE SIMILARITY RESULTS")
print("=" * 60)

print("Total images:", len(cosine_similarities))

print("Passed threshold:", int(passed.sum()))

print("Below threshold:", int(manual_review.sum()))

print(
    "Pass percentage:",
    round(passed.mean() * 100, 2),
    "%"
)

print(
    "Review percentage:",
    round(manual_review.mean() * 100, 2),
    "%"
)

print(
    "Mean similarity:",
    round(float(cosine_similarities.mean()), 4)
)

print(
    "Median similarity:",
    round(float(np.median(cosine_similarities)), 4)
)

print(
    "Minimum similarity:",
    round(float(cosine_similarities.min()), 4)
)

print(
    "Maximum similarity:",
    round(float(cosine_similarities.max()), 4)
)

print(
    "Standard deviation:",
    round(float(cosine_similarities.std()), 4)
)

results_df = twin_manifest.copy()

results_df["cosine_similarity"] = (
    cosine_similarities
)

results_df["threshold"] = THRESHOLD

results_df["quality_status"] = np.where(
    passed,
    "PASS",
    "MANUAL_REVIEW"
)

results_df = results_df.sort_values(
    "cosine_similarity",
    ascending=True
).reset_index(drop=True)

results_df.to_csv(
    SIMILARITY_CSV,
    index=False
)

print("\nDetailed results saved:")
print(SIMILARITY_CSV)

review_df = results_df[
    results_df["cosine_similarity"] < THRESHOLD
]

print("\nLowest similarity images:")

display(
    results_df.head(20)[
        [
            "original_filename",
            "original_embedding_index",
            "cosine_similarity",
            "quality_status",
            "twin_path"
        ]
    ]
)

print("\nImages requiring manual review:", len(review_df))

summary = {

    "dataset": "VERITE",

    "original_model": "SigLIP2",

    "twin_model": "VAE reconstruction + SigLIP2",

    "embedding_dimension": 1152,

    "total_images": 664,

    "threshold": THRESHOLD,

    "passed": int(passed.sum()),

    "manual_review": int(manual_review.sum()),

    "pass_percentage": float(
        passed.mean() * 100
    ),

    "review_percentage": float(
        manual_review.mean() * 100
    ),

    "mean_cosine_similarity": float(
        cosine_similarities.mean()
    ),

    "median_cosine_similarity": float(
        np.median(cosine_similarities)
    ),

    "minimum_cosine_similarity": float(
        cosine_similarities.min()
    ),

    "maximum_cosine_similarity": float(
        cosine_similarities.max()
    ),

    "standard_deviation": float(
        cosine_similarities.std()
    ),

    "alignment_validation": "PASS",

    "results_csv": SIMILARITY_CSV

}

with open(
    SUMMARY_JSON,
    "w"
) as f:

    json.dump(
        summary,
        f,
        indent=4
    )

print("\nSummary saved:")
print(SUMMARY_JSON)

print("\n" + "=" * 60)
print("COSINE SIMILARITY AUDIT COMPLETE")
print("=" * 60)

print("Alignment validation: PASS")

print(
    f"PASS (>= {THRESHOLD}): "
    f"{passed.sum()}/{664}"
)

print(
    f"MANUAL REVIEW (< {THRESHOLD}): "
    f"{manual_review.sum()}/{664}"
)

In [ ]:
import pandas as pd
from pathlib import Path

RESULTS_PATH = (
    "/content/drive/MyDrive/TwinCheck/audit/verite/"
    "vae_twin_similarity/verite_twin_cosine_similarity.csv"
)

assert Path(RESULTS_PATH).exists(), (
    f"Results file not found: {RESULTS_PATH}"
)

results_df = pd.read_csv(RESULTS_PATH)

print("Results successfully loaded!")
print("=" * 50)

print("Number of rows:", len(results_df))
print("\nColumns:")
print(results_df.columns.tolist())

print("\nFirst 5 rows:")
display(results_df.head())

print("\nCosine similarity statistics:")
display(results_df["cosine_similarity"].describe())

In [ ]:
import numpy as np
import pandas as pd
import random
import os
from PIL import Image, ImageDraw
from IPython.display import display

THRESHOLD = 0.92
RANDOM_SEED = 42

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

lowest_10 = results_df.sort_values(
    "cosine_similarity",
    ascending=True
).head(10).copy()

lowest_10["inspection_group"] = "Lowest similarity"

below_threshold = results_df[
    results_df["cosine_similarity"] < THRESHOLD
].copy()

above_threshold = results_df[
    results_df["cosine_similarity"] >= THRESHOLD
].copy()

near_below_5 = below_threshold.sort_values(
    "cosine_similarity",
    ascending=False
).head(5)

near_above_5 = above_threshold.sort_values(
    "cosine_similarity",
    ascending=True
).head(5)

near_below_5["inspection_group"] = (
    "Near threshold - below"
)

near_above_5["inspection_group"] = (
    "Near threshold - above"
)

already_selected_indices = set(
    lowest_10["original_embedding_index"]
).union(
    near_above_5["original_embedding_index"]
)

random_candidates = above_threshold[
    ~above_threshold["original_embedding_index"].isin(
        already_selected_indices
    )
]

random_pass_10 = random_candidates.sample(
    n=10,
    random_state=RANDOM_SEED
).copy()

random_pass_10["inspection_group"] = (
    "Random passing"
)

inspection_df = pd.concat(
    [
        lowest_10,
        near_below_5,
        near_above_5,
        random_pass_10
    ],
    ignore_index=True
)

inspection_df = inspection_df.drop_duplicates(
    subset="original_embedding_index"
)

assert len(inspection_df) == 30

inspection_df = inspection_df.sort_values(
    "cosine_similarity",
    ascending=True
).reset_index(drop=True)

INSPECTION_DIR = (
    "/content/drive/MyDrive/TwinCheck/audit/verite/"
    "vae_twin_similarity/manual_inspection"
)

os.makedirs(
    INSPECTION_DIR,
    exist_ok=True
)

inspection_csv = os.path.join(
    INSPECTION_DIR,
    "verite_30_manual_inspection_selection.csv"
)

inspection_df.to_csv(
    inspection_csv,
    index=False
)

print("Selected images:", len(inspection_df))
print("\nGroup counts:")
print(
    inspection_df["inspection_group"].value_counts()
)

print("\nSaved selection CSV:")
print(inspection_csv)

display(
    inspection_df[
        [
            "original_filename",
            "cosine_similarity",
            "inspection_group",
            "original_path",
            "twin_path"
        ]
    ]
)

CARD_WIDTH = 520
CARD_HEIGHT = 290

THUMB_WIDTH = 240
THUMB_HEIGHT = 210

def create_comparison_card(row):

    card = Image.new(
        "RGB",
        (CARD_WIDTH, CARD_HEIGHT),
        "white"
    )

    draw = ImageDraw.Draw(card)

    filename = str(row["original_filename"])

    score = float(row["cosine_similarity"])

    group = str(row["inspection_group"])

    draw.text(
        (10, 10),
        filename,
        fill="black"
    )

    draw.text(
        (10, 35),
        f"Similarity: {score:.4f}",
        fill="black"
    )

    draw.text(
        (10, 60),
        group,
        fill="black"
    )

    original = Image.open(
        row["original_path"]
    ).convert("RGB")

    twin = Image.open(
        row["twin_path"]
    ).convert("RGB")

    original.thumbnail(
        (THUMB_WIDTH, THUMB_HEIGHT)
    )

    twin.thumbnail(
        (THUMB_WIDTH, THUMB_HEIGHT)
    )

    original_x = 10
    twin_x = 270
    image_y = 80

    card.paste(
        original,
        (
            original_x,
            image_y
        )
    )

    card.paste(
        twin,
        (
            twin_x,
            image_y
        )
    )

    draw.text(
        (original_x, 65),
        "ORIGINAL",
        fill="black"
    )

    draw.text(
        (twin_x, 65),
        "RECONSTRUCTED",
        fill="black"
    )

    return card

cards = []

for _, row in inspection_df.iterrows():

    cards.append(
        create_comparison_card(row)
    )

CARDS_PER_PAGE = 10

SHEET_COLUMNS = 2

SHEET_ROWS = 5

SHEET_WIDTH = CARD_WIDTH * SHEET_COLUMNS

SHEET_HEIGHT = CARD_HEIGHT * SHEET_ROWS

sheet_paths = []

for page_index in range(3):

    start = page_index * CARDS_PER_PAGE

    end = start + CARDS_PER_PAGE

    page_cards = cards[start:end]

    sheet = Image.new(
        "RGB",
        (
            SHEET_WIDTH,
            SHEET_HEIGHT
        ),
        "#eeeeee"
    )

    for i, card in enumerate(page_cards):

        row_position = i // SHEET_COLUMNS

        column_position = i % SHEET_COLUMNS

        x = column_position * CARD_WIDTH

        y = row_position * CARD_HEIGHT

        sheet.paste(
            card,
            (x, y)
        )

    sheet_path = os.path.join(
        INSPECTION_DIR,
        f"manual_inspection_sheet_{page_index + 1}.jpg"
    )

    sheet.save(
        sheet_path,
        quality=95
    )

    sheet_paths.append(sheet_path)

    print("\nSaved:", sheet_path)

    display(sheet)

print("\n" + "="*60)
print("MANUAL INSPECTION SHEETS CREATED")
print("="*60)

In [ ]:
import os
import json
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
from skimage.metrics import structural_similarity as ssim

BASE_DIR = "/content/drive/MyDrive/TwinCheck"

INPUT_CSV = os.path.join(
    BASE_DIR,
    "audit/verite/vae_twin_similarity/"
    "verite_twin_cosine_similarity.csv"
)

OUTPUT_DIR = os.path.join(
    BASE_DIR,
    "audit/verite/vae_twin_similarity/ssim"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

OUTPUT_CSV = os.path.join(
    OUTPUT_DIR,
    "verite_vae_twin_ssim_results.csv"
)

OUTPUT_JSON = os.path.join(
    OUTPUT_DIR,
    "verite_vae_twin_ssim_summary.json"
)

results_df = pd.read_csv(INPUT_CSV)

print("Loaded cosine similarity results:", len(results_df))

assert len(results_df) == 664, (
    f"Expected 664 records, found {len(results_df)}"
)

required_columns = [
    "original_filename",
    "original_path",
    "twin_path",
    "cosine_similarity"
]

for col in required_columns:
    assert col in results_df.columns, (
        f"Required column missing: {col}"
    )

missing_files = []

for _, row in results_df.iterrows():

    if not os.path.exists(row["original_path"]):
        missing_files.append(row["original_path"])

    if not os.path.exists(row["twin_path"]):
        missing_files.append(row["twin_path"])

assert len(missing_files) == 0, (
    f"Missing image files: {missing_files[:10]}"
)

print("Input validation: PASS")
print("All original and twin image paths exist.")

ssim_results = []
failed_results = []

for _, row in tqdm(
    results_df.iterrows(),
    total=len(results_df),
    desc="SSIM evaluation"
):

    filename = row["original_filename"]

    try:

        original = Image.open(
            row["original_path"]
        ).convert("RGB")

        reconstructed = Image.open(
            row["twin_path"]
        ).convert("RGB")

        if original.size != reconstructed.size:

            original = original.resize(
                reconstructed.size,
                Image.Resampling.LANCZOS
            )

        original_array = np.asarray(
            original,
            dtype=np.uint8
        )

        reconstructed_array = np.asarray(
            reconstructed,
            dtype=np.uint8
        )

        if original_array.shape != reconstructed_array.shape:
            raise ValueError(
                f"Shape mismatch: "
                f"{original_array.shape} vs "
                f"{reconstructed_array.shape}"
            )

        score = ssim(
            original_array,
            reconstructed_array,
            channel_axis=2,
            data_range=255
        )

        ssim_results.append({
            "original_filename": filename,
            "original_path": row["original_path"],
            "twin_path": row["twin_path"],
            "cosine_similarity": float(
                row["cosine_similarity"]
            ),
            "ssim": float(score),
            "width": reconstructed.width,
            "height": reconstructed.height,
            "status": "SUCCESS"
        })

    except Exception as error:

        failed_results.append({
            "original_filename": filename,
            "error": str(error)
        })

ssim_df = pd.DataFrame(ssim_results)

print("\n" + "=" * 60)
print("SSIM EVALUATION COMPLETED")
print("=" * 60)

print("Total images:", len(results_df))
print("Successful:", len(ssim_results))
print("Failed:", len(failed_results))

assert len(ssim_results) + len(failed_results) == 664

assert len(ssim_results) > 0

assert ssim_df["ssim"].notna().all()

assert ssim_df["ssim"].between(-1, 1).all()

print("\nValidation: PASS")

print("\nSSIM Statistics:")

display(
    ssim_df["ssim"].describe().to_frame()
)

lowest_20 = ssim_df.sort_values(
    "ssim",
    ascending=True
).head(20)

print("\n20 Lowest SSIM Images:")

display(
    lowest_20[
        [
            "original_filename",
            "cosine_similarity",
            "ssim"
        ]
    ].reset_index(drop=True)
)

highest_20 = ssim_df.sort_values(
    "ssim",
    ascending=False
).head(20)

print("\n20 Highest SSIM Images:")

display(
    highest_20[
        [
            "original_filename",
            "cosine_similarity",
            "ssim"
        ]
    ].reset_index(drop=True)
)

ssim_df.to_csv(
    OUTPUT_CSV,
    index=False
)

print("\nSSIM results saved:")
print(OUTPUT_CSV)

summary = {
    "dataset": "VERITE",
    "total_images": 664,
    "successful": len(ssim_results),
    "failed": len(failed_results),
    "mean_ssim": float(ssim_df["ssim"].mean()),
    "median_ssim": float(ssim_df["ssim"].median()),
    "minimum_ssim": float(ssim_df["ssim"].min()),
    "maximum_ssim": float(ssim_df["ssim"].max()),
    "standard_deviation": float(ssim_df["ssim"].std()),
    "q25": float(ssim_df["ssim"].quantile(0.25)),
    "q75": float(ssim_df["ssim"].quantile(0.75)),
    "failed_images": failed_results
}

with open(OUTPUT_JSON, "w") as f:
    json.dump(summary, f, indent=4)

print("\nSummary JSON saved:")
print(OUTPUT_JSON)

print("\n" + "=" * 60)
print("STEP COMPLETE")
print("=" * 60)

In [ ]:
import os
import json
import pandas as pd
import numpy as np

BASE = "/content/drive/MyDrive/TwinCheck"

COSINE_FILE = os.path.join(
    BASE,
    "audit/verite/vae_twin_similarity/"
    "verite_twin_cosine_similarity.csv"
)

SSIM_FILE = os.path.join(
    BASE,
    "audit/verite/vae_twin_similarity/ssim/"
    "verite_vae_twin_ssim_results.csv"
)

OUT_DIR = os.path.join(
    BASE,
    "audit/verite/vae_twin_similarity/combined_audit"
)

os.makedirs(OUT_DIR, exist_ok=True)

THRESHOLD = 0.92

cosine_df = pd.read_csv(COSINE_FILE)
ssim_df = pd.read_csv(SSIM_FILE)

print("Cosine records:", len(cosine_df))
print("SSIM records:", len(ssim_df))

assert len(cosine_df) == 664
assert len(ssim_df) == 664

combined = cosine_df.merge(
    ssim_df[
        [
            "original_filename",
            "ssim"
        ]
    ],
    on="original_filename",
    how="inner",
    validate="one_to_one"
)

assert len(combined) == 664
assert combined["ssim"].notna().all()

combined["cosine_pass"] = (
    combined["cosine_similarity"] >= THRESHOLD
)

combined["ssim_review_flag"] = (
    combined["ssim"] < 0.50
)

combined["quality_category"] = np.select(
    [
        combined["cosine_pass"] &
        (combined["ssim"] >= 0.80),

        combined["cosine_pass"] &
        (combined["ssim"] >= 0.50) &
        (combined["ssim"] < 0.80),

        combined["cosine_pass"] &
        (combined["ssim"] < 0.50),

        ~combined["cosine_pass"] &
        (combined["ssim"] >= 0.80),

        ~combined["cosine_pass"] &
        (combined["ssim"] >= 0.50),

        ~combined["cosine_pass"] &
        (combined["ssim"] < 0.50),
    ],
    [
        "Semantic PASS + strong structural similarity",
        "Semantic PASS + moderate structural similarity",
        "Semantic PASS + LOW structural similarity - REVIEW",
        "Cosine FAIL + strong structural similarity",
        "Cosine FAIL + moderate structural similarity",
        "Cosine FAIL + LOW structural similarity - REVIEW"
    ],
    default="UNCLASSIFIED"
)

cosine_pass = combined[
    combined["cosine_pass"]
]

cosine_fail = combined[
    ~combined["cosine_pass"]
]

low_ssim = combined[
    combined["ssim_review_flag"]
]

print("\n" + "=" * 60)
print("COMBINED AUDIT")
print("=" * 60)

print("Total images:", len(combined))
print("Cosine passing >= 0.92:", len(cosine_pass))
print("Cosine failing < 0.92:", len(cosine_fail))
print("SSIM below 0.50 (review flag):", len(low_ssim))

print("\nCosine-passing SSIM statistics:")
display(cosine_pass["ssim"].describe())

print("\nCosine-passing category distribution:")
display(
    cosine_pass["quality_category"]
    .value_counts()
    .to_frame("count")
)

review_list = combined.sort_values(
    "ssim",
    ascending=True
).head(30)

print("\n30 LOWEST SSIM IMAGES")
display(
    review_list[
        [
            "original_filename",
            "cosine_similarity",
            "ssim",
            "cosine_pass",
            "quality_category"
        ]
    ].reset_index(drop=True)
)

MASTER_CSV = os.path.join(
    OUT_DIR,
    "verite_combined_cosine_ssim_audit.csv"
)

REVIEW_CSV = os.path.join(
    OUT_DIR,
    "verite_low_ssim_review_list.csv"
)

combined.to_csv(
    MASTER_CSV,
    index=False
)

review_list.to_csv(
    REVIEW_CSV,
    index=False
)

summary = {
    "dataset": "VERITE",
    "total_images": int(len(combined)),
    "cosine_threshold": THRESHOLD,
    "cosine_pass_count": int(len(cosine_pass)),
    "cosine_fail_count": int(len(cosine_fail)),
    "cosine_pass_percentage": float(
        len(cosine_pass) / len(combined) * 100
    ),
    "mean_ssim_all": float(combined["ssim"].mean()),
    "median_ssim_all": float(combined["ssim"].median()),
    "mean_ssim_cosine_pass": float(
        cosine_pass["ssim"].mean()
    ),
    "median_ssim_cosine_pass": float(
        cosine_pass["ssim"].median()
    ),
    "low_ssim_review_flag_threshold": 0.50,
    "low_ssim_review_flag_count": int(len(low_ssim)),
    "low_ssim_flag_is_rejection": False,
    "master_csv": MASTER_CSV,
    "review_csv": REVIEW_CSV
}

SUMMARY_JSON = os.path.join(
    OUT_DIR,
    "verite_combined_cosine_ssim_summary.json"
)

with open(SUMMARY_JSON, "w") as f:
    json.dump(summary, f, indent=4)

print("\nSaved:")
print(MASTER_CSV)
print(REVIEW_CSV)
print(SUMMARY_JSON)

print("\nAUDIT COMPLETE — ORIGINALS AND TWINS UNCHANGED")

In [ ]:
import cv2
import pandas as pd
import numpy as np
from PIL import Image, ImageDraw
from pathlib import Path
from tqdm.auto import tqdm

ROOT = Path("/content/drive/MyDrive/TwinCheck")

IMAGE_MANIFEST = ROOT / "features/vision/verite/verite_embedding_manifest.csv"
OFFICIAL_CSV = ROOT / "data/raw/vision/verite/metadata/VERITE_official.csv"

OUT_DIR = ROOT / "audit/verite/face_exclusion"
OUT_DIR.mkdir(parents=True, exist_ok=True)

manifest = pd.read_csv(IMAGE_MANIFEST)
official = pd.read_csv(OFFICIAL_CSV)

true_rows = official[
    official["label"].astype(str).str.lower().str.strip() == "true"
]

true_names = set(
    Path(str(p)).name
    for p in true_rows["image_path"]
)

manifest["basename"] = manifest["absolute_path"].apply(
    lambda p: Path(str(p)).name
)

true_images = manifest[
    manifest["basename"].isin(true_names)
].drop_duplicates(
    subset="absolute_path"
).reset_index(drop=True)

print("True images to screen:", len(true_images))

cascade_path = (
    cv2.data.haarcascades
    + "haarcascade_frontalface_default.xml"
)

face_detector = cv2.CascadeClassifier(cascade_path)

if face_detector.empty():
    raise RuntimeError("Could not load OpenCV face detector.")

FLAG_AREA_RATIO = 0.04
FLAG_WIDTH_RATIO = 0.15
FLAG_HEIGHT_RATIO = 0.20

records = []

for _, row in tqdm(
    true_images.iterrows(),
    total=len(true_images),
    desc="OpenCV face screening"
):

    path = row["absolute_path"]
    filename = row["basename"]

    try:

        img = cv2.imread(path)

        if img is None:
            raise ValueError("Image could not be read")

        h, w = img.shape[:2]

        gray = cv2.cvtColor(
            img,
            cv2.COLOR_BGR2GRAY
        )

        detected = face_detector.detectMultiScale(
            gray,
            scaleFactor=1.1,
            minNeighbors=5,
            minSize=(25, 25)
        )

        boxes = []
        largest_area = 0
        largest_width = 0
        largest_height = 0

        for (x, y, bw, bh) in detected:

            area_ratio = (bw * bh) / (w * h)
            width_ratio = bw / w
            height_ratio = bh / h

            boxes.append({
                "x": int(x),
                "y": int(y),
                "width": int(bw),
                "height": int(bh),
                "area_ratio": float(area_ratio)
            })

            largest_area = max(
                largest_area,
                area_ratio
            )

            largest_width = max(
                largest_width,
                width_ratio
            )

            largest_height = max(
                largest_height,
                height_ratio
            )

        flagged = (
            largest_area >= FLAG_AREA_RATIO
            or largest_width >= FLAG_WIDTH_RATIO
            or largest_height >= FLAG_HEIGHT_RATIO
        )

        records.append({
            "filename": filename,
            "image_path": path,
            "width": w,
            "height": h,
            "face_count": len(detected),
            "largest_face_area_ratio": largest_area,
            "largest_face_width_ratio": largest_width,
            "largest_face_height_ratio": largest_height,
            "potentially_large_face": flagged,
            "status": "SUCCESS",
            "face_boxes": boxes
        })

    except Exception as e:

        records.append({
            "filename": filename,
            "image_path": path,
            "face_count": -1,
            "largest_face_area_ratio": np.nan,
            "largest_face_width_ratio": np.nan,
            "largest_face_height_ratio": np.nan,
            "potentially_large_face": True,
            "status": "FAILED",
            "error": str(e),
            "face_boxes": []
        })

face_df = pd.DataFrame(records)

print("\n" + "=" * 60)
print("FACE SCREENING SUMMARY")
print("=" * 60)

print("Total screened:", len(face_df))
print("Successful:", (face_df["status"] == "SUCCESS").sum())
print("Failed:", (face_df["status"] != "SUCCESS").sum())
print("Images with detected faces:", (face_df["face_count"] > 0).sum())
print("Potentially large faces flagged:",
      face_df["potentially_large_face"].sum())

display(
    face_df[
        face_df["potentially_large_face"]
    ][[
        "filename",
        "face_count",
        "largest_face_area_ratio",
        "largest_face_width_ratio",
        "largest_face_height_ratio"
    ]].sort_values(
        "largest_face_area_ratio",
        ascending=False
    )
)

audit_csv = OUT_DIR / "verite_true_face_screening.csv"

face_df.drop(
    columns=["face_boxes"],
    errors="ignore"
).to_csv(audit_csv, index=False)

print("\nCSV saved:", audit_csv)

flagged = face_df[
    face_df["potentially_large_face"]
    | (face_df["status"] != "SUCCESS")
].copy()

if len(flagged) == 0:

    print("No images were flagged by this detector.")

else:

    CARD_W = 360
    CARD_H = 300
    COLS = 3
    ROWS_PER_SHEET = 6
    PER_SHEET = COLS * ROWS_PER_SHEET

    total_sheets = int(np.ceil(len(flagged) / PER_SHEET))

    for sheet_idx in range(total_sheets):

        subset = flagged.iloc[
            sheet_idx * PER_SHEET:
            (sheet_idx + 1) * PER_SHEET
        ]

        rows = int(np.ceil(len(subset) / COLS))

        sheet = Image.new(
            "RGB",
            (COLS * CARD_W, rows * CARD_H),
            "white"
        )

        draw = ImageDraw.Draw(sheet)

        for j, (_, r) in enumerate(subset.iterrows()):

            x0 = (j % COLS) * CARD_W
            y0 = (j // COLS) * CARD_H

            try:

                original = Image.open(
                    r["image_path"]
                ).convert("RGB")

                ow, oh = original.size

                scale = min(
                    (CARD_W - 20) / ow,
                    220 / oh
                )

                dw = int(ow * scale)
                dh = int(oh * scale)

                resized = original.resize((dw, dh))

                offset_x = x0 + (CARD_W - dw) // 2
                offset_y = y0 + 35

                sheet.paste(
                    resized,
                    (offset_x, offset_y)
                )

                for box in r.get("face_boxes", []):

                    bx = offset_x + int(box["x"] * scale)
                    by = offset_y + int(box["y"] * scale)
                    bw = int(box["width"] * scale)
                    bh = int(box["height"] * scale)

                    draw.rectangle(
                        [bx, by, bx + bw, by + bh],
                        outline="red",
                        width=3
                    )

                label = (
                    f"{r['filename']} | Faces: {r['face_count']}\n"
                    f"Area ratio: "
                    f"{r.get('largest_face_area_ratio', 0):.3f}"
                )

                draw.text(
                    (x0 + 8, y0 + 5),
                    label,
                    fill="black"
                )

            except Exception as e:

                draw.text(
                    (x0 + 10, y0 + 10),
                    f"{r['filename']}\nERROR: {e}",
                    fill="red"
                )

        sheet_path = OUT_DIR / (
            f"face_review_sheet_{sheet_idx + 1}.jpg"
        )

        sheet.save(
            sheet_path,
            quality=90
        )

        print("Review sheet saved:", sheet_path)

print("\n" + "=" * 60)
print("FACE SCREENING FINISHED")
print("No originals or twins were deleted or modified.")
print("=" * 60)

In [ ]:
from PIL import Image, ImageDraw
import math

flagged = face_df[
    face_df["potentially_large_face"]
].copy()

CARD_W = 320
CARD_H = 230
COLS = 4
ROWS = math.ceil(len(flagged) / COLS)

sheet = Image.new(
    "RGB",
    (COLS * CARD_W, ROWS * CARD_H),
    "white"
)

draw = ImageDraw.Draw(sheet)

for j, (_, r) in enumerate(flagged.iterrows()):

    x0 = (j % COLS) * CARD_W
    y0 = (j // COLS) * CARD_H

    img = Image.open(r["image_path"]).convert("RGB")
    ow, oh = img.size

    scale = min(
        (CARD_W - 20) / ow,
        (CARD_H - 50) / oh
    )

    dw = int(ow * scale)
    dh = int(oh * scale)

    img = img.resize((dw, dh))

    offset_x = x0 + (CARD_W - dw) // 2
    offset_y = y0 + 30

    sheet.paste(img, (offset_x, offset_y))

    for box in r["face_boxes"]:

        bx = offset_x + int(box["x"] * scale)
        by = offset_y + int(box["y"] * scale)
        bw = int(box["width"] * scale)
        bh = int(box["height"] * scale)

        draw.rectangle(
            [bx, by, bx+bw, by+bh],
            outline="red",
            width=2
        )

    draw.text(
        (x0 + 5, y0 + 5),
        f"{r['filename']} | Faces: {r['face_count']}",
        fill="black"
    )

output = OUT_DIR / "ALL_62_FLAGGED_FACE_REVIEW.jpg"

sheet.save(output, quality=95)

print("Saved:", output)
print("Flagged images:", len(flagged))
display(sheet)

In [ ]:
from pathlib import Path
import pandas as pd
import json

ROOT = Path("/content/drive/MyDrive/TwinCheck")

OUT_DIR = ROOT / "audit/verite/face_exclusion"
OUT_DIR.mkdir(parents=True, exist_ok=True)

FACE_EXCLUDE = [
    "true_103.jpg", "true_106.jpg", "true_110.jpg",
    "true_117.jpg", "true_13.jpg", "true_146.jpg",
    "true_15.jpg", "true_164.jpg", "true_18.jpg",
    "true_181.jpg", "true_182.jpg", "true_2.jpg",
    "true_21.jpg", "true_210.jpg", "true_226.jpg",
    "true_231.jpg", "true_234.jpg", "true_24.jpg",
    "true_251.jpg", "true_269.jpg", "true_271.jpg",
    "true_274.jpg", "true_276.jpg", "true_278.jpg",
    "true_28.jpg", "true_292.jpg", "true_299.jpg",
    "true_3.jpg", "true_311.jpg", "true_315.jpg",
    "true_36.jpg", "true_45.jpg", "true_53.jpg",
    "true_57.jpg", "true_58.jpg", "true_59.jpg",
    "true_65.jpg", "true_81.jpg", "true_99.jpg"
]

face_df = pd.read_csv(
    OUT_DIR / "verite_true_face_screening.csv"
)

manifest = pd.read_csv(
    ROOT / "features/vision/verite/verite_embedding_manifest.csv"
)

manifest["basename"] = manifest["absolute_path"].apply(
    lambda p: Path(p).name
)

face_df["manual_decision"] = face_df["filename"].apply(
    lambda name: (
        "EXCLUDE_LARGE_IDENTIFIABLE_FACE"
        if name in FACE_EXCLUDE
        else "RETAIN"
    )
)

missing_exclusions = sorted(
    set(FACE_EXCLUDE) - set(face_df["filename"])
)

assert not missing_exclusions, (
    f"Excluded filenames missing from audit: {missing_exclusions}"
)

decision_csv = OUT_DIR / "verite_manual_face_exclusion_decisions.csv"

face_df.drop(
    columns=["face_boxes"],
    errors="ignore"
).to_csv(decision_csv, index=False)

eligible_true = manifest[
    manifest["basename"].str.startswith("true_")
    & ~manifest["basename"].isin(FACE_EXCLUDE)
].copy()

excluded_true = manifest[
    manifest["basename"].isin(FACE_EXCLUDE)
].copy()

eligible_csv = OUT_DIR / "verite_true_images_face_eligible.csv"
excluded_csv = OUT_DIR / "verite_true_images_face_excluded.csv"

eligible_true.to_csv(eligible_csv, index=False)
excluded_true.to_csv(excluded_csv, index=False)

assert len(excluded_true) == 39
assert len(eligible_true) == 338 - 39

assert all(
    Path(p).exists()
    for p in eligible_true["absolute_path"]
)

assert all(
    Path(p).exists()
    for p in excluded_true["absolute_path"]
)

summary = {
    "true_images_total": 338,
    "manual_face_exclusions": len(excluded_true),
    "eligible_true_images": len(eligible_true),
    "all_excluded_files_exist": True,
    "all_eligible_files_exist": True,
    "originals_modified": False,
    "twins_deleted": False,
    "decision_method": "Manual visual review of flagged contact sheet",
    "excluded_filenames": FACE_EXCLUDE
}

summary_json = OUT_DIR / "verite_face_exclusion_summary.json"

summary_json.write_text(
    json.dumps(summary, indent=2)
)

print("=" * 65)
print("FACE EXCLUSION FINAL AUDIT")
print("=" * 65)

print("Original true images:", 338)
print("Manually excluded:", len(excluded_true))
print("Eligible true images:", len(eligible_true))
print("Excluded files verified:", len(excluded_true))
print("Eligible files verified:", len(eligible_true))
print("Original images modified: NO")
print("Twins deleted: NO")
print("\nVALIDATION: PASS")

print("\nSaved:")
print(decision_csv)
print(eligible_csv)
print(excluded_csv)
print(summary_json)

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import json

ROOT = Path("/content/drive/MyDrive/TwinCheck")

FACE_DIR = ROOT / "audit/verite/face_exclusion"
SIM_DIR = ROOT / "audit/verite/vae_twin_similarity"

eligible_face_file = FACE_DIR / "verite_true_images_face_eligible.csv"

cosine_file = SIM_DIR / "verite_twin_cosine_similarity.csv"

twin_manifest_file = (
    ROOT / "features/vision/verite/"
    "siglip2_vae_twins/twin_embedding_manifest.csv"
)

ssim_file = (
    SIM_DIR / "ssim/verite_vae_twin_ssim_results.csv"
)

OUT_DIR = ROOT / "audit/verite/final_eligible_twins"
OUT_DIR.mkdir(parents=True, exist_ok=True)

THRESHOLD = 0.92

face_eligible = pd.read_csv(eligible_face_file)
cosine_df = pd.read_csv(cosine_file)
twin_manifest = pd.read_csv(twin_manifest_file)
ssim_df = pd.read_csv(ssim_file)

print("Loaded:")
print("Face eligible:", len(face_eligible))
print("Cosine records:", len(cosine_df))
print("Twin manifest:", len(twin_manifest))
print("SSIM records:", len(ssim_df))

print("\nColumns:")
print("Face:", face_eligible.columns.tolist())
print("Cosine:", cosine_df.columns.tolist())
print("Twin:", twin_manifest.columns.tolist())
print("SSIM:", ssim_df.columns.tolist())

assert "row_index" in face_eligible.columns
assert "original_embedding_index" in cosine_df.columns
assert "original_embedding_index" in twin_manifest.columns

assert "cosine_similarity" in cosine_df.columns
assert "twin_path" in twin_manifest.columns

face_eligible["filename"] = face_eligible[
    "absolute_path"
].apply(lambda p: Path(p).name)

assert face_eligible["filename"].str.startswith(
    "true_"
).all()

face_eligible = face_eligible.rename(
    columns={"row_index": "original_embedding_index"}
)

cosine_df["cosine_pass"] = (
    cosine_df["cosine_similarity"] >= THRESHOLD
)

cosine_pass = cosine_df[
    cosine_df["cosine_pass"]
].copy()

selected = face_eligible.merge(
    cosine_pass[
        ["original_embedding_index", "cosine_similarity"]
    ],
    on="original_embedding_index",
    how="inner",
    validate="one_to_one"
)

selected = selected.merge(
    twin_manifest[
        [
            "original_embedding_index",
            "original_filename",
            "twin_path"
        ]
    ],
    on="original_embedding_index",
    how="inner",
    validate="one_to_one"
)

if "original_filename" in ssim_df.columns:

    selected = selected.merge(
        ssim_df[
            ["original_filename", "ssim"]
        ],
        on="original_filename",
        how="left"
    )

elif "filename" in ssim_df.columns:

    selected = selected.merge(
        ssim_df[["filename", "ssim"]],
        left_on="original_filename",
        right_on="filename",
        how="left"
    )

else:
    raise ValueError(
        "Cannot safely align SSIM results. "
        "Check the SSIM CSV columns."
    )

selected["original_exists"] = selected[
    "absolute_path"
].apply(lambda p: Path(p).exists())

selected["twin_exists"] = selected[
    "twin_path"
].apply(lambda p: Path(p).exists())

selected["cosine_pass"] = (
    selected["cosine_similarity"] >= THRESHOLD
)

assert selected["original_exists"].all()
assert selected["twin_exists"].all()
assert selected["cosine_pass"].all()

assert selected["filename"].str.startswith(
    "true_"
).all()

assert selected["original_embedding_index"].is_unique

assert len(selected) == len(
    selected["original_embedding_index"].unique()
)

final_manifest_file = (
    OUT_DIR / "verite_final_eligible_true_vae_twins.csv"
)

selected.to_csv(final_manifest_file, index=False)

summary = {
    "original_true_images": 338,
    "manually_face_excluded": 39,
    "face_eligible_true_images": 299,
    "cosine_threshold": THRESHOLD,
    "final_eligible_true_image_twins": int(len(selected)),
    "all_original_files_exist": bool(
        selected["original_exists"].all()
    ),
    "all_twin_files_exist": bool(
        selected["twin_exists"].all()
    ),
    "all_cosine_scores_pass": bool(
        selected["cosine_pass"].all()
    ),
    "alignment_method": "original_embedding_index",
    "ssim_used_as_hard_exclusion": False,
    "originals_modified": False,
    "twins_modified": False,
    "final_manifest": str(final_manifest_file)
}

summary_file = OUT_DIR / "final_eligible_twins_summary.json"

summary_file.write_text(
    json.dumps(summary, indent=2)
)

print("\n" + "=" * 65)
print("FINAL ELIGIBLE TRUE IMAGE TWIN AUDIT")
print("=" * 65)

print("Face eligible true images:", len(face_eligible))
print("Cosine passing true images:", len(cosine_pass))
print("Final intersection:", len(selected))
print("Original files verified:", selected["original_exists"].sum())
print("Twin files verified:", selected["twin_exists"].sum())
print("Cosine passing:", selected["cosine_pass"].sum())

print("\nSSIM summary for final selected set:")
print(selected["ssim"].describe())

print("\nVALIDATION: PASS")
print("\nSaved:")
print(final_manifest_file)
print(summary_file)

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import json

ROOT = Path("/content/drive/MyDrive/TwinCheck")

FINAL_MANIFEST = (
    ROOT / "audit/verite/final_eligible_twins/"
    "verite_final_eligible_true_vae_twins.csv"
)

ORIGINAL_MANIFEST = (
    ROOT / "features/vision/verite/"
    "verite_embedding_manifest.csv"
)

TWIN_MANIFEST = (
    ROOT / "features/vision/verite/"
    "siglip2_vae_twins/twin_embedding_manifest.csv"
)

COSINE_FILE = (
    ROOT / "audit/verite/vae_twin_similarity/"
    "verite_twin_cosine_similarity.csv"
)

SSIM_FILE = (
    ROOT / "audit/verite/vae_twin_similarity/ssim/"
    "verite_vae_twin_ssim_results.csv"
)

SIGLIP_RAW = (
    ROOT / "features/vision/verite/siglip2_so400m/"
    "embeddings_raw.npy"
)

SIGLIP_NORMALIZED = (
    ROOT / "features/vision/verite/siglip2_so400m/"
    "embeddings_normalized.npy"
)

TWIN_SIGLIP_NORMALIZED = (
    ROOT / "features/vision/verite/siglip2_vae_twins/"
    "embeddings_normalized.npy"
)

CLIP_RAW = (
    ROOT / "features/vision/verite/clip_vitl14/"
    "embeddings_raw.npy"
)

CLIP_NORMALIZED = (
    ROOT / "features/vision/verite/clip_vitl14/"
    "embeddings_normalized.npy"
)

OUT_DIR = (
    ROOT / "features/vision/verite/"
    "final_eligible_true_vae_pairs"
)

OUT_DIR.mkdir(parents=True, exist_ok=True)

selected = pd.read_csv(FINAL_MANIFEST)
original_manifest = pd.read_csv(ORIGINAL_MANIFEST)
twin_manifest = pd.read_csv(TWIN_MANIFEST)

cosine_df = pd.read_csv(COSINE_FILE)
ssim_df = pd.read_csv(SSIM_FILE)

print("Selected pairs:", len(selected))
print("Original manifest:", len(original_manifest))
print("Twin manifest:", len(twin_manifest))

assert "original_embedding_index" in selected.columns
assert "embedding_row" in twin_manifest.columns

assert selected["original_embedding_index"].is_unique

package_manifest = selected.merge(
    twin_manifest[
        [
            "original_embedding_index",
            "embedding_row",
            "original_filename",
            "twin_path"
        ]
    ],
    on="original_embedding_index",
    how="inner",
    suffixes=("", "_twin"),
    validate="one_to_one"
)

assert len(package_manifest) == len(selected)

siglip_raw = np.load(SIGLIP_RAW)
siglip_norm = np.load(SIGLIP_NORMALIZED)

twin_siglip_norm = np.load(TWIN_SIGLIP_NORMALIZED)

clip_raw = np.load(CLIP_RAW)
clip_norm = np.load(CLIP_NORMALIZED)

original_indices = package_manifest[
    "original_embedding_index"
].astype(int).to_numpy()

twin_rows = package_manifest[
    "embedding_row"
].astype(int).to_numpy()

assert original_indices.min() >= 0
assert original_indices.max() < len(siglip_raw)

assert twin_rows.min() >= 0
assert twin_rows.max() < len(twin_siglip_norm)

selected_siglip_raw = siglip_raw[original_indices]
selected_siglip_norm = siglip_norm[original_indices]

selected_clip_raw = clip_raw[original_indices]
selected_clip_norm = clip_norm[original_indices]

selected_twin_siglip_norm = twin_siglip_norm[twin_rows]

n = len(package_manifest)

assert selected_siglip_raw.shape == (n, 1152)
assert selected_siglip_norm.shape == (n, 1152)
assert selected_twin_siglip_norm.shape == (n, 1152)

assert selected_clip_raw.shape == (n, 768)
assert selected_clip_norm.shape == (n, 768)

assert np.isfinite(selected_siglip_raw).all()
assert np.isfinite(selected_siglip_norm).all()
assert np.isfinite(selected_twin_siglip_norm).all()
assert np.isfinite(selected_clip_raw).all()
assert np.isfinite(selected_clip_norm).all()

original_siglip_norms = np.linalg.norm(
    selected_siglip_norm, axis=1
)

twin_siglip_norms = np.linalg.norm(
    selected_twin_siglip_norm, axis=1
)

original_clip_norms = np.linalg.norm(
    selected_clip_norm, axis=1
)

assert np.allclose(
    original_siglip_norms, 1.0, atol=1e-4
)

assert np.allclose(
    twin_siglip_norms, 1.0, atol=1e-4
)

assert np.allclose(
    original_clip_norms, 1.0, atol=1e-4
)

np.save(
    OUT_DIR / "original_siglip2_raw.npy",
    selected_siglip_raw
)

np.save(
    OUT_DIR / "original_siglip2_normalized.npy",
    selected_siglip_norm
)

np.save(
    OUT_DIR / "twin_siglip2_normalized.npy",
    selected_twin_siglip_norm
)

np.save(
    OUT_DIR / "original_clip_raw.npy",
    selected_clip_raw
)

np.save(
    OUT_DIR / "original_clip_normalized.npy",
    selected_clip_norm
)

package_manifest.to_csv(
    OUT_DIR / "final_pair_manifest.csv",
    index=False
)

summary = {
    "dataset": "VERITE",
    "subset": "True images",
    "number_of_selected_pairs": int(n),
    "selection_criteria": {
        "true_images_only": True,
        "manual_face_exclusions_applied": True,
        "cosine_similarity_threshold": 0.92,
        "ssim_used_as_hard_filter": False
    },
    "features": {
        "original_siglip2_raw": list(
            selected_siglip_raw.shape
        ),
        "original_siglip2_normalized": list(
            selected_siglip_norm.shape
        ),
        "twin_siglip2_normalized": list(
            selected_twin_siglip_norm.shape
        ),
        "original_clip_raw": list(
            selected_clip_raw.shape
        ),
        "original_clip_normalized": list(
            selected_clip_norm.shape
        )
    },
    "alignment": {
        "original_index_used": True,
        "twin_embedding_row_used": True,
        "unique_original_indices": bool(
            package_manifest[
                "original_embedding_index"
            ].is_unique
        )
    },
    "normalized_embeddings_verified": True,
    "all_features_finite": True,
    "originals_modified": False,
    "twins_modified": False
}

summary_path = OUT_DIR / "feature_package_audit.json"

summary_path.write_text(
    json.dumps(summary, indent=2)
)

required_files = [
    "original_siglip2_raw.npy",
    "original_siglip2_normalized.npy",
    "twin_siglip2_normalized.npy",
    "original_clip_raw.npy",
    "original_clip_normalized.npy",
    "final_pair_manifest.csv",
    "feature_package_audit.json"
]

missing = [
    f for f in required_files
    if not (OUT_DIR / f).exists()
]

assert not missing, f"Missing output files: {missing}"

print("\n" + "=" * 65)
print("FINAL FEATURE PACKAGE COMPLETE")
print("=" * 65)

print("Selected pairs:", n)
print("Original SigLIP2:", selected_siglip_norm.shape)
print("Twin SigLIP2:", selected_twin_siglip_norm.shape)
print("Original CLIP:", selected_clip_norm.shape)
print("All arrays finite: PASS")
print("Normalized vectors: PASS")
print("Index alignment: PASS")
print("Required files: PASS")

print("\nPackage directory:")
print(OUT_DIR)

print("\nVALIDATION: PASS")

In [ ]:
from pathlib import Path
from zipfile import ZipFile
import pandas as pd
import json
import os

PROJECT_ROOT = Path("/content/drive/MyDrive/TwinCheck")

Y1_ZIP = (
    PROJECT_ROOT
    / "data/raw/vision/m4fc/Y1_outputs.zip"
)

M4FC_AUDIT_DIR = (
    PROJECT_ROOT
    / "audit/y1_m4fc"
)

M4FC_AUDIT_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 70)
print("M4FC ZIP AUDIT")
print("=" * 70)

if not Y1_ZIP.exists():
    raise FileNotFoundError(
        f"ZIP file not found:\n{Y1_ZIP}"
    )

if not Y1_ZIP.is_file():
    raise RuntimeError(
        f"Expected a file but found something else:\n{Y1_ZIP}"
    )

zip_size_mb = Y1_ZIP.stat().st_size / (1024 * 1024)

print("ZIP path:", Y1_ZIP)
print(f"ZIP size: {zip_size_mb:.2f} MB")

with ZipFile(Y1_ZIP, "r") as zf:

    print("\nChecking ZIP integrity...")

    bad_member = zf.testzip()

    if bad_member is not None:
        raise RuntimeError(
            f"ZIP integrity check FAILED.\n"
            f"Corrupted member: {bad_member}"
        )

    print("ZIP integrity: PASS")

    inventory = []

    for info in zf.infolist():

        if info.is_dir():
            continue

        archive_path = info.filename
        extension = Path(archive_path).suffix.lower()

        inventory.append({
            "archive_path": archive_path,
            "filename": Path(archive_path).name,
            "extension": extension,
            "size_bytes": info.file_size,
            "compressed_bytes": info.compress_size,
            "compression_ratio": (
                round(info.compress_size / info.file_size, 4)
                if info.file_size > 0
                else None
            ),
        })

    inventory_df = pd.DataFrame(inventory)

print("\nTotal files inside ZIP:", len(inventory_df))

print("\nFile inventory:")
display(inventory_df)

extension_summary = (
    inventory_df["extension"]
    .value_counts(dropna=False)
    .rename_axis("extension")
    .reset_index(name="count")
)

print("\nFile extension summary:")
display(extension_summary)

IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".tif",
    ".tiff",
    ".webp"
}

image_inventory = inventory_df[
    inventory_df["extension"].isin(IMAGE_EXTENSIONS)
].copy()

print("\nImage files:", len(image_inventory))

display(image_inventory)

manifest_inventory = inventory_df[
    inventory_df["extension"].isin({
        ".csv",
        ".json",
        ".jsonl",
        ".txt",
        ".tsv",
        ".xlsx"
    })
].copy()

print("\nPotential manifest / metadata files:")
display(manifest_inventory)

inventory_csv = (
    M4FC_AUDIT_DIR
    / "Y1_outputs_zip_inventory.csv"
)

extension_csv = (
    M4FC_AUDIT_DIR
    / "Y1_outputs_zip_extension_summary.csv"
)

image_csv = (
    M4FC_AUDIT_DIR
    / "Y1_outputs_zip_image_inventory.csv"
)

manifest_csv = (
    M4FC_AUDIT_DIR
    / "Y1_outputs_zip_metadata_inventory.csv"
)

inventory_df.to_csv(inventory_csv, index=False)
extension_summary.to_csv(extension_csv, index=False)
image_inventory.to_csv(image_csv, index=False)
manifest_inventory.to_csv(manifest_csv, index=False)

audit_summary = {
    "zip_path": str(Y1_ZIP),
    "zip_size_bytes": Y1_ZIP.stat().st_size,
    "zip_integrity": "PASS",
    "total_files": len(inventory_df),
    "total_image_files": len(image_inventory),
    "total_metadata_files": len(manifest_inventory),
    "extension_counts": {
        str(row["extension"]): int(row["count"])
        for _, row in extension_summary.iterrows()
    }
}

summary_json = (
    M4FC_AUDIT_DIR
    / "Y1_outputs_zip_audit_summary.json"
)

with open(summary_json, "w") as f:
    json.dump(audit_summary, f, indent=4)

print("\n" + "=" * 70)
print("M4FC ZIP INVENTORY COMPLETE")
print("=" * 70)

print("ZIP integrity: PASS")
print("Total files:", len(inventory_df))
print("Image files:", len(image_inventory))
print("Metadata files:", len(manifest_inventory))

print("\nSaved audit files:")
print(inventory_csv)
print(extension_csv)
print(image_csv)
print(manifest_csv)
print(summary_json)

In [ ]:
from pathlib import Path
from zipfile import ZipFile
import hashlib
import pandas as pd
import numpy as np
from PIL import Image
import json
import os

PROJECT_ROOT = Path("/content/drive/MyDrive/TwinCheck")

Y1_ZIP = (
    PROJECT_ROOT
    / "data/raw/vision/m4fc/Y1_outputs.zip"
)

M4FC_AUDIT_DIR = (
    PROJECT_ROOT
    / "audit/y1_m4fc"
)

M4FC_STAGE = (
    M4FC_AUDIT_DIR
    / "extracted_m4fc_images"
)

M4FC_STAGE.mkdir(parents=True, exist_ok=True)

print("=" * 70)
print("M4FC SAFE EXTRACTION AND AUDIT")
print("=" * 70)

IMAGE_PREFIX = "vision/m4fc_images/"

with ZipFile(Y1_ZIP, "r") as zf:

    selected_members = [
        info for info in zf.infolist()
        if not info.is_dir()
        and info.filename.startswith(IMAGE_PREFIX)
    ]

print("Selected files:", len(selected_members))

if len(selected_members) != 604:
    print("WARNING: Expected 604 files.")
    print("Actual selected files:", len(selected_members))

extracted_paths = []

with ZipFile(Y1_ZIP, "r") as zf:

    for info in selected_members:

        archive_name = info.filename

        relative_path = archive_name[len(IMAGE_PREFIX):]

        if not relative_path:
            continue

        destination = M4FC_STAGE / relative_path

        destination_resolved = destination.resolve()
        stage_resolved = M4FC_STAGE.resolve()

        if not (
            destination_resolved == stage_resolved
            or stage_resolved in destination_resolved.parents
        ):
            raise RuntimeError(
                f"Unsafe archive path detected: {archive_name}"
            )

        if destination.exists():
            raise FileExistsError(
                f"Extraction would overwrite an existing file:\n"
                f"{destination}\n"
                "Stop and inspect the staging directory."
            )

        destination.parent.mkdir(
            parents=True,
            exist_ok=True
        )

        with zf.open(info, "r") as source:
            with open(destination, "wb") as target:
                target.write(source.read())

        extracted_paths.append(destination)

print("Extraction complete.")
print("Files extracted:", len(extracted_paths))
print("Staging directory:", M4FC_STAGE)

image_paths = sorted(
    [
        p for p in M4FC_STAGE.rglob("*")
        if p.is_file()
        and p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"}
    ]
)

manifest_path = M4FC_STAGE / "manifest.csv"

print("\nExtracted image count:", len(image_paths))
print("Manifest exists:", manifest_path.exists())

if not manifest_path.exists():
    raise FileNotFoundError(
        "manifest.csv was not extracted."
    )

if len(image_paths) != 603:
    raise RuntimeError(
        f"Expected 603 images, found {len(image_paths)}."
    )

def sha256_file(path, chunk_size=1024 * 1024):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        for chunk in iter(
            lambda: f.read(chunk_size),
            b""
        ):
            h.update(chunk)

    return h.hexdigest()

image_audit = []

missing_or_unreadable = []
metadata_errors = []

for path in image_paths:

    relative_path = str(path.relative_to(M4FC_STAGE))

    try:

        with Image.open(path) as im:

            image_format = im.format
            width, height = im.size
            mode = im.mode

            im.verify()

        with Image.open(path) as im:

            im.load()

        file_hash = sha256_file(path)

        image_audit.append({
            "relative_path": relative_path,
            "filename": path.name,
            "extension": path.suffix.lower(),
            "format": image_format,
            "width": width,
            "height": height,
            "mode": mode,
            "size_bytes": path.stat().st_size,
            "sha256": file_hash,
            "readable": True,
        })

    except Exception as e:

        missing_or_unreadable.append({
            "path": str(path),
            "error": repr(e)
        })

image_audit_df = pd.DataFrame(image_audit)

print("\nImage validation results")
print("-" * 50)
print("Images successfully decoded:", len(image_audit_df))
print("Unreadable/corrupt images:", len(missing_or_unreadable))

if missing_or_unreadable:
    display(pd.DataFrame(missing_or_unreadable))

if not image_audit_df.empty:

    print("\nImage format counts:")
    display(
        image_audit_df["format"]
        .value_counts()
        .rename_axis("format")
        .reset_index(name="count")
    )

    print("\nImage dimensions:")
    display(
        image_audit_df.groupby(
            ["width", "height"]
        ).size().reset_index(name="count")
    )

    print("\nImage color modes:")
    display(
        image_audit_df["mode"]
        .value_counts()
        .rename_axis("mode")
        .reset_index(name="count")
    )

duplicate_filename_count = (
    image_audit_df["filename"].duplicated().sum()
)

duplicate_hash_count = (
    image_audit_df["sha256"].duplicated().sum()
)

print("\nDuplicate checks")
print("-" * 50)
print("Duplicate filenames:", duplicate_filename_count)
print("Duplicate SHA-256 image contents:", duplicate_hash_count)

duplicate_hash_rows = image_audit_df[
    image_audit_df["sha256"].duplicated(keep=False)
].sort_values("sha256")

if len(duplicate_hash_rows) > 0:

    print("\nDuplicate-content image records:")
    display(duplicate_hash_rows)

manifest = pd.read_csv(manifest_path)

print("\nManifest loaded")
print("-" * 50)
print("Manifest rows:", len(manifest))
print("Manifest columns:")
print(manifest.columns.tolist())

print("\nManifest sample:")
display(manifest.head(10))

print("\nManifest data types:")
display(
    manifest.dtypes.astype(str)
    .rename("dtype")
    .reset_index()
    .rename(columns={"index": "column"})
)

possible_path_columns = [
    c for c in manifest.columns
    if any(
        term in c.lower()
        for term in [
            "path",
            "file",
            "image",
            "filename",
        ]
    )
]

possible_id_columns = [
    c for c in manifest.columns
    if any(
        term in c.lower()
        for term in [
            "id",
            "claim",
            "image",
        ]
    )
]

print("\nPotential path columns:")
print(possible_path_columns)

print("\nPotential ID columns:")
print(possible_id_columns)

extracted_filenames = {
    p.name for p in image_paths
}

mapping_results = {}

for col in possible_path_columns:

    values = manifest[col].dropna().astype(str)

    mapped_names = []

    for value in values:

        candidate_name = Path(value).name

        mapped_names.append(
            candidate_name in extracted_filenames
        )

    mapping_results[col] = {
        "rows_checked": len(values),
        "mapped_rows": int(sum(mapped_names)),
        "unmapped_rows": int(len(mapped_names) - sum(mapped_names)),
        "mapping_rate": (
            float(np.mean(mapped_names))
            if mapped_names
            else None
        )
    }

print("\nPath mapping checks:")
print(json.dumps(mapping_results, indent=4))

image_audit_csv = (
    M4FC_AUDIT_DIR
    / "m4fc_extracted_image_audit.csv"
)

manifest_copy_csv = (
    M4FC_AUDIT_DIR
    / "m4fc_extracted_manifest_copy.csv"
)

manifest_columns_txt = (
    M4FC_AUDIT_DIR
    / "m4fc_manifest_columns.txt"
)

image_audit_df.to_csv(
    image_audit_csv,
    index=False
)

manifest.to_csv(
    manifest_copy_csv,
    index=False
)

with open(manifest_columns_txt, "w") as f:

    f.write("\n".join(manifest.columns.tolist()))

summary = {
    "zip_path": str(Y1_ZIP),
    "staging_directory": str(M4FC_STAGE),
    "extracted_file_count": len(extracted_paths),
    "image_count": len(image_paths),
    "manifest_rows": len(manifest),
    "image_decode_success_count": len(image_audit_df),
    "unreadable_images": len(missing_or_unreadable),
    "duplicate_filenames": int(duplicate_filename_count),
    "duplicate_image_sha256": int(duplicate_hash_count),
    "manifest_columns": manifest.columns.tolist(),
    "potential_path_columns": possible_path_columns,
    "potential_id_columns": possible_id_columns,
    "path_mapping_results": mapping_results,
}

summary_path = (
    M4FC_AUDIT_DIR
    / "m4fc_extraction_audit_summary.json"
)

with open(summary_path, "w") as f:
    json.dump(summary, f, indent=4)

print("\n" + "=" * 70)
print("M4FC EXTRACTION AND AUDIT SUMMARY")
print("=" * 70)

print("Extracted files:", len(extracted_paths))
print("Images found:", len(image_paths))
print("Images successfully decoded:", len(image_audit_df))
print("Manifest rows:", len(manifest))
print("Unreadable images:", len(missing_or_unreadable))
print("Duplicate filenames:", duplicate_filename_count)
print("Duplicate image contents:", duplicate_hash_count)

print("\nAudit files saved:")
print(image_audit_csv)
print(manifest_copy_csv)
print(manifest_columns_txt)
print(summary_path)

print("\nSTAGING ONLY — NOT YET MOVED TO FINAL DATA DIRECTORY.")

In [ ]:
from pathlib import Path
import pandas as pd
import hashlib
import shutil
import json
from PIL import Image

PROJECT_ROOT = Path("/content/drive/MyDrive/TwinCheck")

M4FC_AUDIT_DIR = PROJECT_ROOT / "audit/y1_m4fc"

M4FC_STAGE = (
    M4FC_AUDIT_DIR
    / "extracted_m4fc_images"
)

SOURCE_MANIFEST = M4FC_STAGE / "manifest.csv"

CLEAN_M4FC_DIR = (
    PROJECT_ROOT
    / "data/processed/vision/m4fc_images_f2_clean"
)

CLEAN_MANIFEST_PATH = (
    CLEAN_M4FC_DIR
    / "m4fc_f2_clean_manifest.csv"
)

if CLEAN_M4FC_DIR.exists():

    raise FileExistsError(
        f"Clean dataset directory already exists:\n"
        f"{CLEAN_M4FC_DIR}\n\n"
        "This cell will not overwrite existing data. "
        "Inspect the directory before deciding what to do."
    )

CLEAN_M4FC_DIR.mkdir(
    parents=True,
    exist_ok=False
)

manifest = pd.read_csv(SOURCE_MANIFEST)

required_columns = [
    "id",
    "image_path",
    "claim_language",
    "split",
    "status",
    "url_used",
]

missing_columns = [
    c for c in required_columns
    if c not in manifest.columns
]

if missing_columns:

    raise ValueError(
        f"Manifest missing required columns: {missing_columns}"
    )

clean_manifest = manifest[
    manifest["status"].astype(str).str.lower() == "ok"
].copy()

failed_manifest = manifest[
    manifest["status"].astype(str).str.lower() != "ok"
].copy()

print("Manifest total:", len(manifest))
print("Successful records:", len(clean_manifest))
print("Failed records:", len(failed_manifest))

if len(clean_manifest) != 603:

    raise RuntimeError(
        f"Expected 603 successful records; found {len(clean_manifest)}"
    )

clean_manifest["source_filename"] = (
    clean_manifest["image_path"]
    .apply(lambda x: Path(str(x)).name)
)

source_images = {
    p.name: p
    for p in M4FC_STAGE.rglob("*")
    if p.is_file()
    and p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"}
}

missing_source_images = []

for row in clean_manifest.itertuples(index=False):

    filename = row.source_filename

    if filename not in source_images:

        missing_source_images.append(filename)

if missing_source_images:

    raise RuntimeError(
        "Some successful manifest records do not have source images:\n"
        f"{missing_source_images[:20]}"
    )

if clean_manifest["id"].duplicated().any():

    duplicates = clean_manifest[
        clean_manifest["id"].duplicated(keep=False)
    ]

    display(duplicates)

    raise RuntimeError(
        "Duplicate IDs detected in successful records."
    )

if clean_manifest["image_path"].duplicated().any():

    duplicates = clean_manifest[
        clean_manifest["image_path"].duplicated(keep=False)
    ]

    display(duplicates)

    raise RuntimeError(
        "Duplicate image paths detected in successful records."
    )

def sha256_file(path, chunk_size=1024 * 1024):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        for chunk in iter(
            lambda: f.read(chunk_size),
            b""
        ):

            h.update(chunk)

    return h.hexdigest()

copy_records = []

for row in clean_manifest.itertuples(index=False):

    source_path = source_images[row.source_filename]

    destination_filename = f"{row.id}.jpg"

    destination_path = (
        CLEAN_M4FC_DIR
        / destination_filename
    )

    if destination_path.exists():

        raise FileExistsError(
            f"Unexpected existing destination:\n"
            f"{destination_path}"
        )

    with Image.open(source_path) as im:

        im.verify()

    with Image.open(source_path) as im:

        im.load()

        image_format = im.format
        width, height = im.size
        mode = im.mode

    if image_format != "JPEG":

        raise RuntimeError(
            f"Unexpected image format for {source_path}: "
            f"{image_format}"
        )

    if mode != "RGB":

        raise RuntimeError(
            f"Unexpected image mode for {source_path}: "
            f"{mode}"
        )

    shutil.copy2(
        source_path,
        destination_path
    )

    source_sha256 = sha256_file(source_path)
    copied_sha256 = sha256_file(destination_path)

    if source_sha256 != copied_sha256:

        raise RuntimeError(
            f"SHA-256 mismatch after copy:\n"
            f"Source: {source_path}\n"
            f"Destination: {destination_path}"
        )

    copy_records.append({

        "id": row.id,
        "source_path": str(source_path),
        "clean_image_path": str(destination_path),
        "filename": destination_filename,
        "claim_language": row.claim_language,
        "split": row.split,
        "status": row.status,
        "url_used": row.url_used,
        "format": image_format,
        "width": width,
        "height": height,
        "mode": mode,
        "size_bytes": destination_path.stat().st_size,
        "source_sha256": source_sha256,
        "clean_sha256": copied_sha256,
        "copy_verified": True,
    })

    if len(copy_records) % 100 == 0:

        print(
            f"Verified copies: {len(copy_records)} / "
            f"{len(clean_manifest)}"
        )

clean_df = pd.DataFrame(copy_records)

if len(clean_df) != 603:

    raise RuntimeError(
        f"Expected 603 clean image records; found {len(clean_df)}"
    )

if not clean_df["copy_verified"].all():

    raise RuntimeError(
        "Not all copied files passed hash verification."
    )

if clean_df["id"].duplicated().any():

    raise RuntimeError(
        "Duplicate IDs in final clean manifest."
    )

if clean_df["clean_image_path"].duplicated().any():

    raise RuntimeError(
        "Duplicate clean image paths."
    )

missing_final_files = [
    p for p in clean_df["clean_image_path"]
    if not Path(p).is_file()
]

if missing_final_files:

    raise RuntimeError(
        f"Missing clean files: {missing_final_files[:10]}"
    )

split_summary = (
    clean_df.groupby("split")
    .size()
    .reset_index(name="image_count")
)

print("\nClean dataset split counts:")
display(split_summary)

expected_split_counts = {
    "train": 363,
    "dev": 70,
    "test": 170,
}

actual_split_counts = dict(
    zip(
        split_summary["split"],
        split_summary["image_count"]
    )
)

for split, expected_count in expected_split_counts.items():

    actual_count = actual_split_counts.get(split, 0)

    if actual_count != expected_count:

        raise RuntimeError(
            f"Split count mismatch for {split}: "
            f"expected {expected_count}, got {actual_count}"
        )

clean_df.to_csv(
    CLEAN_MANIFEST_PATH,
    index=False
)

failed_records_path = (
    M4FC_AUDIT_DIR
    / "m4fc_failed_download_records.csv"
)

failed_manifest.to_csv(
    failed_records_path,
    index=False
)

clean_summary = {

    "dataset": "M4FC",
    "creation_method": "Fresh extraction and verified copy",
    "source_zip": str(
        PROJECT_ROOT
        / "data/raw/vision/m4fc/Y1_outputs.zip"
    ),

    "total_original_manifest_records": int(len(manifest)),
    "successful_image_records": int(len(clean_df)),
    "failed_download_records": int(len(failed_manifest)),

    "image_format": "JPEG",
    "image_mode": "RGB",
    "image_dimensions": "Original dimensions preserved",

    "train_count": int(
        (clean_df["split"] == "train").sum()
    ),

    "dev_count": int(
        (clean_df["split"] == "dev").sum()
    ),

    "test_count": int(
        (clean_df["split"] == "test").sum()
    ),

    "all_copies_hash_verified": bool(
        clean_df["copy_verified"].all()
    ),

    "all_files_exist": bool(
        len(missing_final_files) == 0
    ),

    "clean_manifest": str(CLEAN_MANIFEST_PATH),
    "failed_records_manifest": str(failed_records_path),
}

summary_path = (
    M4FC_AUDIT_DIR
    / "m4fc_f2_clean_dataset_summary.json"
)

with open(summary_path, "w") as f:

    json.dump(
        clean_summary,
        f,
        indent=4
    )

assert len(clean_df) == 603
assert clean_df["copy_verified"].all()
assert not clean_df["id"].duplicated().any()
assert not clean_df["clean_image_path"].duplicated().any()
assert len(missing_final_files) == 0
assert actual_split_counts == expected_split_counts

print("\n" + "=" * 70)
print("M4FC CLEAN DATASET CREATION — PASS")
print("=" * 70)

print("Images copied:", len(clean_df))
print("Train:", actual_split_counts["train"])
print("Dev:", actual_split_counts["dev"])
print("Test:", actual_split_counts["test"])
print("Failed downloads excluded:", len(failed_manifest))
print("Hash verification: PASS")
print("File existence verification: PASS")
print("ID uniqueness: PASS")
print("Split counts: PASS")

print("\nClean dataset directory:")
print(CLEAN_M4FC_DIR)

print("\nClean manifest:")
print(CLEAN_MANIFEST_PATH)

print("\nFailed records audit:")
print(failed_records_path)

print("\nSummary:")
print(summary_path)

In [ ]:
from pathlib import Path
import pandas as pd
import hashlib
import json
from PIL import Image

PROJECT_ROOT = Path("/content/drive/MyDrive/TwinCheck")

CLEAN_M4FC_DIR = (
    PROJECT_ROOT
    / "data/processed/vision/m4fc_images_f2_clean"
)

CLEAN_MANIFEST_PATH = (
    CLEAN_M4FC_DIR
    / "m4fc_f2_clean_manifest.csv"
)

AUDIT_DIR = PROJECT_ROOT / "audit/y1_m4fc"
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

if not CLEAN_MANIFEST_PATH.is_file():
    raise FileNotFoundError(CLEAN_MANIFEST_PATH)

df = pd.read_csv(CLEAN_MANIFEST_PATH)

print("Manifest rows:", len(df))
print("Columns:", df.columns.tolist())

required_columns = [
    "id",
    "clean_image_path",
    "split",
    "claim_language",
    "clean_sha256",
    "copy_verified",
]

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_columns:
    raise RuntimeError(
        f"Missing columns: {missing_columns}"
    )

assert len(df) == 603, f"Expected 603 rows, got {len(df)}"

assert not df["id"].duplicated().any(), \
    "Duplicate IDs found"

assert not df["clean_image_path"].duplicated().any(), \
    "Duplicate image paths found"

assert df["copy_verified"].astype(bool).all(), \
    "Some copies were not marked verified"

def sha256_file(path, chunk_size=1024 * 1024):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        for chunk in iter(
            lambda: f.read(chunk_size),
            b""
        ):
            h.update(chunk)

    return h.hexdigest()

missing_files = []
hash_mismatches = []
image_errors = []
outside_directory = []

observed_hashes = []

clean_root = CLEAN_M4FC_DIR.resolve()

for row in df.itertuples(index=False):

    path = Path(row.clean_image_path)

    resolved_path = path.resolve()

    if not (
        resolved_path == clean_root
        or clean_root in resolved_path.parents
    ):

        outside_directory.append(str(path))
        continue

    if not path.is_file():

        missing_files.append(str(path))
        continue

    try:

        with Image.open(path) as im:

            image_format = im.format
            width, height = im.size
            mode = im.mode

            im.verify()

        with Image.open(path) as im:

            im.load()

        if image_format != "JPEG":

            image_errors.append(
                (str(path), "Unexpected format", image_format)
            )

        if mode != "RGB":

            image_errors.append(
                (str(path), "Unexpected mode", mode)
            )

    except Exception as e:

        image_errors.append(
            (str(path), repr(e))
        )

        continue

    observed_hash = sha256_file(path)

    observed_hashes.append(observed_hash)

    expected_hash = str(row.clean_sha256)

    if observed_hash != expected_hash:

        hash_mismatches.append({
            "id": row.id,
            "path": str(path),
            "expected_sha256": expected_hash,
            "observed_sha256": observed_hash,
        })

hash_series = pd.Series(observed_hashes)

duplicate_hashes = (
    hash_series[hash_series.duplicated(keep=False)]
    .tolist()
)

split_counts = (
    df["split"]
    .value_counts()
    .sort_index()
)

expected_counts = {
    "dev": 70,
    "test": 170,
    "train": 363,
}

actual_counts = {
    str(split): int(count)
    for split, count in split_counts.items()
}

print("\n" + "=" * 70)
print("FINAL CLEAN M4FC AUDIT")
print("=" * 70)

print("Rows:", len(df))
print("Missing files:", len(missing_files))
print("Image errors:", len(image_errors))
print("SHA-256 mismatches:", len(hash_mismatches))
print("Files outside clean directory:", len(outside_directory))
print("Duplicate contents:", len(duplicate_hashes))

print("\nSplit counts:")
display(
    split_counts
    .rename_axis("split")
    .reset_index(name="count")
)

assert len(df) == 603
assert len(missing_files) == 0
assert len(image_errors) == 0
assert len(hash_mismatches) == 0
assert len(outside_directory) == 0
assert len(duplicate_hashes) == 0
assert actual_counts == expected_counts

final_audit = {
    "dataset": "M4FC",
    "final_audit_status": "PASS",
    "image_count": int(len(df)),
    "missing_files": int(len(missing_files)),
    "image_errors": int(len(image_errors)),
    "sha256_mismatches": int(len(hash_mismatches)),
    "outside_directory_paths": int(len(outside_directory)),
    "duplicate_image_contents": int(len(duplicate_hashes)),
    "split_counts": actual_counts,
    "clean_manifest": str(CLEAN_MANIFEST_PATH),
    "clean_directory": str(CLEAN_M4FC_DIR),
}

final_audit_path = (
    AUDIT_DIR
    / "m4fc_final_clean_dataset_audit.json"
)

with open(final_audit_path, "w") as f:

    json.dump(
        final_audit,
        f,
        indent=4
    )

print("\n" + "=" * 70)
print("M4FC FINAL CLEAN DATA AUDIT — PASS")
print("=" * 70)

print("603 images verified")
print("All image files readable")
print("All SHA-256 hashes match clean_sha256")
print("No duplicate image contents")
print("All files within clean dataset directory")
print("Train/Dev/Test counts verified")

print("\nAudit saved:")
print(final_audit_path)

In [ ]:
import os
import gc
import json
import time
import hashlib
import warnings

from pathlib import Path

import numpy as np
import pandas as pd
import torch

from PIL import Image
from tqdm.auto import tqdm

from transformers import (
    SiglipModel,
    SiglipProcessor,
    CLIPModel,
    CLIPProcessor,
)

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/TwinCheck"
)

DATA_DIR = (
    PROJECT_ROOT
    / "data/processed/vision/m4fc_images_f2_clean"
)

MANIFEST_PATH = (
    DATA_DIR
    / "m4fc_f2_clean_manifest.csv"
)

OUTPUT_ROOT = (
    PROJECT_ROOT
    / "features/vision/m4fc"
)

AUDIT_DIR = (
    PROJECT_ROOT
    / "audit/y1_m4fc"
)

OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

AUDIT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

SIGLIP_ID = (
    "google/siglip2-so400m-patch14-384"
)

CLIP_ID = (
    "openai/clip-vit-large-patch14"
)

SIGLIP_INITIAL_BATCH_SIZE = 4
CLIP_INITIAL_BATCH_SIZE = 8

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

MODEL_DTYPE = (
    torch.float16
    if DEVICE.type == "cuda"
    else torch.float32
)

print("=" * 70)
print("F2 M4FC FULL EMBEDDING EXTRACTION")
print("=" * 70)

print("Device:", DEVICE)
print("Model dtype:", MODEL_DTYPE)

if torch.cuda.is_available():

    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

    print(
        "GPU memory:",
        round(
            torch.cuda.get_device_properties(0)
            .total_memory / (1024 ** 3),
            2
        ),
        "GB"
    )

if not MANIFEST_PATH.exists():

    raise FileNotFoundError(
        f"Manifest not found: {MANIFEST_PATH}"
    )

df = pd.read_csv(MANIFEST_PATH)

required_columns = [
    "id",
    "clean_image_path",
    "split",
    "claim_language",
    "clean_sha256",
    "copy_verified",
]

missing_columns = [
    col
    for col in required_columns
    if col not in df.columns
]

if missing_columns:

    raise RuntimeError(
        f"Missing manifest columns: {missing_columns}"
    )

if len(df) != 603:

    raise RuntimeError(
        f"Expected 603 images, found {len(df)}"
    )

if df["id"].duplicated().any():

    raise RuntimeError(
        "Duplicate image IDs found."
    )

if df["clean_image_path"].duplicated().any():

    raise RuntimeError(
        "Duplicate clean image paths found."
    )

if not df["copy_verified"].astype(bool).all():

    raise RuntimeError(
        "Some clean image copies are not verified."
    )

expected_split_counts = {
    "train": 363,
    "dev": 70,
    "test": 170,
}

actual_split_counts = {
    str(k): int(v)
    for k, v in df["split"].value_counts().items()
}

if actual_split_counts != expected_split_counts:

    raise RuntimeError(
        "Unexpected split counts:\n"
        f"{actual_split_counts}"
    )

print("\nInput manifest checks PASS")
print("Images:", len(df))
print("Splits:", actual_split_counts)

SIGLIP_OUTPUT_DIR = (
    OUTPUT_ROOT
    / "siglip2_so400m"
)

CLIP_OUTPUT_DIR = (
    OUTPUT_ROOT
    / "clip_vitl14"
)

SIGLIP_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

CLIP_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

expected_output_names = [
    "image_embeddings_raw.npy",
    "image_embeddings_normalized.npy",
    "embedding_manifest.csv",
    "metadata.json",
]

for output_dir in [
    SIGLIP_OUTPUT_DIR,
    CLIP_OUTPUT_DIR,
]:

    existing_outputs = [
        output_dir / name
        for name in expected_output_names
        if (output_dir / name).exists()
    ]

    if existing_outputs:

        raise RuntimeError(
            "Existing output files detected. "
            "This cell will not overwrite them.\n"
            f"Directory: {output_dir}\n"
            f"Files: {existing_outputs}\n"
            "Rename/archive the existing outputs before "
            "running a new extraction."
        )

print("\nOutput safety check PASS")

def load_image_batch(paths):

    images = []

    for path in paths:

        path = Path(path)

        if not path.exists():

            raise FileNotFoundError(path)

        with Image.open(path) as im:

            images.append(
                im.convert("RGB").copy()
            )

    return images

def extract_siglip_features(
    model,
    pixel_values
):

    output = model.get_image_features(
        pixel_values=pixel_values
    )

    if isinstance(output, torch.Tensor):

        features = output

    elif hasattr(output, "pooler_output"):

        features = output.pooler_output

        if hasattr(model, "visual_projection"):

            projection = model.visual_projection

            if isinstance(
                projection,
                torch.nn.Module
            ):

                features = projection(features)

    elif hasattr(output, "last_hidden_state"):

        features = output.last_hidden_state[:, 0, :]

    else:

        raise TypeError(
            "Unexpected SigLIP output type: "
            f"{type(output)}"
        )

    return features.float()

def extract_clip_features(
    model,
    pixel_values
):

    output = model.get_image_features(
        pixel_values=pixel_values
    )

    if isinstance(output, torch.Tensor):

        features = output

    elif hasattr(output, "pooler_output"):

        features = output.pooler_output

    elif hasattr(output, "last_hidden_state"):

        features = output.last_hidden_state[:, 0, :]

    else:

        raise TypeError(
            "Unexpected CLIP output type: "
            f"{type(output)}"
        )

    return features.float()

def extract_dataset_embeddings(
    model,
    processor,
    dataset_df,
    output_dir,
    model_name,
    feature_function,
    initial_batch_size,
):

    n = len(dataset_df)

    batch_size = initial_batch_size

    raw_embeddings = None

    normalized_embeddings = None

    cursor = 0

    successful_batches = 0

    start_time = time.time()

    progress = tqdm(
        total=n,
        desc=f"Extracting {model_name}"
    )

    while cursor < n:

        current_end = min(
            cursor + batch_size,
            n
        )

        batch_df = dataset_df.iloc[
            cursor:current_end
        ]

        batch_paths = batch_df[
            "clean_image_path"
        ].tolist()

        try:

            images = load_image_batch(
                batch_paths
            )

            inputs = processor(
                images=images,
                return_tensors="pt"
            )

            pixel_values = inputs[
                "pixel_values"
            ].to(
                device=DEVICE,
                dtype=MODEL_DTYPE
            )

            with torch.inference_mode():

                features = feature_function(
                    model,
                    pixel_values
                )

                features = features.float()

                if features.ndim != 2:

                    raise RuntimeError(
                        "Embedding output must be 2D. "
                        f"Received shape: {features.shape}"
                    )

                if features.shape[0] != len(batch_df):

                    raise RuntimeError(
                        "Batch embedding count mismatch."
                    )

                if not torch.isfinite(features).all():

                    raise RuntimeError(
                        "NaN or infinite embeddings detected."
                    )

                normalized = torch.nn.functional.normalize(
                    features,
                    p=2,
                    dim=1
                )

                raw_cpu = (
                    features
                    .detach()
                    .cpu()
                    .numpy()
                    .astype(np.float32)
                )

                normalized_cpu = (
                    normalized
                    .detach()
                    .cpu()
                    .numpy()
                    .astype(np.float32)
                )

            if raw_embeddings is None:

                embedding_dim = raw_cpu.shape[1]

                raw_embeddings = np.empty(
                    (n, embedding_dim),
                    dtype=np.float32
                )

                normalized_embeddings = np.empty(
                    (n, embedding_dim),
                    dtype=np.float32
                )

                print(
                    f"\n{model_name} embedding dimension:",
                    embedding_dim
                )

            raw_embeddings[
                cursor:current_end
            ] = raw_cpu

            normalized_embeddings[
                cursor:current_end
            ] = normalized_cpu

            successful_batches += 1

            progress.update(
                current_end - cursor
            )

            cursor = current_end

            del images
            del inputs
            del pixel_values
            del features
            del normalized
            del raw_cpu
            del normalized_cpu

            gc.collect()

            if torch.cuda.is_available():

                torch.cuda.empty_cache()

        except RuntimeError as e:

            error_text = str(e).lower()

            is_oom = (
                "out of memory" in error_text
                or "cuda error" in error_text
            )

            if is_oom and batch_size > 1:

                new_batch_size = max(
                    1,
                    batch_size // 2
                )

                print(
                    "\nGPU memory issue detected."
                )

                print(
                    f"Reducing batch size: "
                    f"{batch_size} -> {new_batch_size}"
                )

                batch_size = new_batch_size

                gc.collect()

                if torch.cuda.is_available():

                    torch.cuda.empty_cache()

                continue

            raise

        except Exception:

            raise

    progress.close()

    elapsed_seconds = (
        time.time() - start_time
    )

    if raw_embeddings is None:

        raise RuntimeError(
            "No embeddings were extracted."
        )

    if raw_embeddings.shape[0] != n:

        raise RuntimeError(
            "Incorrect number of raw embeddings."
        )

    if normalized_embeddings.shape[0] != n:

        raise RuntimeError(
            "Incorrect number of normalized embeddings."
        )

    if not np.isfinite(raw_embeddings).all():

        raise RuntimeError(
            "Raw embeddings contain non-finite values."
        )

    if not np.isfinite(
        normalized_embeddings
    ).all():

        raise RuntimeError(
            "Normalized embeddings contain non-finite values."
        )

    norms = np.linalg.norm(
        normalized_embeddings,
        axis=1
    )

    if not np.allclose(
        norms,
        1.0,
        atol=1e-5
    ):

        raise RuntimeError(
            "Normalized embeddings do not have unit norm."
        )

    raw_path = (
        output_dir
        / "image_embeddings_raw.npy"
    )

    normalized_path = (
        output_dir
        / "image_embeddings_normalized.npy"
    )

    np.save(
        raw_path,
        raw_embeddings
    )

    np.save(
        normalized_path,
        normalized_embeddings
    )

    embedding_manifest = dataset_df.copy()

    embedding_manifest.insert(
        0,
        "embedding_row",
        np.arange(n)
    )

    embedding_manifest[
        "embedding_model"
    ] = model_name

    embedding_manifest[
        "embedding_dimension"
    ] = raw_embeddings.shape[1]

    embedding_manifest[
        "embedding_dtype"
    ] = "float32"

    manifest_path = (
        output_dir
        / "embedding_manifest.csv"
    )

    embedding_manifest.to_csv(
        manifest_path,
        index=False
    )

    metadata = {
        "dataset": "M4FC",
        "model_name": model_name,
        "model_id": (
            SIGLIP_ID
            if model_name == "SigLIP2-so400m"
            else CLIP_ID
        ),
        "image_count": int(n),
        "embedding_dimension": int(
            raw_embeddings.shape[1]
        ),
        "model_inference_dtype": str(
            MODEL_DTYPE
        ),
        "saved_embedding_dtype": "float32",
        "normalized_embeddings": True,
        "normalization": "L2",
        "initial_batch_size": int(
            initial_batch_size
        ),
        "final_batch_size": int(
            batch_size
        ),
        "successful_batches": int(
            successful_batches
        ),
        "elapsed_seconds": float(
            elapsed_seconds
        ),
        "embedding_files": {
            "raw": str(raw_path),
            "normalized": str(normalized_path),
        },
        "manifest": str(manifest_path),
        "split_counts": {
            str(k): int(v)
            for k, v in dataset_df[
                "split"
            ].value_counts().items()
        },
    }

    metadata_path = (
        output_dir
        / "metadata.json"
    )

    with open(
        metadata_path,
        "w"
    ) as f:

        json.dump(
            metadata,
            f,
            indent=4
        )

    summary = {
        "model": model_name,
        "image_count": int(n),
        "embedding_shape": list(
            raw_embeddings.shape
        ),
        "normalized_shape": list(
            normalized_embeddings.shape
        ),
        "raw_dtype": str(
            raw_embeddings.dtype
        ),
        "normalized_dtype": str(
            normalized_embeddings.dtype
        ),
        "all_finite": bool(
            np.isfinite(raw_embeddings).all()
        ),
        "normalized_unit_norms": bool(
            np.allclose(
                norms,
                1.0,
                atol=1e-5
            )
        ),
        "successful_batches": int(
            successful_batches
        ),
        "final_batch_size": int(
            batch_size
        ),
        "elapsed_seconds": float(
            elapsed_seconds
        ),
        "output_directory": str(
            output_dir
        ),
    }

    print("\n" + "=" * 70)
    print(f"{model_name} EXTRACTION SUMMARY")
    print("=" * 70)

    print("Images:", n)
    print("Embedding shape:", raw_embeddings.shape)
    print("Raw dtype:", raw_embeddings.dtype)
    print(
        "Normalized dtype:",
        normalized_embeddings.dtype
    )
    print(
        "All finite:",
        summary["all_finite"]
    )
    print(
        "Unit norms:",
        summary["normalized_unit_norms"]
    )
    print(
        "Successful batches:",
        successful_batches
    )
    print(
        "Final batch size:",
        batch_size
    )
    print(
        "Elapsed seconds:",
        round(elapsed_seconds, 2)
    )

    print("\nSaved:")
    print(raw_path)
    print(normalized_path)
    print(manifest_path)
    print(metadata_path)

    return summary

print("\n")
print("#" * 70)
print("STARTING SIGLIP 2 EXTRACTION")
print("#" * 70)

siglip_processor = SiglipProcessor.from_pretrained(
    SIGLIP_ID
)

siglip_model = SiglipModel.from_pretrained(
    SIGLIP_ID,
    torch_dtype=MODEL_DTYPE
).to(DEVICE)

siglip_model.eval()

siglip_summary = extract_dataset_embeddings(
    model=siglip_model,
    processor=siglip_processor,
    dataset_df=df,
    output_dir=SIGLIP_OUTPUT_DIR,
    model_name="SigLIP2-so400m",
    feature_function=extract_siglip_features,
    initial_batch_size=SIGLIP_INITIAL_BATCH_SIZE,
)

del siglip_model
del siglip_processor

gc.collect()

if torch.cuda.is_available():

    torch.cuda.empty_cache()

print("\n")
print("#" * 70)
print("STARTING CLIP VIIT-L/14 EXTRACTION")
print("#" * 70)

clip_processor = CLIPProcessor.from_pretrained(
    CLIP_ID
)

clip_model = CLIPModel.from_pretrained(
    CLIP_ID,
    torch_dtype=MODEL_DTYPE
).to(DEVICE)

clip_model.eval()

clip_summary = extract_dataset_embeddings(
    model=clip_model,
    processor=clip_processor,
    dataset_df=df,
    output_dir=CLIP_OUTPUT_DIR,
    model_name="CLIP-ViT-L14",
    feature_function=extract_clip_features,
    initial_batch_size=CLIP_INITIAL_BATCH_SIZE,
)

del clip_model
del clip_processor

gc.collect()

if torch.cuda.is_available():

    torch.cuda.empty_cache()

combined_summary = {
    "dataset": "M4FC",
    "image_count": int(len(df)),
    "dataset_manifest": str(MANIFEST_PATH),
    "siglip2": siglip_summary,
    "clip": clip_summary,
    "overall_status": "EXTRACTION_COMPLETED",
}

combined_summary_path = (
    AUDIT_DIR
    / "m4fc_full_embedding_extraction_summary.json"
)

with open(
    combined_summary_path,
    "w"
) as f:

    json.dump(
        combined_summary,
        f,
        indent=4
    )

print("\n")
print("=" * 70)
print("M4FC FULL IMAGE EMBEDDING EXTRACTION COMPLETED")
print("=" * 70)

print("\nSigLIP 2:")
print(
    "Shape:",
    siglip_summary["embedding_shape"]
)

print("\nCLIP:")
print(
    "Shape:",
    clip_summary["embedding_shape"]
)

print("\nCombined report:")
print(combined_summary_path)

print("\nNext step: independent embedding audit.")

In [ ]:
import json
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/TwinCheck"
)

CLEAN_DIR = (
    PROJECT_ROOT
    / "data/processed/vision/m4fc_images_f2_clean"
)

SOURCE_MANIFEST = (
    CLEAN_DIR
    / "m4fc_f2_clean_manifest.csv"
)

FEATURE_ROOT = (
    PROJECT_ROOT
    / "features/vision/m4fc"
)

AUDIT_DIR = (
    PROJECT_ROOT
    / "audit/y1_m4fc"
)

AUDIT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

MODEL_CONFIGS = {
    "SigLIP2": {
        "directory": (
            FEATURE_ROOT
            / "siglip2_so400m"
        ),
        "expected_dimension": 1152,
    },

    "CLIP": {
        "directory": (
            FEATURE_ROOT
            / "clip_vitl14"
        ),
        "expected_dimension": 768,
    },
}

source_df = pd.read_csv(
    SOURCE_MANIFEST
)

print("Source manifest rows:", len(source_df))

if len(source_df) != 603:

    raise RuntimeError(
        "Source manifest row count is not 603."
    )

def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):

            h.update(chunk)

    return h.hexdigest()

def audit_model_embeddings(
    model_name,
    config
):

    print("\n" + "=" * 70)
    print("AUDITING:", model_name)
    print("=" * 70)

    output_dir = config["directory"]

    expected_dim = config[
        "expected_dimension"
    ]

    raw_path = (
        output_dir
        / "image_embeddings_raw.npy"
    )

    norm_path = (
        output_dir
        / "image_embeddings_normalized.npy"
    )

    manifest_path = (
        output_dir
        / "embedding_manifest.csv"
    )

    metadata_path = (
        output_dir
        / "metadata.json"
    )

    required_files = [
        raw_path,
        norm_path,
        manifest_path,
        metadata_path,
    ]

    missing_files = [
        str(p)
        for p in required_files
        if not p.exists()
    ]

    if missing_files:

        raise RuntimeError(
            f"{model_name}: Missing files:\n"
            f"{missing_files}"
        )

    print("Required files exist: PASS")

    raw = np.load(
        raw_path,
        allow_pickle=False
    )

    normalized = np.load(
        norm_path,
        allow_pickle=False
    )

    embedding_df = pd.read_csv(
        manifest_path
    )

    with open(
        metadata_path,
        "r"
    ) as f:

        metadata = json.load(f)

    shape_pass = (
        raw.shape == (603, expected_dim)
        and normalized.shape == (
            603,
            expected_dim
        )
    )

    if not shape_pass:

        raise RuntimeError(
            f"{model_name}: Shape failure.\n"
            f"Raw: {raw.shape}\n"
            f"Normalized: {normalized.shape}"
        )

    print(
        "Shape:",
        raw.shape,
        "PASS"
    )

    dtype_pass = (
        raw.dtype == np.float32
        and normalized.dtype == np.float32
    )

    if not dtype_pass:

        raise RuntimeError(
            f"{model_name}: Incorrect dtype."
        )

    print("FP32 dtype: PASS")

    raw_finite = np.isfinite(
        raw
    ).all()

    normalized_finite = np.isfinite(
        normalized
    ).all()

    if not raw_finite:

        raise RuntimeError(
            f"{model_name}: Raw embeddings contain NaN/Inf."
        )

    if not normalized_finite:

        raise RuntimeError(
            f"{model_name}: Normalized embeddings contain NaN/Inf."
        )

    print("Finite values: PASS")

    norms = np.linalg.norm(
        normalized,
        axis=1
    )

    unit_norm_pass = np.allclose(
        norms,
        1.0,
        atol=1e-5
    )

    if not unit_norm_pass:

        raise RuntimeError(
            f"{model_name}: Unit norm check failed.\n"
            f"Min norm: {norms.min()}\n"
            f"Max norm: {norms.max()}"
        )

    print("Unit norms: PASS")

    recomputed = (
        raw
        /
        np.maximum(
            np.linalg.norm(
                raw,
                axis=1,
                keepdims=True
            ),
            1e-12
        )
    )

    normalization_consistency = np.allclose(
        recomputed,
        normalized,
        atol=1e-5,
        rtol=1e-5
    )

    if not normalization_consistency:

        raise RuntimeError(
            f"{model_name}: Normalized embeddings "
            "do not match raw embeddings."
        )

    print(
        "Raw-to-normalized consistency: PASS"
    )

    if len(embedding_df) != 603:

        raise RuntimeError(
            f"{model_name}: Manifest has "
            f"{len(embedding_df)} rows."
        )

    print("Embedding manifest row count: PASS")

    required_manifest_columns = [
        "embedding_row",
        "id",
        "split",
        "clean_image_path",
    ]

    missing_columns = [
        col
        for col in required_manifest_columns
        if col not in embedding_df.columns
    ]

    if missing_columns:

        raise RuntimeError(
            f"{model_name}: Missing manifest columns "
            f"{missing_columns}"
        )

    expected_rows = np.arange(
        603
    )

    actual_rows = embedding_df[
        "embedding_row"
    ].to_numpy()

    row_order_pass = np.array_equal(
        actual_rows,
        expected_rows
    )

    if not row_order_pass:

        raise RuntimeError(
            f"{model_name}: embedding_row order incorrect."
        )

    print("Embedding row indexing: PASS")

    source_ids = source_df[
        "id"
    ].astype(str).to_numpy()

    embedding_ids = embedding_df[
        "id"
    ].astype(str).to_numpy()

    id_alignment_pass = np.array_equal(
        source_ids,
        embedding_ids
    )

    if not id_alignment_pass:

        raise RuntimeError(
            f"{model_name}: IDs/order do not match source."
        )

    print("ID alignment: PASS")

    source_splits = source_df[
        "split"
    ].astype(str).to_numpy()

    embedding_splits = embedding_df[
        "split"
    ].astype(str).to_numpy()

    split_alignment_pass = np.array_equal(
        source_splits,
        embedding_splits
    )

    if not split_alignment_pass:

        raise RuntimeError(
            f"{model_name}: Split alignment failed."
        )

    print("Split alignment: PASS")

    source_paths = source_df[
        "clean_image_path"
    ].astype(str).to_numpy()

    embedding_paths = embedding_df[
        "clean_image_path"
    ].astype(str).to_numpy()

    path_alignment_pass = np.array_equal(
        source_paths,
        embedding_paths
    )

    if not path_alignment_pass:

        raise RuntimeError(
            f"{model_name}: Image path alignment failed."
        )

    print("Image path alignment: PASS")

    duplicate_ids = (
        embedding_df["id"]
        .duplicated()
        .sum()
    )

    if duplicate_ids != 0:

        raise RuntimeError(
            f"{model_name}: Duplicate IDs detected."
        )

    print("Duplicate IDs: PASS")

    split_counts = {
        str(k): int(v)
        for k, v in
        embedding_df["split"]
        .value_counts()
        .items()
    }

    expected_counts = {
        "train": 363,
        "dev": 70,
        "test": 170,
    }

    if split_counts != expected_counts:

        raise RuntimeError(
            f"{model_name}: Split counts incorrect:\n"
            f"{split_counts}"
        )

    print("Split counts: PASS")

    metadata_row_count = metadata.get(
        "image_count"
    )

    metadata_dimension = metadata.get(
        "embedding_dimension"
    )

    if metadata_row_count != 603:

        raise RuntimeError(
            f"{model_name}: Metadata image count incorrect."
        )

    if metadata_dimension != expected_dim:

        raise RuntimeError(
            f"{model_name}: Metadata dimension incorrect."
        )

    print("Metadata checks: PASS")

    raw_sha256 = sha256_file(
        raw_path
    )

    norm_sha256 = sha256_file(
        norm_path
    )

    manifest_sha256 = sha256_file(
        manifest_path
    )

    report = {
        "model": model_name,
        "status": "PASS",
        "image_count": 603,
        "embedding_dimension": expected_dim,
        "raw_shape": list(raw.shape),
        "normalized_shape": list(normalized.shape),
        "raw_dtype": str(raw.dtype),
        "normalized_dtype": str(normalized.dtype),
        "all_finite": True,
        "unit_norms": True,
        "raw_normalization_consistency": True,
        "manifest_row_count": 603,
        "row_index_alignment": True,
        "id_alignment": True,
        "split_alignment": True,
        "image_path_alignment": True,
        "duplicate_ids": int(duplicate_ids),
        "split_counts": split_counts,
        "raw_embedding_sha256": raw_sha256,
        "normalized_embedding_sha256": norm_sha256,
        "manifest_sha256": manifest_sha256,
        "raw_file_size_bytes": int(
            raw_path.stat().st_size
        ),
        "normalized_file_size_bytes": int(
            norm_path.stat().st_size
        ),
        "metadata": metadata,
    }

    report_path = (
        AUDIT_DIR
        / f"m4fc_{model_name.lower()}_embedding_audit.json"
    )

    with open(
        report_path,
        "w"
    ) as f:

        json.dump(
            report,
            f,
            indent=4
        )

    print("\nAudit report saved:")
    print(report_path)

    print("\nFINAL RESULT:", "PASS")

    return report

siglip_audit = audit_model_embeddings(
    "SigLIP2",
    MODEL_CONFIGS["SigLIP2"]
)

clip_audit = audit_model_embeddings(
    "CLIP",
    MODEL_CONFIGS["CLIP"]
)

combined_audit = {
    "dataset": "M4FC",
    "image_count": 603,
    "models": {
        "SigLIP2": siglip_audit,
        "CLIP": clip_audit,
    },
    "overall_status": "PASS",
}

combined_audit_path = (
    AUDIT_DIR
    / "m4fc_independent_embedding_audit.json"
)

with open(
    combined_audit_path,
    "w"
) as f:

    json.dump(
        combined_audit,
        f,
        indent=4
    )

print("\n")
print("=" * 70)
print("INDEPENDENT EMBEDDING AUDIT COMPLETE")
print("=" * 70)

print("SigLIP 2: PASS")
print("CLIP: PASS")

print("\nCombined report:")
print(combined_audit_path)

In [ ]:
from pathlib import Path
from zipfile import ZipFile
import pandas as pd
import json

ROOT = Path("/content/drive/MyDrive/TwinCheck")
ZIP_PATH = ROOT / "data/raw/vision/m4fc/Y1_outputs.zip"

CLEAN_MANIFEST = (
    ROOT / "data/processed/vision/m4fc_images_f2_clean"
    / "m4fc_f2_clean_manifest.csv"
)

JSONL_PATH = "ar/uncleaned/m4fc_ar_test_ctx.jsonl"

OUT_DIR = ROOT / "audit/y1_m4fc/label_alignment"
OUT_DIR.mkdir(parents=True, exist_ok=True)

with ZipFile(ZIP_PATH) as zf:
    records = [
        json.loads(line)
        for line in zf.read(JSONL_PATH).decode("utf-8", errors="replace").splitlines()
        if line.strip()
    ]

ann = pd.DataFrame(records)

clean = pd.read_csv(CLEAN_MANIFEST)

true_ann = ann[
    ann["orig_label"].astype(str).str.strip() == "true_caption"
].copy()

true_ann["match_filename"] = true_ann["image_path"].apply(
    lambda x: Path(str(x)).name
)

clean["match_filename"] = clean["clean_image_path"].apply(
    lambda x: Path(str(x)).name
)

true_ann = true_ann.rename(columns={"id": "annotation_id"})
clean = clean.rename(columns={"id": "clean_image_id"})

merged = true_ann.merge(
    clean,
    on="match_filename",
    how="left",
    indicator=True
)

merged["image_exists"] = merged["clean_image_path"].apply(
    lambda p: Path(str(p)).exists()
    if pd.notna(p) and str(p).strip()
    else False
)

eligible = merged[
    (merged["_merge"] == "both") &
    (merged["image_exists"])
].copy()

duplicate_annotation_filenames = true_ann["match_filename"].duplicated().sum()
duplicate_eligible_image_ids = eligible["clean_image_id"].duplicated().sum()

print("=" * 65)
print("M4FC TRUE-CAPTION ALIGNMENT AUDIT")
print("=" * 65)

print("True-caption annotation records:", len(true_ann))
print("Filename matches:", int((merged["_merge"] == "both").sum()))
print("Unmatched annotations:", int((merged["_merge"] == "left_only").sum()))
print("Matched images physically exist:", int(merged["image_exists"].sum()))
print("Eligible true images:", len(eligible))
print("Unique eligible image IDs:", eligible["clean_image_id"].nunique())
print("Duplicate annotation filenames:", duplicate_annotation_filenames)
print("Duplicate eligible image IDs:", duplicate_eligible_image_ids)

audit_csv = OUT_DIR / "m4fc_true_caption_filename_alignment.csv"
eligible_csv = OUT_DIR / "m4fc_eligible_true_images.csv"
summary_json = OUT_DIR / "m4fc_true_image_eligibility_summary.json"

merged.to_csv(audit_csv, index=False)
eligible.to_csv(eligible_csv, index=False)

summary = {
    "true_caption_annotations": int(len(true_ann)),
    "filename_matches": int((merged["_merge"] == "both").sum()),
    "unmatched_annotations": int((merged["_merge"] == "left_only").sum()),
    "matched_files_exist": int(merged["image_exists"].sum()),
    "eligible_true_images": int(len(eligible)),
    "unique_eligible_image_ids": int(eligible["clean_image_id"].nunique()),
    "duplicate_annotation_filenames": int(duplicate_annotation_filenames),
    "duplicate_eligible_image_ids": int(duplicate_eligible_image_ids),
    "alignment_method": "filename matching",
    "audit_csv": str(audit_csv),
    "eligible_csv": str(eligible_csv)
}

with open(summary_json, "w") as f:
    json.dump(summary, f, indent=2)

print("\nSample matched images:")
display(
    eligible[
        ["annotation_id", "clean_image_id", "match_filename",
         "orig_label", "clean_image_path", "image_exists"]
    ].head(10)
)

print("\nSaved audit:", audit_csv)
print("Saved eligible list:", eligible_csv)
print("\nAUDIT COMPLETE — ORIGINAL IMAGES UNMODIFIED")

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import json

ROOT = Path("/content/drive/MyDrive/TwinCheck")

ELIGIBLE_CSV = (
    ROOT / "audit/y1_m4fc/label_alignment"
    / "m4fc_eligible_true_images.csv"
)

MODEL_CONFIGS = {
    "SigLIP2": ROOT / "features/vision/m4fc/siglip2_so400m",
    "CLIP": ROOT / "features/vision/m4fc/clip_vitl14",
}

OUT_DIR = ROOT / "audit/y1_m4fc/embedding_alignment"
OUT_DIR.mkdir(parents=True, exist_ok=True)

eligible = pd.read_csv(ELIGIBLE_CSV)
eligible["clean_image_id"] = eligible["clean_image_id"].astype(str)

eligible_ids = set(eligible["clean_image_id"])

print("=" * 75)
print("M4FC EMBEDDING ALIGNMENT AUDIT")
print("Eligible images:", len(eligible_ids))
print("=" * 75)

all_results = []

source_path_map = dict(
    zip(eligible["clean_image_id"], eligible["clean_image_path"])
)

filename_map = dict(
    zip(eligible["clean_image_id"], eligible["match_filename"])
)

for model_name, model_dir in MODEL_CONFIGS.items():

    print(f"\n{'='*75}")
    print(f"MODEL: {model_name}")
    print("=" * 75)

    manifest = pd.read_csv(
        model_dir / "embedding_manifest.csv"
    )

    manifest["id"] = manifest["id"].astype(str)

    normalized = np.load(
        model_dir / "image_embeddings_normalized.npy"
    )

    raw = np.load(
        model_dir / "image_embeddings_raw.npy"
    )

    model_ids = set(manifest["id"])

    matched_ids = eligible_ids & model_ids
    missing_ids = eligible_ids - model_ids

    selected = manifest[
        manifest["id"].isin(eligible_ids)
    ].copy()

    selected = selected.sort_values("id").reset_index(drop=True)

    rows = selected["embedding_row"].astype(int).to_numpy()

    row_valid = (
        (rows >= 0) &
        (rows < len(normalized)) &
        (rows < len(raw))
    )

    if not row_valid.all():
        print("ERROR: Invalid embedding row indices detected.")
        selected_rows_valid = rows[row_valid]
    else:
        selected_rows_valid = rows

    selected_norm = normalized[selected_rows_valid]
    selected_raw = raw[selected_rows_valid]

    norm_values = np.linalg.norm(selected_norm, axis=1)
    raw_norm_values = np.linalg.norm(selected_raw, axis=1)

    finite_normalized = np.isfinite(selected_norm).all(axis=1)
    finite_raw = np.isfinite(selected_raw).all(axis=1)

    nonzero_normalized = norm_values > 0
    nonzero_raw = raw_norm_values > 0

    unit_norm_check = np.isclose(
        norm_values, 1.0, atol=1e-4
    )

    selected["source_clean_path"] = selected["id"].map(
        source_path_map
    )

    selected["expected_filename"] = selected["id"].map(
        filename_map
    )

    selected["filename_match"] = (
        selected["filename"].astype(str)
        == selected["expected_filename"].astype(str)
    )

    selected["source_image_exists"] = selected[
        "source_clean_path"
    ].apply(
        lambda p: Path(str(p)).exists()
        if pd.notna(p) else False
    )

    selected["embedding_row_valid"] = row_valid

    selected["normalized_finite"] = finite_normalized
    selected["raw_finite"] = finite_raw

    selected["normalized_nonzero"] = nonzero_normalized
    selected["raw_nonzero"] = nonzero_raw

    selected["normalized_unit_norm"] = unit_norm_check

    model_csv = OUT_DIR / (
        f"{model_name.lower()}_155_alignment_audit.csv"
    )

    selected.to_csv(model_csv, index=False)

    result = {
        "model": model_name,
        "eligible_images": len(eligible_ids),
        "manifest_rows": len(manifest),
        "embedding_array_rows": len(normalized),
        "embedding_dimension": int(normalized.shape[1]),
        "matched_ids": len(matched_ids),
        "missing_ids_count": len(missing_ids),
        "missing_ids": sorted(missing_ids),
        "selected_rows": len(selected),
        "unique_selected_ids": int(selected["id"].nunique()),
        "duplicate_ids": int(selected["id"].duplicated().sum()),
        "all_embedding_rows_valid": bool(row_valid.all()),
        "all_normalized_finite": bool(finite_normalized.all()),
        "all_raw_finite": bool(finite_raw.all()),
        "all_normalized_nonzero": bool(nonzero_normalized.all()),
        "all_raw_nonzero": bool(nonzero_raw.all()),
        "all_normalized_unit_norm": bool(unit_norm_check.all()),
        "all_filenames_match": bool(selected["filename_match"].all()),
        "all_source_images_exist": bool(
            selected["source_image_exists"].all()
        ),
        "normalized_norm_min": float(norm_values.min()),
        "normalized_norm_max": float(norm_values.max()),
        "normalized_norm_mean": float(norm_values.mean()),
        "audit_csv": str(model_csv),
    }

    all_results.append(result)

    for key, value in result.items():
        print(f"{key}: {value}")

    print("\nSample:")
    display(
        selected[
            [
                "id",
                "filename",
                "expected_filename",
                "embedding_row",
                "source_clean_path",
                "filename_match",
                "source_image_exists",
            ]
        ].head(5)
    )

    del normalized, raw, selected_norm, selected_raw

overall_pass = all(
    r["matched_ids"] == len(eligible_ids)
    and r["missing_ids_count"] == 0
    and r["duplicate_ids"] == 0
    and r["all_embedding_rows_valid"]
    and r["all_normalized_finite"]
    and r["all_raw_finite"]
    and r["all_normalized_nonzero"]
    and r["all_raw_nonzero"]
    and r["all_normalized_unit_norm"]
    and r["all_filenames_match"]
    and r["all_source_images_exist"]
    for r in all_results
)

report = {
    "eligible_true_images": len(eligible_ids),
    "models": all_results,
    "overall_pass": bool(overall_pass),
}

report_path = OUT_DIR / (
    "m4fc_155_embedding_alignment_summary.json"
)

with open(report_path, "w") as f:
    json.dump(report, f, indent=2)

print("\n" + "=" * 75)
print("OVERALL RESULT:", "PASS" if overall_pass else "CHECK REQUIRED")
print("=" * 75)

print("Report saved:", report_path)
print("Original images and embeddings remain unchanged.")

In [ ]:
from pathlib import Path
import pandas as pd
import cv2
import json
import math
from PIL import Image, ImageDraw
import numpy as np

ROOT = Path("/content/drive/MyDrive/TwinCheck")

ELIGIBLE_CSV = (
    ROOT / "audit/y1_m4fc/label_alignment"
    / "m4fc_eligible_true_images.csv"
)

OUT_DIR = ROOT / "audit/y1_m4fc/face_screening"
OUT_DIR.mkdir(parents=True, exist_ok=True)

CONTACT_DIR = OUT_DIR / "contact_sheets"
CONTACT_DIR.mkdir(parents=True, exist_ok=True)

eligible = pd.read_csv(ELIGIBLE_CSV)

cascade_path = cv2.data.haarcascades + (
    "haarcascade_frontalface_default.xml"
)

face_detector = cv2.CascadeClassifier(cascade_path)

if face_detector.empty():
    raise RuntimeError("Could not load OpenCV face detector.")

results = []

print("Screening", len(eligible), "images...")
print("This is a screening aid, not an automatic exclusion decision.")

for _, row in eligible.iterrows():

    image_path = Path(row["clean_image_path"])
    image_id = str(row["clean_image_id"])

    image = cv2.imread(str(image_path))

    if image is None:
        results.append({
            "id": image_id,
            "image_path": str(image_path),
            "image_readable": False,
            "face_count": -1,
            "largest_face_fraction": None,
            "detector_flag": "IMAGE_READ_FAILURE"
        })
        continue

    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    height, width = gray.shape

    faces = face_detector.detectMultiScale(
        gray,
        scaleFactor=1.1,
        minNeighbors=5,
        minSize=(30, 30)
    )

    face_areas = [
        fw * fh for (fx, fy, fw, fh) in faces
    ]

    largest_face_fraction = (
        max(face_areas) / (width * height)
        if face_areas else 0.0
    )

    detector_flag = (
        "FACE_DETECTED_REVIEW"
        if len(faces) > 0
        else "NO_FACE_DETECTED"
    )

    results.append({
        "id": image_id,
        "image_path": str(image_path),
        "image_readable": True,
        "width": width,
        "height": height,
        "face_count": len(faces),
        "largest_face_fraction": largest_face_fraction,
        "detector_flag": detector_flag
    })

screening = pd.DataFrame(results)

screening_csv = OUT_DIR / "m4fc_155_face_screening.csv"
screening.to_csv(screening_csv, index=False)

valid_rows = screening[
    screening["image_readable"] == True
].reset_index(drop=True)

images_per_sheet = 30
thumb_size = (180, 140)
cols = 5
rows_per_sheet = math.ceil(images_per_sheet / cols)

for sheet_num, start in enumerate(
    range(0, len(valid_rows), images_per_sheet), start=1
):

    batch = valid_rows.iloc[
        start:start + images_per_sheet
    ]

    sheet_width = cols * thumb_size[0]
    sheet_height = rows_per_sheet * 180

    sheet = Image.new(
        "RGB",
        (sheet_width, sheet_height),
        "white"
    )

    draw = ImageDraw.Draw(sheet)

    for local_idx, (_, record) in enumerate(batch.iterrows()):

        image = Image.open(record["image_path"]).convert("RGB")
        image.thumbnail(thumb_size)

        col = local_idx % cols
        row_idx = local_idx // cols

        x = col * thumb_size[0]
        y = row_idx * 180

        sheet.paste(image, (x, y + 25))

        label = (
            f"{record['id']} | "
            f"Faces:{record['face_count']}"
        )

        draw.text(
            (x + 3, y + 3),
            label,
            fill="black"
        )

    sheet_path = CONTACT_DIR / (
        f"m4fc_face_review_sheet_{sheet_num:02d}.jpg"
    )

    sheet.save(sheet_path, quality=90)

    print("Saved:", sheet_path)

detected = screening[
    screening["detector_flag"] == "FACE_DETECTED_REVIEW"
]

failed = screening[
    screening["image_readable"] == False
]

summary = {
    "total_images": len(screening),
    "readable_images": int(screening["image_readable"].sum()),
    "image_read_failures": len(failed),
    "images_with_detected_faces": len(detected),
    "images_without_detected_faces": int(
        (screening["detector_flag"] == "NO_FACE_DETECTED").sum()
    ),
    "manual_review_required": True,
    "contact_sheet_directory": str(CONTACT_DIR),
    "screening_csv": str(screening_csv)
}

summary_path = OUT_DIR / "m4fc_face_screening_summary.json"

with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2)

print("\n" + "=" * 65)
print("FACE SCREENING SUMMARY")
print("=" * 65)

for key, value in summary.items():
    print(f"{key}: {value}")

print("\nDetected-face samples:")
display(detected.head(20))

print("\nScreening complete.")
print("Original images were NOT modified.")

In [ ]:
from pathlib import Path
from IPython.display import display, Image

sheet_dir = Path(
    "/content/drive/MyDrive/TwinCheck/audit/y1_m4fc/face_screening/contact_sheets"
)

sheets = sorted(sheet_dir.glob("m4fc_face_review_sheet_*.jpg"))

print(f"Found {len(sheets)} contact sheets")

for sheet in sheets:
    print(f"\n{sheet.name}")
    display(Image(filename=str(sheet)))

In [ ]:
from pathlib import Path
import pandas as pd
import json

PROJECT_DIR = Path("/content/drive/MyDrive/TwinCheck")

INPUT_CSV = (
    PROJECT_DIR
    / "audit/y1_m4fc/label_alignment/m4fc_eligible_true_images.csv"
)

OUTPUT_DIR = (
    PROJECT_DIR
    / "audit/y1_m4fc/face_screening/final_manual_review"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EXCLUDED_IDS = [
    "m4fc_000206",
    "m4fc_000314",
    "m4fc_000317",
    "m4fc_000324",
    "m4fc_000348",
    "m4fc_000445",
    "m4fc_000470",
    "m4fc_000478",
    "m4fc_000518",
    "m4fc_000663",
    "m4fc_000667",
]

EXCLUSION_REASON = (
    "Manual contact-sheet review: prominent potentially identifiable face(s)"
)

df = pd.read_csv(INPUT_CSV)

ID_COLUMN = "clean_image_id"

if ID_COLUMN not in df.columns:
    raise ValueError(
        f"Column '{ID_COLUMN}' not found. "
        f"Available columns: {df.columns.tolist()}"
    )

df[ID_COLUMN] = df[ID_COLUMN].astype(str).str.strip()

if df[ID_COLUMN].duplicated().any():
    raise ValueError("Duplicate clean_image_id values found.")

missing_ids = sorted(set(EXCLUDED_IDS) - set(df[ID_COLUMN]))

if missing_ids:
    raise ValueError(
        f"Exclusion IDs not found in dataset: {missing_ids}"
    )

excluded_df = df[df[ID_COLUMN].isin(EXCLUDED_IDS)].copy()
eligible_df = df[~df[ID_COLUMN].isin(EXCLUDED_IDS)].copy()

excluded_df["manual_exclusion_reason"] = EXCLUSION_REASON

eligible_path = (
    OUTPUT_DIR / "m4fc_final_face_eligible_candidates.csv"
)

excluded_path = (
    OUTPUT_DIR / "m4fc_manual_face_exclusions.csv"
)

summary_path = (
    OUTPUT_DIR / "m4fc_final_face_exclusion_summary.json"
)

eligible_df.to_csv(eligible_path, index=False)
excluded_df.to_csv(excluded_path, index=False)

summary = {
    "id_column_used": ID_COLUMN,
    "original_eligible_images": int(len(df)),
    "manually_excluded_images": int(len(excluded_df)),
    "remaining_candidate_images": int(len(eligible_df)),
    "excluded_image_ids": EXCLUDED_IDS,
    "exclusion_reason": EXCLUSION_REASON,
    "duplicate_original_ids": int(df[ID_COLUMN].duplicated().sum()),
    "duplicate_remaining_ids": int(
        eligible_df[ID_COLUMN].duplicated().sum()
    ),
    "validation": "PASS",
    "original_images_modified": False,
}

with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2)

assert len(excluded_df) == 11
assert len(df) == len(excluded_df) + len(eligible_df)
assert eligible_df[ID_COLUMN].is_unique
assert excluded_df[ID_COLUMN].is_unique
assert set(eligible_df[ID_COLUMN]).isdisjoint(
    set(excluded_df[ID_COLUMN])
)

print("=" * 60)
print("M4FC FACE EXCLUSION AUDIT")
print("=" * 60)
print("Original eligible:", len(df))
print("Excluded:", len(excluded_df))
print("Remaining:", len(eligible_df))
print("Validation: PASS")

print("\nExcluded IDs:")
print(excluded_df[ID_COLUMN].tolist())

print("\nSaved:")
print(eligible_path)
print(excluded_path)
print(summary_path)

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import json

PROJECT_DIR = Path("/content/drive/MyDrive/TwinCheck")

CANDIDATE_CSV = (
    PROJECT_DIR
    / "audit/y1_m4fc/face_screening/final_manual_review/"
    / "m4fc_final_face_eligible_candidates.csv"
)

MODEL_DIRS = {
    "SigLIP2": PROJECT_DIR / "features/vision/m4fc/siglip2_so400m",
    "CLIP": PROJECT_DIR / "features/vision/m4fc/clip_vitl14",
}

OUTPUT_DIR = (
    PROJECT_DIR
    / "audit/y1_m4fc/embedding_alignment/final_144"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

candidates = pd.read_csv(CANDIDATE_CSV)
candidates["clean_image_id"] = candidates["clean_image_id"].astype(str)

if candidates["clean_image_id"].duplicated().any():
    raise ValueError("Duplicate candidate image IDs found.")

model_results = {}
summary = {}

for model_name, model_dir in MODEL_DIRS.items():

    print(f"\nAuditing {model_name}...")

    manifest = pd.read_csv(model_dir / "embedding_manifest.csv")

    raw_embeddings = np.load(
        model_dir / "image_embeddings_raw.npy"
    )

    norm_embeddings = np.load(
        model_dir / "image_embeddings_normalized.npy"
    )

    if manifest["id"].duplicated().any():
        raise ValueError(f"{model_name}: duplicate IDs in manifest")

    if manifest["embedding_row"].duplicated().any():
        raise ValueError(f"{model_name}: duplicate embedding rows")

    mapped = candidates[["clean_image_id", "clean_image_path"]].merge(
        manifest[["id", "embedding_row", "filename"]],
        left_on="clean_image_id",
        right_on="id",
        how="left",
        indicator=True
    )

    mapped["manifest_match"] = mapped["_merge"].eq("both")
    mapped.drop(columns=["_merge"], inplace=True)

    mapped["embedding_row"] = pd.to_numeric(
        mapped["embedding_row"], errors="coerce"
    )

    mapped["row_valid"] = (
        mapped["embedding_row"].notna()
        & (mapped["embedding_row"] >= 0)
        & (mapped["embedding_row"] < len(raw_embeddings))
        & (mapped["embedding_row"] < len(norm_embeddings))
    )

    mapped["original_file_exists"] = mapped[
        "clean_image_path"
    ].apply(lambda p: Path(str(p)).exists())

    mapped["raw_finite"] = False
    mapped["normalized_finite"] = False
    mapped["normalized_unit_norm"] = False

    valid_rows = mapped[mapped["row_valid"]].index

    for idx in valid_rows:

        row = int(mapped.loc[idx, "embedding_row"])

        raw_vec = raw_embeddings[row]
        norm_vec = norm_embeddings[row]

        mapped.loc[idx, "raw_finite"] = np.isfinite(raw_vec).all()
        mapped.loc[idx, "normalized_finite"] = np.isfinite(norm_vec).all()

        norm_value = np.linalg.norm(norm_vec)

        mapped.loc[idx, "normalized_unit_norm"] = (
            np.isfinite(norm_value)
            and np.isclose(norm_value, 1.0, atol=1e-4)
        )

    mapped["audit_pass"] = (
        mapped["manifest_match"]
        & mapped["row_valid"]
        & mapped["original_file_exists"]
        & mapped["raw_finite"]
        & mapped["normalized_finite"]
        & mapped["normalized_unit_norm"]
    )

    audit_path = OUTPUT_DIR / f"{model_name.lower()}_144_alignment_audit.csv"
    mapped.to_csv(audit_path, index=False)

    model_summary = {
        "model": model_name,
        "candidate_count": int(len(mapped)),
        "manifest_matches": int(mapped["manifest_match"].sum()),
        "valid_embedding_rows": int(mapped["row_valid"].sum()),
        "original_files_exist": int(mapped["original_file_exists"].sum()),
        "raw_embeddings_finite": int(mapped["raw_finite"].sum()),
        "normalized_embeddings_finite": int(
            mapped["normalized_finite"].sum()
        ),
        "normalized_unit_norm": int(
            mapped["normalized_unit_norm"].sum()
        ),
        "audit_pass_count": int(mapped["audit_pass"].sum()),
        "audit_fail_count": int((~mapped["audit_pass"]).sum()),
        "raw_array_shape": list(raw_embeddings.shape),
        "normalized_array_shape": list(norm_embeddings.shape),
        "audit_csv": str(audit_path),
    }

    model_results[model_name] = mapped
    summary[model_name] = model_summary

    print(f"Candidate count: {len(mapped)}")
    print(f"Manifest matches: {mapped['manifest_match'].sum()}")
    print(f"Valid rows: {mapped['row_valid'].sum()}")
    print(f"Original files exist: {mapped['original_file_exists'].sum()}")
    print(f"Raw finite: {mapped['raw_finite'].sum()}")
    print(f"Normalized finite: {mapped['normalized_finite'].sum()}")
    print(f"Unit norm: {mapped['normalized_unit_norm'].sum()}")
    print(f"PASS: {mapped['audit_pass'].sum()}")
    print(f"FAIL: {(~mapped['audit_pass']).sum()}")

combined_pass = (
    model_results["SigLIP2"]["audit_pass"]
    & model_results["CLIP"]["audit_pass"]
)

summary["overall"] = {
    "candidate_count": int(len(candidates)),
    "both_models_pass": int(combined_pass.sum()),
    "both_models_fail_or_partial": int((~combined_pass).sum()),
    "overall_status": (
        "PASS" if combined_pass.all() else "REVIEW_REQUIRED"
    ),
}

summary_path = OUTPUT_DIR / "m4fc_144_embedding_alignment_summary.json"

with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2)

print("\n" + "=" * 65)
print("FINAL COMBINED ALIGNMENT AUDIT")
print("=" * 65)
print("Candidates:", len(candidates))
print("Both models pass:", int(combined_pass.sum()))
print("Any failure:", int((~combined_pass).sum()))
print("Overall status:", summary["overall"]["overall_status"])
print("\nSummary saved:")
print(summary_path)

In [ ]:
from pathlib import Path
from collections import defaultdict
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
from tqdm.auto import tqdm
import json
import time
import gc

from diffusers import AutoencoderKL

PROJECT_DIR = Path("/content/drive/MyDrive/TwinCheck")

CANDIDATE_CSV = (
    PROJECT_DIR
    / "audit/y1_m4fc/face_screening/final_manual_review/"
    / "m4fc_final_face_eligible_candidates.csv"
)

OUTPUT_DIR = (
    PROJECT_DIR
    / "data/processed/vision/m4fc/vae_reconstructed"
)

AUDIT_DIR = (
    PROJECT_DIR
    / "audit/y1_m4fc/vae_reconstruction"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "stabilityai/sd-vae-ft-mse"

MAX_SIDE = 512
JPEG_QUALITY = 85

INITIAL_BATCH_SIZE = 4

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32

candidates = pd.read_csv(CANDIDATE_CSV)

assert len(candidates) == 144
assert candidates["clean_image_id"].is_unique

print(f"Images to reconstruct: {len(candidates)}")
print(f"Device: {DEVICE}")
print(f"Precision: {DTYPE}")

print("\nLoading VAE...")

vae = AutoencoderKL.from_pretrained(
    MODEL_NAME
).to(device=DEVICE, dtype=DTYPE)

vae.eval()

def prepare_image(image_path):

    image = Image.open(image_path).convert("RGB")

    original_width, original_height = image.size

    scale = min(1.0, MAX_SIDE / max(original_width, original_height))

    new_width = max(
        8,
        int(round(original_width * scale / 8) * 8)
    )

    new_height = max(
        8,
        int(round(original_height * scale / 8) * 8)
    )

    image = image.resize(
        (new_width, new_height),
        Image.Resampling.LANCZOS
    )

    array = np.asarray(image).astype(np.float32) / 255.0

    tensor = torch.from_numpy(array).permute(2, 0, 1)

    tensor = tensor * 2.0 - 1.0

    return tensor, image, (original_width, original_height)

def calculate_psnr(original, reconstructed):

    original = np.asarray(original).astype(np.float32) / 255.0
    reconstructed = np.asarray(reconstructed).astype(np.float32) / 255.0

    mse = np.mean((original - reconstructed) ** 2)

    if mse == 0:
        return float("inf")

    return float(10 * np.log10(1.0 / mse))

@torch.inference_mode()
def reconstruct_batch(tensors):

    batch = torch.stack(tensors).to(
        device=DEVICE,
        dtype=DTYPE
    )

    latent = vae.encode(batch).latent_dist.mean

    reconstructed = vae.decode(latent).sample

    reconstructed = (
        reconstructed.float()
        .clamp(-1, 1)
        .add(1)
        .div(2)
        .mul(255)
        .round()
        .byte()
        .cpu()
    )

    return reconstructed

print("\nPreparing images...")

prepared = []
shape_groups = defaultdict(list)
input_failures = []

for _, row in tqdm(candidates.iterrows(), total=len(candidates)):

    image_id = str(row["clean_image_id"])
    image_path = Path(row["clean_image_path"])

    output_path = OUTPUT_DIR / f"{image_id}.jpg"

    if output_path.exists() and output_path.stat().st_size > 0:
        continue

    try:
        tensor, resized_image, original_size = prepare_image(image_path)

        item = {
            "id": image_id,
            "input_path": str(image_path),
            "output_path": str(output_path),
            "tensor": tensor,
            "resized_image": resized_image,
            "original_size": original_size,
            "processed_size": resized_image.size,
        }

        shape_groups[tuple(tensor.shape)].append(item)

    except Exception as e:

        input_failures.append({
            "id": image_id,
            "input_path": str(image_path),
            "error": str(e),
        })

print(f"Images needing generation: {sum(map(len, shape_groups.values()))}")
print(f"Shape groups: {len(shape_groups)}")
print(f"Input failures: {len(input_failures)}")

records = []
oom_retries = 0
generation_errors = []

start_time = time.time()

for shape, items in tqdm(
    list(shape_groups.items()),
    desc="Shape groups"
):

    for start in range(0, len(items), INITIAL_BATCH_SIZE):

        batch_items = items[start:start + INITIAL_BATCH_SIZE]

        current_batch_size = len(batch_items)

        success = False

        while not success:

            try:

                batch_tensors = [
                    item["tensor"]
                    for item in batch_items[:current_batch_size]
                ]

                reconstructed_batch = reconstruct_batch(batch_tensors)

                for item, reconstructed_tensor in zip(
                    batch_items[:current_batch_size],
                    reconstructed_batch
                ):

                    reconstructed_image = Image.fromarray(
                        reconstructed_tensor.permute(1, 2, 0).numpy(),
                        mode="RGB"
                    )

                    reconstructed_image = reconstructed_image.resize(
                        item["processed_size"],
                        Image.Resampling.LANCZOS
                    )

                    reconstructed_image.save(
                        item["output_path"],
                        format="JPEG",
                        quality=JPEG_QUALITY,
                        optimize=True
                    )

                    psnr = calculate_psnr(
                        item["resized_image"],
                        reconstructed_image
                    )

                    records.append({
                        "id": item["id"],
                        "input_path": item["input_path"],
                        "output_path": item["output_path"],
                        "original_width": item["original_size"][0],
                        "original_height": item["original_size"][1],
                        "processed_width": item["processed_size"][0],
                        "processed_height": item["processed_size"][1],
                        "jpeg_quality": JPEG_QUALITY,
                        "psnr": psnr,
                        "output_exists": Path(item["output_path"]).exists(),
                        "output_size_bytes": Path(
                            item["output_path"]
                        ).stat().st_size,
                        "status": "success",
                    })

                success = True

            except torch.cuda.OutOfMemoryError:

                oom_retries += 1

                gc.collect()

                if DEVICE == "cuda":
                    torch.cuda.empty_cache()

                if current_batch_size > 1:

                    current_batch_size = max(
                        1,
                        current_batch_size // 2
                    )

                    print(
                        f"\nOOM: retrying with batch size "
                        f"{current_batch_size}"
                    )

                else:

                    for item in batch_items:

                        generation_errors.append({
                            "id": item["id"],
                            "input_path": item["input_path"],
                            "error": "CUDA OOM even with batch size 1",
                        })

                    success = True

            except Exception as e:

                generation_errors.append({
                    "id": batch_items[0]["id"],
                    "input_path": batch_items[0]["input_path"],
                    "error": str(e),
                })

                success = True

            if current_batch_size < len(batch_items):
                remaining_items = batch_items[current_batch_size:]

                for item in remaining_items:

                    try:

                        reconstructed_batch = reconstruct_batch(
                            [item["tensor"]]
                        )

                        reconstructed_image = Image.fromarray(
                            reconstructed_batch[0].permute(
                                1, 2, 0
                            ).numpy(),
                            mode="RGB"
                        )

                        reconstructed_image = reconstructed_image.resize(
                            item["processed_size"],
                            Image.Resampling.LANCZOS
                        )

                        reconstructed_image.save(
                            item["output_path"],
                            format="JPEG",
                            quality=JPEG_QUALITY,
                            optimize=True
                        )

                        records.append({
                            "id": item["id"],
                            "input_path": item["input_path"],
                            "output_path": item["output_path"],
                            "original_width": item["original_size"][0],
                            "original_height": item["original_size"][1],
                            "processed_width": item["processed_size"][0],
                            "processed_height": item["processed_size"][1],
                            "jpeg_quality": JPEG_QUALITY,
                            "psnr": calculate_psnr(
                                item["resized_image"],
                                reconstructed_image
                            ),
                            "output_exists": True,
                            "output_size_bytes": Path(
                                item["output_path"]
                            ).stat().st_size,
                            "status": "success",
                        })

                    except Exception as e:

                        generation_errors.append({
                            "id": item["id"],
                            "input_path": item["input_path"],
                            "error": str(e),
                        })

manifest_path = AUDIT_DIR / "m4fc_vae_144_reconstruction_manifest.csv"

manifest_df = pd.DataFrame(records)

if len(manifest_df) > 0:
    manifest_df.to_csv(manifest_path, index=False)

failures_df = pd.DataFrame(
    input_failures + generation_errors
)

failure_path = AUDIT_DIR / "m4fc_vae_144_generation_failures.csv"

failures_df.to_csv(failure_path, index=False)

elapsed = time.time() - start_time

summary = {
    "dataset": "M4FC",
    "stage": "VAE reconstruction twins",
    "candidate_count": int(len(candidates)),
    "generated_successfully": int(
        (manifest_df["status"] == "success").sum()
    ) if len(manifest_df) else 0,
    "generation_failures": int(len(failures_df)),
    "oom_retries": int(oom_retries),
    "model": MODEL_NAME,
    "device": DEVICE,
    "precision": str(DTYPE),
    "max_side": MAX_SIDE,
    "jpeg_quality": JPEG_QUALITY,
    "elapsed_seconds": float(elapsed),
    "elapsed_minutes": float(elapsed / 60),
    "mean_psnr": float(manifest_df["psnr"].mean())
        if len(manifest_df) else None,
    "median_psnr": float(manifest_df["psnr"].median())
        if len(manifest_df) else None,
    "min_psnr": float(manifest_df["psnr"].min())
        if len(manifest_df) else None,
    "max_psnr": float(manifest_df["psnr"].max())
        if len(manifest_df) else None,
    "manifest": str(manifest_path),
    "failure_csv": str(failure_path),
}

summary_path = AUDIT_DIR / "m4fc_vae_144_reconstruction_summary.json"

with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2)

print("\n" + "=" * 70)
print("M4FC VAE RECONSTRUCTION SUMMARY")
print("=" * 70)

for key, value in summary.items():
    print(f"{key}: {value}")

print("\nManifest:")
print(manifest_path)

print("\nFailure report:")
print(failure_path)

print("\nSummary JSON:")
print(summary_path)

del vae
gc.collect()

if DEVICE == "cuda":
    torch.cuda.empty_cache()

In [ ]:
import os, gc, json, time
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
from transformers import AutoProcessor, AutoModel

PROJECT_DIR = Path("/content/drive/MyDrive/TwinCheck")

ORIGINAL_DIR = PROJECT_DIR / "data/processed/vision/m4fc_images_f2_clean"
TWIN_DIR = PROJECT_DIR / "data/processed/vision/m4fc/vae_reconstructed"

CANDIDATE_CSV = (
    PROJECT_DIR / "audit/y1_m4fc/face_screening/"
    "final_manual_review/m4fc_final_face_eligible_candidates.csv"
)

OUTPUT_DIR = PROJECT_DIR / "features/vision/m4fc/vae_similarity_144"
AUDIT_DIR = PROJECT_DIR / "audit/y1_m4fc/vae_similarity_144"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 2

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("GPU:", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "CPU")
print("Free GPU memory:", round(torch.cuda.mem_get_info()[0]/1024**3, 2), "GB")
print("Batch size:", BATCH_SIZE)

candidates = pd.read_csv(CANDIDATE_CSV)
ids = candidates["clean_image_id"].astype(str).tolist()

original_paths = [ORIGINAL_DIR / f"{x}.jpg" for x in ids]
twin_paths = [TWIN_DIR / f"{x}.jpg" for x in ids]

print("Candidates:", len(ids))

MODELS = {
    "siglip2": "google/siglip2-so400m-patch14-384",
    "clip": "openai/clip-vit-large-patch14",
}

results = []
shape_summary = {}
start_time = time.time()

for model_name, model_id in MODELS.items():

    print("\n" + "=" * 70)
    print("LOADING:", model_name)
    print("=" * 70)

    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

    processor = AutoProcessor.from_pretrained(model_id)

    model = AutoModel.from_pretrained(
        model_id,
        dtype=torch.float16 if DEVICE == "cuda" else torch.float32
    ).to(DEVICE)

    model.eval()

    def extract_features(images):

        inputs = processor(images=images, return_tensors="pt")
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

        with torch.inference_mode():

            if DEVICE == "cuda":
                with torch.autocast("cuda", dtype=torch.float16):
                    vision_output = model.vision_model(
                        pixel_values=inputs["pixel_values"]
                    )

                    pooled = vision_output.pooler_output

                    if model_name == "clip":
                        features = model.visual_projection(pooled)
                    else:
                        features = pooled

            else:
                vision_output = model.vision_model(
                    pixel_values=inputs["pixel_values"]
                )

                pooled = vision_output.pooler_output

                if model_name == "clip":
                    features = model.visual_projection(pooled)
                else:
                    features = pooled

        output_np = features.float().cpu().numpy()

        del inputs, vision_output, pooled, features

        return output_np

    def encode_paths(paths, label):

        all_features = []
        pos = 0
        current_batch = BATCH_SIZE

        pbar = tqdm(total=len(paths), desc=label)

        while pos < len(paths):

            batch_paths = paths[pos:pos + current_batch]

            images = []
            for path in batch_paths:
                with Image.open(path) as im:
                    images.append(im.convert("RGB"))

            try:

                batch_features = extract_features(images)

                all_features.append(batch_features)

                pos += len(batch_paths)
                pbar.update(len(batch_paths))

                del images, batch_features

                if DEVICE == "cuda":
                    torch.cuda.empty_cache()

            except torch.cuda.OutOfMemoryError:

                del images
                gc.collect()
                torch.cuda.empty_cache()

                if current_batch > 1:
                    current_batch = max(1, current_batch // 2)
                    print(f"\nOOM: batch reduced to {current_batch}")
                else:
                    raise RuntimeError(
                        f"OOM with batch size 1 at index {pos}"
                    )

        pbar.close()

        return np.concatenate(all_features, axis=0)

    print("\nEncoding originals...")
    original_embeddings = encode_paths(
        original_paths,
        f"{model_name} originals"
    )

    np.save(
        OUTPUT_DIR / f"{model_name}_original_raw.npy",
        original_embeddings
    )

    print("\nEncoding VAE twins...")
    twin_embeddings = encode_paths(
        twin_paths,
        f"{model_name} twins"
    )

    np.save(
        OUTPUT_DIR / f"{model_name}_twin_raw.npy",
        twin_embeddings
    )

    original_norm = original_embeddings / np.maximum(
        np.linalg.norm(original_embeddings, axis=1, keepdims=True),
        1e-12
    )

    twin_norm = twin_embeddings / np.maximum(
        np.linalg.norm(twin_embeddings, axis=1, keepdims=True),
        1e-12
    )

    np.save(
        OUTPUT_DIR / f"{model_name}_original_normalized.npy",
        original_norm
    )

    np.save(
        OUTPUT_DIR / f"{model_name}_twin_normalized.npy",
        twin_norm
    )

    similarities = np.sum(original_norm * twin_norm, axis=1)

    shape_summary[model_name] = {
        "original_shape": list(original_embeddings.shape),
        "twin_shape": list(twin_embeddings.shape)
    }

    for i, image_id in enumerate(ids):

        results.append({
            "id": image_id,
            "model": model_name,
            "original_path": str(original_paths[i]),
            "twin_path": str(twin_paths[i]),
            "cosine_similarity": float(similarities[i]),
            "threshold_0_92_pass": bool(similarities[i] >= 0.92)
        })

    print("\n", model_name, "RESULTS")
    print("Embedding shape:", original_embeddings.shape)
    print("Mean similarity:", float(similarities.mean()))
    print("Median similarity:", float(np.median(similarities)))
    print("Min:", float(similarities.min()))
    print("Max:", float(similarities.max()))
    print("Pass >= 0.92:", int((similarities >= 0.92).sum()))
    print("Fail < 0.92:", int((similarities < 0.92).sum()))

    checkpoint = pd.DataFrame(results)
    checkpoint.to_csv(
        OUTPUT_DIR / "similarity_progress_checkpoint.csv",
        index=False
    )

    del original_embeddings, twin_embeddings
    del original_norm, twin_norm, similarities
    del model, processor

    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

results_df = pd.DataFrame(results)

results_csv = OUTPUT_DIR / "m4fc_144_original_vae_cosine_similarity.csv"
results_df.to_csv(results_csv, index=False)

pivot = results_df.pivot(
    index="id",
    columns="model",
    values="cosine_similarity"
).reset_index()

pivot["siglip2_pass_0_92"] = pivot["siglip2"] >= 0.92
pivot["clip_pass_0_92"] = pivot["clip"] >= 0.92

pivot["both_pass_0_92"] = (
    pivot["siglip2_pass_0_92"] &
    pivot["clip_pass_0_92"]
)

pivot_csv = OUTPUT_DIR / "m4fc_144_similarity_pivot.csv"
pivot.to_csv(pivot_csv, index=False)

summary = {
    "dataset": "M4FC",
    "stage": "Original vs VAE twin similarity",
    "candidate_count": len(ids),
    "threshold": 0.92,
    "embedding_shapes": shape_summary,
    "both_models_pass": int(pivot["both_pass_0_92"].sum()),
    "siglip2_only_pass": int(
        (pivot["siglip2_pass_0_92"] & ~pivot["clip_pass_0_92"]).sum()
    ),
    "clip_only_pass": int(
        (pivot["clip_pass_0_92"] & ~pivot["siglip2_pass_0_92"]).sum()
    ),
    "both_fail": int(
        (~pivot["siglip2_pass_0_92"] & ~pivot["clip_pass_0_92"]).sum()
    ),
    "models": {},
    "elapsed_seconds": time.time() - start_time,
    "results_csv": str(results_csv),
    "pivot_csv": str(pivot_csv)
}

for model_name in MODELS:

    model_df = results_df[results_df["model"] == model_name]
    sim = model_df["cosine_similarity"].values

    summary["models"][model_name] = {
        "count": int(len(sim)),
        "mean": float(np.mean(sim)),
        "median": float(np.median(sim)),
        "min": float(np.min(sim)),
        "max": float(np.max(sim)),
        "pass": int((sim >= 0.92).sum()),
        "fail": int((sim < 0.92).sum())
    }

summary["elapsed_minutes"] = summary["elapsed_seconds"] / 60

summary_json = AUDIT_DIR / "m4fc_144_similarity_summary.json"

with open(summary_json, "w") as f:
    json.dump(summary, f, indent=2)

print("\n" + "=" * 70)
print("FINAL M4FC SIMILARITY SUMMARY")
print("=" * 70)
print(json.dumps(summary, indent=2))

print("\nLowest similarity pairs:")
print(pivot.sort_values(["siglip2", "clip"]).head(10).to_string(index=False))

print("\nHighest similarity pairs:")
print(pivot.sort_values(["siglip2", "clip"], ascending=False).head(10).to_string(index=False))

print("\nResults CSV:", results_csv)
print("Pivot CSV:", pivot_csv)
print("Summary JSON:", summary_json)

In [ ]:
from pathlib import Path
import pandas as pd
import json

PROJECT_DIR = Path("/content/drive/MyDrive/TwinCheck")

SIM_DIR = PROJECT_DIR / "features/vision/m4fc/vae_similarity_144"
AUDIT_DIR = PROJECT_DIR / "audit/y1_m4fc/vae_similarity_144"

RESULTS_CSV = SIM_DIR / "m4fc_144_original_vae_cosine_similarity.csv"

FINAL_DIR = AUDIT_DIR / "final_joint_eligible_092"
FINAL_DIR.mkdir(parents=True, exist_ok=True)

THRESHOLD = 0.92

df = pd.read_csv(RESULTS_CSV)

print("Loaded:", len(df), "rows")
print("Models:", df["model"].unique().tolist())

wide = df.pivot(
    index=["id", "original_path", "twin_path"],
    columns="model",
    values="cosine_similarity"
).reset_index()

wide.columns.name = None

wide["siglip2_pass_0_92"] = wide["siglip2"] >= THRESHOLD
wide["clip_pass_0_92"] = wide["clip"] >= THRESHOLD

wide["both_pass_0_92"] = (
    wide["siglip2_pass_0_92"] &
    wide["clip_pass_0_92"]
)

eligible = (
    wide[wide["both_pass_0_92"]]
    .sort_values("id")
    .reset_index(drop=True)
)

excluded = (
    wide[~wide["both_pass_0_92"]]
    .sort_values("id")
    .reset_index(drop=True)
)

manifest_path = FINAL_DIR / "m4fc_vae_final_joint_eligible_092.csv"
excluded_path = FINAL_DIR / "m4fc_vae_excluded_joint_eligible_092.csv"
summary_path = FINAL_DIR / "m4fc_vae_final_joint_eligible_092_summary.json"

eligible.to_csv(manifest_path, index=False)
excluded.to_csv(excluded_path, index=False)

summary = {
    "dataset": "M4FC",
    "stage": "Final VAE twin eligibility",
    "candidate_count": int(len(wide)),
    "threshold": THRESHOLD,
    "criterion": "Both CLIP and SigLIP2 cosine similarity >= 0.92",
    "eligible_count": int(len(eligible)),
    "excluded_count": int(len(excluded)),
    "eligible_percentage": round(
        len(eligible) / len(wide) * 100, 4
    ),
    "clip_pass_count": int(wide["clip_pass_0_92"].sum()),
    "siglip2_pass_count": int(wide["siglip2_pass_0_92"].sum()),
    "joint_pass_count": int(wide["both_pass_0_92"].sum()),
    "manifest_path": str(manifest_path),
    "excluded_path": str(excluded_path),
}

with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2)

print("\n" + "=" * 60)
print("FINAL M4FC VAE ELIGIBILITY")
print("=" * 60)

print(f"Unique candidates : {len(wide)}")
print(f"CLIP passes      : {wide['clip_pass_0_92'].sum()}")
print(f"SigLIP2 passes   : {wide['siglip2_pass_0_92'].sum()}")
print(f"Both pass        : {len(eligible)}")
print(f"Excluded         : {len(excluded)}")
print(f"Eligible         : {len(eligible)/len(wide)*100:.2f}%")

print("\nFinal eligible pairs:")
display(
    eligible[
        [
            "id",
            "original_path",
            "twin_path",
            "clip",
            "siglip2",
            "clip_pass_0_92",
            "siglip2_pass_0_92",
            "both_pass_0_92",
        ]
    ]
)

print("\nSaved files:")
print(manifest_path)
print(excluded_path)
print(summary_path)

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import hashlib
import json

ROOT = Path("/content/drive/MyDrive/TwinCheck")

TEST_MANIFEST = (
    ROOT / "data/processed/vision/genimage"
    / "genimage_test_manifest.csv"
)

SDXL_ROOT = (
    ROOT / "data/copies/vision/genimage_sdxl_img2img_500"
)

SOURCE_MANIFEST_PATH = SDXL_ROOT / "sdxl_500_source_manifest.csv"

SDXL_ROOT.mkdir(parents=True, exist_ok=True)

TARGET_PER_CLASS = 250
SEED = 42

REQUIRED_COLUMNS = [
    "split",
    "class_name",
    "normalized_export_path",
    "normalized_md5",
    "generator",
]

if not TEST_MANIFEST.exists():
    raise FileNotFoundError(
        f"Test manifest not found:\n{TEST_MANIFEST}"
    )

df = pd.read_csv(TEST_MANIFEST)

missing = set(REQUIRED_COLUMNS) - set(df.columns)

if missing:
    raise ValueError(
        f"Test manifest missing columns: {sorted(missing)}"
    )

test_df = df[df["split"].astype(str) == "test"].copy()

print("Test manifest rows:", len(test_df))
print("\nTest class counts:")
print(test_df["class_name"].value_counts())

assert len(test_df) == 2000, (
    f"Expected 2,000 test rows, found {len(test_df)}"
)

assert test_df["class_name"].value_counts().to_dict() == {
    "real": 1000,
    "fake": 1000
}, "Unexpected test class distribution."

test_df["normalized_export_path"] = (
    test_df["normalized_export_path"].astype(str)
)

missing_files = test_df[
    ~test_df["normalized_export_path"].map(
        lambda p: Path(p).is_file()
    )
]

if len(missing_files):
    raise FileNotFoundError(
        f"{len(missing_files)} test source images are missing. "
        "Stopping before selection."
    )

print("All test source images exist.")

selected_parts = []

for cls in ["real", "fake"]:

    subset = test_df[
        test_df["class_name"].astype(str) == cls
    ].copy()

    selected = subset.sample(
        n=TARGET_PER_CLASS,
        random_state=SEED
    )

    selected_parts.append(selected)

selected_df = pd.concat(
    selected_parts,
    ignore_index=True
)

selected_df = selected_df.sort_values(
    ["class_name", "normalized_md5"]
).reset_index(drop=True)

assert len(selected_df) == 500
assert selected_df["normalized_md5"].nunique() == 500

selected_df["source_id"] = selected_df.apply(
    lambda row: hashlib.sha256(
        str(row["normalized_md5"]).encode()
    ).hexdigest()[:16],
    axis=1
)

selected_df["sdxl_status"] = "pending"
selected_df["sdxl_output_path"] = ""
selected_df["sdxl_error"] = ""

if SOURCE_MANIFEST_PATH.exists():

    existing = pd.read_csv(SOURCE_MANIFEST_PATH)

    if len(existing) != 500:
        raise RuntimeError(
            "Existing SDXL source manifest has unexpected row count. "
            "Do not overwrite it automatically."
        )

    if set(existing["normalized_md5"]) != set(
        selected_df["normalized_md5"]
    ):
        raise RuntimeError(
            "Existing source manifest differs from this selection. "
            "Preserving existing manifest; investigate before proceeding."
        )

    print("\nExisting source manifest validated.")
    selected_df = existing

else:

    selected_df.to_csv(
        SOURCE_MANIFEST_PATH,
        index=False
    )

    print("\nNew source manifest saved.")

print("\n" + "=" * 70)
print("SDXL SOURCE SELECTION COMPLETE")
print("=" * 70)

print("Selected:", len(selected_df))
print(selected_df["class_name"].value_counts())

print("Source manifest:")
print(SOURCE_MANIFEST_PATH)

print("\nSelection is deterministic and balanced.")
print("No images were generated.")
print("No existing images were overwritten.")

In [ ]:
import os, gc, time, json, hashlib
from pathlib import Path
import pandas as pd
import torch
from PIL import Image

ROOT = Path("/content/drive/MyDrive/TwinCheck")
SDXL_ROOT = ROOT / "data/copies/vision/genimage_sdxl_img2img_500"

SOURCE_MANIFEST = SDXL_ROOT / "sdxl_500_source_manifest.csv"
MANIFEST_PATH = SDXL_ROOT / "sdxl_500_generation_manifest.csv"
OUTPUT_DIR = SDXL_ROOT / "images"
SUMMARY_PATH = SDXL_ROOT / "sdxl_generation_summary.json"

MODEL_ID = "stabilityai/stable-diffusion-xl-base-1.0"

SEED_BASE = 42000
IMAGE_MAX_SIZE = 512
NUM_STEPS = 20
STRENGTH = 0.25
GUIDANCE_SCALE = 5.0

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if "pipe" not in globals() or pipe is None:

    print("Pipeline not retained. Loading SDXL...")

    from diffusers import StableDiffusionXLImg2ImgPipeline

    pipe = StableDiffusionXLImg2ImgPipeline.from_pretrained(
        MODEL_ID,
        torch_dtype=DTYPE,
        variant="fp16" if DEVICE == "cuda" else None,
        use_safetensors=True
    )

    pipe.to(DEVICE)

else:
    print("Reusing the already loaded SDXL pipeline.")

try:
    pipe.vae.enable_slicing()
    print("VAE slicing: enabled")
except Exception as e:
    print("VAE slicing unavailable:", str(e)[:200])

try:
    pipe.vae.enable_tiling()
    print("VAE tiling: enabled")
except Exception as e:
    print("VAE tiling unavailable:", str(e)[:200])

try:
    pipe.unet.set_attention_slice("max")
    print("UNet attention slicing: enabled")
except Exception as e:
    print("Attention slicing unavailable:", str(e)[:200])

try:
    pipe.set_progress_bar_config(disable=False)
except Exception:
    pass

gc.collect()
if DEVICE == "cuda":
    torch.cuda.empty_cache()

if MANIFEST_PATH.exists():
    manifest = pd.read_csv(MANIFEST_PATH)
    print("Resuming from generation checkpoint.")
else:
    manifest = pd.read_csv(SOURCE_MANIFEST)

for col in ["sdxl_status", "sdxl_output_path", "sdxl_error"]:
    if col not in manifest.columns:
        manifest[col] = ""

manifest["sdxl_status"] = (
    manifest["sdxl_status"].astype("string").fillna("pending")
)

manifest["sdxl_output_path"] = (
    manifest["sdxl_output_path"].astype("string").fillna("")
)

manifest["sdxl_error"] = (
    manifest["sdxl_error"].astype("string").fillna("")
)

for col in [
    "sdxl_seed",
    "sdxl_steps",
    "sdxl_strength",
    "sdxl_guidance_scale"
]:
    if col not in manifest.columns:
        manifest[col] = pd.NA

assert len(manifest) == 500

def checkpoint():
    tmp = MANIFEST_PATH.with_suffix(".tmp.csv")
    manifest.to_csv(tmp, index=False)
    os.replace(tmp, MANIFEST_PATH)

def valid_img(path):
    try:
        with Image.open(path) as im:
            im.verify()
        with Image.open(path) as im:
            return im.width > 0 and im.height > 0
    except Exception:
        return False

def resize_image(im, max_size=512):
    im = im.convert("RGB")
    w, h = im.size
    scale = max_size / max(w, h)

    nw = max(64, int(w * scale))
    nh = max(64, int(h * scale))

    nw = (nw // 8) * 8
    nh = (nh // 8) * 8

    return im.resize((nw, nh), Image.Resampling.LANCZOS)

for i, row in manifest.iterrows():

    out = OUTPUT_DIR / f"{row['source_id']}.jpg"

    if out.exists() and valid_img(out):
        manifest.at[i, "sdxl_status"] = "completed"
        manifest.at[i, "sdxl_output_path"] = str(out)
        manifest.at[i, "sdxl_error"] = ""

    elif str(row["sdxl_status"]) == "completed":
        manifest.at[i, "sdxl_status"] = "pending"
        manifest.at[i, "sdxl_output_path"] = ""

checkpoint()

PROMPT = (
    "A realistic photographic image, natural visual appearance, "
    "realistic lighting and coherent textures, preserve the main "
    "objects, scene composition, and overall content of the input image."
)

NEGATIVE_PROMPT = (
    "painting, illustration, cartoon, anime, CGI, 3D render, "
    "artificial textures, distorted objects, blurry, low quality, "
    "oversaturated, changed scene, extra objects"
)

session_start = time.time()

completed_initial = int(
    (manifest["sdxl_status"] == "completed").sum()
)

print("\nCompleted already:", completed_initial)
print("Remaining:", 500 - completed_initial)

for idx, row in manifest.iterrows():

    if str(row["sdxl_status"]) == "completed":
        continue

    source_path = Path(row["normalized_export_path"])
    output_path = OUTPUT_DIR / f"{row['source_id']}.jpg"

    print(f"\n[{idx+1}/500] {row['class_name']}")

    try:

        if not source_path.exists():
            raise FileNotFoundError(source_path)

        if output_path.exists() and valid_img(output_path):
            manifest.at[idx, "sdxl_status"] = "completed"
            manifest.at[idx, "sdxl_output_path"] = str(output_path)
            checkpoint()
            continue

        with Image.open(source_path) as im:
            source_image = resize_image(im, IMAGE_MAX_SIZE)

        seed_hash = hashlib.sha256(
            str(row["source_id"]).encode()
        ).hexdigest()

        seed = SEED_BASE + (
            int(seed_hash[:8], 16) % 1_000_000
        )

        generator = torch.Generator(
            device=DEVICE
        ).manual_seed(seed)

        with torch.inference_mode():

            result = pipe(
                prompt=PROMPT,
                negative_prompt=NEGATIVE_PROMPT,
                image=source_image,
                strength=STRENGTH,
                num_inference_steps=NUM_STEPS,
                guidance_scale=GUIDANCE_SCALE,
                generator=generator
            )

        twin = result.images[0].convert("RGB")

        tmp_out = output_path.with_suffix(".tmp.jpg")

        twin.save(
            tmp_out,
            format="JPEG",
            quality=95,
            optimize=True
        )

        if not valid_img(tmp_out):
            raise RuntimeError("Output image validation failed.")

        os.replace(tmp_out, output_path)

        manifest.at[idx, "sdxl_status"] = "completed"
        manifest.at[idx, "sdxl_output_path"] = str(output_path)
        manifest.at[idx, "sdxl_error"] = ""
        manifest.at[idx, "sdxl_seed"] = seed
        manifest.at[idx, "sdxl_steps"] = NUM_STEPS
        manifest.at[idx, "sdxl_strength"] = STRENGTH
        manifest.at[idx, "sdxl_guidance_scale"] = GUIDANCE_SCALE

        checkpoint()

        print("SUCCESS:", output_path.name)

    except torch.cuda.OutOfMemoryError:

        manifest.at[idx, "sdxl_status"] = "failed"
        manifest.at[idx, "sdxl_error"] = "CUDA_OUT_OF_MEMORY"
        checkpoint()

        gc.collect()
        torch.cuda.empty_cache()

        print("GPU memory exhausted. Stopping safely.")
        break

    except Exception as e:

        manifest.at[idx, "sdxl_status"] = "failed"
        manifest.at[idx, "sdxl_error"] = repr(e)[:1000]
        checkpoint()

        print("ERROR:", repr(e))
        continue

    finally:
        gc.collect()
        if DEVICE == "cuda":
            torch.cuda.empty_cache()

checkpoint()

completed = int(
    (manifest["sdxl_status"] == "completed").sum()
)

failed = int(
    (manifest["sdxl_status"] == "failed").sum()
)

pending = int(
    (manifest["sdxl_status"] == "pending").sum()
)

summary = {
    "model": MODEL_ID,
    "total": len(manifest),
    "completed": completed,
    "failed": failed,
    "pending": pending,
    "steps": NUM_STEPS,
    "strength": STRENGTH,
    "guidance_scale": GUIDANCE_SCALE,
    "image_max_size": IMAGE_MAX_SIZE,
    "elapsed_seconds": round(time.time() - session_start, 2),
    "updated_at": time.strftime("%Y-%m-%d %H:%M:%S")
}

with open(SUMMARY_PATH, "w") as f:
    json.dump(summary, f, indent=2)

print("\n" + "=" * 70)
print("SDXL GENERATION SESSION FINISHED")
print("=" * 70)

print(json.dumps(summary, indent=2))
print("\nManifest:", MANIFEST_PATH)
print("Images:", OUTPUT_DIR)
print("Summary:", SUMMARY_PATH)

In [ ]:
import os, json, hashlib
from pathlib import Path
import pandas as pd
from PIL import Image

ROOT = Path("/content/drive/MyDrive/TwinCheck")
SDXL_ROOT = ROOT / "data/copies/vision/genimage_sdxl_img2img_500"

SOURCE_MANIFEST = SDXL_ROOT / "sdxl_500_source_manifest.csv"
GEN_MANIFEST = SDXL_ROOT / "sdxl_500_generation_manifest.csv"
SUMMARY_PATH = SDXL_ROOT / "sdxl_generation_summary.json"
OUTPUT_DIR = SDXL_ROOT / "images"

assert SOURCE_MANIFEST.exists(), "Source manifest missing!"
assert GEN_MANIFEST.exists(), "Generation manifest missing!"
assert SUMMARY_PATH.exists(), "Generation summary missing!"
assert OUTPUT_DIR.exists(), "Output image directory missing!"

source = pd.read_csv(SOURCE_MANIFEST)
gen = pd.read_csv(GEN_MANIFEST)

with open(SUMMARY_PATH, "r") as f:
    summary = json.load(f)

print("=" * 70)
print("SDXL GENERATION AUDIT")
print("=" * 70)

print("\nGeneration summary:")
print(json.dumps(summary, indent=2))

print("\n--- MANIFEST VALIDATION ---")

print("Source rows:", len(source))
print("Generation manifest rows:", len(gen))

assert len(source) == 500, "Source manifest does not contain 500 rows!"
assert len(gen) == 500, "Generation manifest does not contain 500 rows!"

assert "source_id" in gen.columns
assert "class_name" in gen.columns
assert "sdxl_status" in gen.columns
assert "sdxl_output_path" in gen.columns

print("Unique source IDs:", gen["source_id"].nunique())

duplicate_ids = gen["source_id"].duplicated().sum()
print("Duplicate source IDs:", duplicate_ids)

assert duplicate_ids == 0, "Duplicate source IDs found!"

print("\n--- CLASS DISTRIBUTION ---")

class_counts = gen["class_name"].value_counts()
print(class_counts)

assert class_counts.get("real", 0) == 250
assert class_counts.get("fake", 0) == 250

print("\n--- GENERATION STATUS ---")

status_counts = gen["sdxl_status"].value_counts(dropna=False)
print(status_counts)

completed = int((gen["sdxl_status"] == "completed").sum())
failed = int((gen["sdxl_status"] == "failed").sum())
pending = int((gen["sdxl_status"] == "pending").sum())

print("Completed:", completed)
print("Failed:", failed)
print("Pending:", pending)

assert completed == 500, "Not all images are marked completed!"
assert failed == 0, "Generation failures exist!"
assert pending == 0, "Pending images exist!"

print("\n--- IMAGE FILE VALIDATION ---")

missing_files = []
corrupt_files = []
non_jpeg_files = []
non_rgb_files = []
invalid_dimensions = []
wrong_paths = []

image_records = []

for _, row in gen.iterrows():

    source_id = str(row["source_id"])
    expected_path = OUTPUT_DIR / f"{source_id}.jpg"

    recorded_path = str(row["sdxl_output_path"])

    if not expected_path.exists():
        missing_files.append(source_id)
        continue

    if recorded_path and Path(recorded_path) != expected_path:
        wrong_paths.append(source_id)

    try:

        with Image.open(expected_path) as im:

            image_format = im.format
            image_mode = im.mode
            width, height = im.size

            im.verify()

        with Image.open(expected_path) as im:

            image_format = im.format
            image_mode = im.mode
            width, height = im.size

        if image_format != "JPEG":
            non_jpeg_files.append(source_id)

        if image_mode != "RGB":
            non_rgb_files.append(source_id)

        if width <= 0 or height <= 0:
            invalid_dimensions.append(source_id)

        sha256 = hashlib.sha256(
            expected_path.read_bytes()
        ).hexdigest()

        image_records.append({
            "source_id": source_id,
            "class_name": row["class_name"],
            "output_path": str(expected_path),
            "format": image_format,
            "mode": image_mode,
            "width": width,
            "height": height,
            "file_size_bytes": expected_path.stat().st_size,
            "sha256": sha256
        })

    except Exception as e:

        corrupt_files.append({
            "source_id": source_id,
            "error": repr(e)
        })

audit_df = pd.DataFrame(image_records)

if len(audit_df) > 0:

    duplicate_hashes = audit_df[
        audit_df["sha256"].duplicated(keep=False)
    ]

else:
    duplicate_hashes = pd.DataFrame()

print("\nMissing files:", len(missing_files))
print("Corrupt files:", len(corrupt_files))
print("Non-JPEG files:", len(non_jpeg_files))
print("Non-RGB files:", len(non_rgb_files))
print("Invalid dimensions:", len(invalid_dimensions))
print("Manifest path mismatches:", len(wrong_paths))
print("Duplicate image hashes:", len(duplicate_hashes))

if len(audit_df) > 0:

    print("\n--- IMAGE DIMENSIONS ---")

    print(
        audit_df.groupby(["width", "height"])
        .size()
        .sort_values(ascending=False)
    )

    print("\n--- FILE SIZE ---")

    print(
        audit_df["file_size_bytes"]
        .describe()
    )

AUDIT_CSV = SDXL_ROOT / "sdxl_generation_image_audit.csv"
AUDIT_JSON = SDXL_ROOT / "sdxl_generation_audit_summary.json"

audit_df.to_csv(AUDIT_CSV, index=False)

audit_summary = {
    "total_manifest_rows": len(gen),
    "unique_source_ids": int(gen["source_id"].nunique()),
    "completed": completed,
    "failed": failed,
    "pending": pending,
    "class_distribution": {
        str(k): int(v) for k, v in class_counts.items()
    },
    "valid_images": len(audit_df),
    "missing_files": len(missing_files),
    "corrupt_files": len(corrupt_files),
    "non_jpeg_files": len(non_jpeg_files),
    "non_rgb_files": len(non_rgb_files),
    "invalid_dimensions": len(invalid_dimensions),
    "manifest_path_mismatches": len(wrong_paths),
    "duplicate_image_hashes": int(len(duplicate_hashes)),
    "audit_pass": (
        len(gen) == 500
        and completed == 500
        and failed == 0
        and pending == 0
        and len(audit_df) == 500
        and len(missing_files) == 0
        and len(corrupt_files) == 0
        and len(non_jpeg_files) == 0
        and len(non_rgb_files) == 0
        and len(invalid_dimensions) == 0
        and len(wrong_paths) == 0
        and len(duplicate_hashes) == 0
    )
}

with open(AUDIT_JSON, "w") as f:
    json.dump(audit_summary, f, indent=2)

print("\n" + "=" * 70)
print("FINAL AUDIT RESULT")
print("=" * 70)

print(json.dumps(audit_summary, indent=2))

print("\nAudit CSV:", AUDIT_CSV)
print("Audit JSON:", AUDIT_JSON)

if audit_summary["audit_pass"]:
    print("\nPASS: All 500 SDXL outputs passed this audit.")
else:
    print("\nREVIEW REQUIRED: Some audit checks did not pass.")

In [ ]:
import os, gc, json, time, hashlib
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from PIL import Image
from tqdm.auto import tqdm
from transformers import AutoProcessor, AutoModel

ROOT = Path("/content/drive/MyDrive/TwinCheck")

SDXL_ROOT = ROOT / "data/copies/vision/genimage_sdxl_img2img_500"

GEN_MANIFEST_PATH = SDXL_ROOT / "sdxl_500_generation_manifest.csv"

SDXL_IMAGE_DIR = SDXL_ROOT / "images"

GENIMAGE_FEATURES = ROOT / "features/vision/genimage"

GENIMAGE_EMBEDDING_MANIFEST = (
    GENIMAGE_FEATURES
    / "f1_genimage_13000_embedding_input_manifest.csv"
)

ORIGINAL_SIGLIP_NORMALIZED = (
    GENIMAGE_FEATURES
    / "siglip2_so400m"
    / "embeddings_normalized.npy"
)

OUTPUT_DIR = SDXL_ROOT / "features/siglip2_so400m"

AUDIT_DIR = SDXL_ROOT / "audit/siglip2_similarity"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

TWIN_RAW_PATH = OUTPUT_DIR / "sdxl_twin_embeddings_raw.npy"

TWIN_NORMALIZED_PATH = (
    OUTPUT_DIR / "sdxl_twin_embeddings_normalized.npy"
)

ORIGINAL_SUBSET_PATH = (
    OUTPUT_DIR / "matched_original_embeddings_normalized.npy"
)

PAIR_MANIFEST_PATH = (
    OUTPUT_DIR / "sdxl_pair_embedding_manifest.csv"
)

SIMILARITY_PATH = (
    OUTPUT_DIR / "sdxl_original_twin_cosine_similarity.csv"
)

SUMMARY_PATH = (
    AUDIT_DIR / "sdxl_siglip2_similarity_summary.json"
)

CHECKPOINT_RAW_PATH = (
    OUTPUT_DIR / "embedding_checkpoint_raw.npy"
)

CHECKPOINT_STATUS_PATH = (
    OUTPUT_DIR / "embedding_checkpoint_status.csv"
)

CHECKPOINT_META_PATH = (
    OUTPUT_DIR / "embedding_checkpoint_metadata.json"
)

MODEL_ID = "google/siglip2-so400m-patch14-384"

N = 500
DIM = 1152
BATCH_SIZE = 4

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

DTYPE = torch.float16 if DEVICE.type == "cuda" else torch.float32

print("Device:", DEVICE)

if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

sdxl = pd.read_csv(GEN_MANIFEST_PATH).reset_index(drop=True)

original_manifest = pd.read_csv(
    GENIMAGE_EMBEDDING_MANIFEST
)

original_embeddings = np.load(
    ORIGINAL_SIGLIP_NORMALIZED,
    mmap_mode="r"
)

assert len(sdxl) == N
assert sdxl["source_id"].nunique() == N
assert (sdxl["sdxl_status"] == "completed").all()

assert original_embeddings.shape == (13000, DIM)

print("SDXL images:", len(sdxl))
print("Original embeddings:", original_embeddings.shape)

original_lookup = original_manifest.set_index(
    "normalized_md5"
)

pair_rows = []

for _, row in sdxl.iterrows():

    md5 = str(row["normalized_md5"])

    if md5 not in original_lookup.index:
        raise RuntimeError(f"Original embedding missing: {md5}")

    orig = original_lookup.loc[md5]

    if isinstance(orig, pd.DataFrame):
        raise RuntimeError(f"Ambiguous original match: {md5}")

    if str(orig["class_name"]) != str(row["class_name"]):
        raise RuntimeError(
            f"Class mismatch for {row['source_id']}"
        )

    pair_rows.append({
        "source_id": str(row["source_id"]),
        "class_name": str(row["class_name"]),
        "normalized_md5": md5,
        "original_embedding_row": int(orig["embedding_row"]),
        "original_image_path": str(orig["normalized_export_path"]),
        "sdxl_image_path": str(row["sdxl_output_path"]),
        "split": str(orig["split"])
    })

pairs = pd.DataFrame(pair_rows)

assert len(pairs) == N
assert pairs["source_id"].nunique() == N
assert pairs["original_embedding_row"].nunique() == N

print("Original-to-SDXL matches:", len(pairs), "PASS")

for p in pairs["sdxl_image_path"]:

    if not Path(p).exists():
        raise FileNotFoundError(p)

print("All SDXL twin files exist.")

fingerprint = hashlib.sha256(
    "\n".join(pairs["source_id"].astype(str)).encode()
).hexdigest()

checkpoint_exists = all([
    CHECKPOINT_RAW_PATH.exists(),
    CHECKPOINT_STATUS_PATH.exists(),
    CHECKPOINT_META_PATH.exists()
])

if checkpoint_exists:

    meta = json.loads(
        CHECKPOINT_META_PATH.read_text()
    )

    if meta["fingerprint"] != fingerprint:
        raise RuntimeError(
            "Checkpoint fingerprint mismatch. "
            "Stopping without overwriting checkpoint."
        )

    embeddings = np.load(CHECKPOINT_RAW_PATH)

    status = pd.read_csv(CHECKPOINT_STATUS_PATH)

    assert embeddings.shape == (N, DIM)
    assert len(status) == N
    assert status["source_id"].astype(str).tolist() == (
        pairs["source_id"].astype(str).tolist()
    )

    print("Existing checkpoint reused.")

else:

    embeddings = np.full(
        (N, DIM),
        np.nan,
        dtype=np.float32
    )

    status = pd.DataFrame({
        "source_id": pairs["source_id"],
        "status": ["pending"] * N,
        "error": [""] * N
    })

    np.save(CHECKPOINT_RAW_PATH, embeddings)

    status.to_csv(
        CHECKPOINT_STATUS_PATH,
        index=False
    )

    CHECKPOINT_META_PATH.write_text(
        json.dumps({
            "fingerprint": fingerprint,
            "model": MODEL_ID,
            "count": N,
            "dimension": DIM
        }, indent=2)
    )

    print("Checkpoint initialized.")

def save_checkpoint():

    tmp = CHECKPOINT_RAW_PATH.with_suffix(".tmp.npy")

    np.save(tmp, embeddings)

    os.replace(tmp, CHECKPOINT_RAW_PATH)

    tmp_csv = CHECKPOINT_STATUS_PATH.with_suffix(".tmp.csv")

    status.to_csv(tmp_csv, index=False)

    os.replace(tmp_csv, CHECKPOINT_STATUS_PATH)

gc.collect()

if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

print("\nLoading SigLIP2...")

processor = AutoProcessor.from_pretrained(MODEL_ID)

model = AutoModel.from_pretrained(
    MODEL_ID,
    dtype=DTYPE
).to(DEVICE)

model.eval()

def encode_paths(paths):

    images = []

    for path in paths:

        with Image.open(path) as im:
            images.append(im.convert("RGB").copy())

    inputs = processor(
        images=images,
        return_tensors="pt"
    )

    inputs = {
        k: v.to(DEVICE)
        for k, v in inputs.items()
        if isinstance(v, torch.Tensor)
    }

    with torch.inference_mode():

        if DEVICE.type == "cuda":

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16
            ):

                output = model.get_image_features(**inputs)

        else:

            output = model.get_image_features(**inputs)

    if isinstance(output, torch.Tensor):

        features = output

    else:

        features = None

        for attr in [
            "pooler_output",
            "image_embeds",
            "embeddings"
        ]:

            candidate = getattr(output, attr, None)

            if isinstance(candidate, torch.Tensor):

                features = candidate
                break

        if features is None:

            try:

                candidate = output[0]

                if isinstance(candidate, torch.Tensor):
                    features = candidate

            except Exception:
                pass

    if features is None:

        raise RuntimeError(
            "Could not locate tensor image embeddings "
            "inside model output."
        )

    features = features.float()

    if features.ndim == 3:

        features = features[:, 0, :]

    if features.ndim != 2:

        raise RuntimeError(
            f"Unexpected embedding shape: {features.shape}"
        )

    if features.shape[1] != DIM:

        raise RuntimeError(
            f"Unexpected embedding dimension: {features.shape}. "
            f"Expected {DIM}."
        )

    features = features.cpu().numpy()

    del inputs, output

    return features

first_pending = np.where(
    status["status"].to_numpy() != "completed"
)[0]

if len(first_pending) > 0:

    test_idx = int(first_pending[0])

    print("\nTesting corrected embedding function...")

    test_feature = encode_paths([
        pairs.loc[test_idx, "sdxl_image_path"]
    ])

    print("Test embedding shape:", test_feature.shape)

    assert test_feature.shape == (1, DIM)
    assert np.isfinite(test_feature).all()

    print("TEST PASS.")

start_time = time.time()

pending = [
    i for i in range(N)
    if status.loc[i, "status"] != "completed"
]

print("\nAlready completed:", N - len(pending))
print("Remaining:", len(pending))

for start in tqdm(
    range(0, len(pending), BATCH_SIZE),
    desc="Extracting SDXL embeddings"
):

    indices = pending[start:start + BATCH_SIZE]

    paths = [
        pairs.loc[i, "sdxl_image_path"]
        for i in indices
    ]

    try:

        feats = encode_paths(paths)

        for j, idx in enumerate(indices):

            embeddings[idx] = feats[j]

            status.loc[idx, "status"] = "completed"
            status.loc[idx, "error"] = ""

    except Exception as batch_error:

        print("\nBatch error; retrying individually:",
              repr(batch_error))

        for idx in indices:

            path = pairs.loc[idx, "sdxl_image_path"]

            try:

                feats = encode_paths([path])

                embeddings[idx] = feats[0]

                status.loc[idx, "status"] = "completed"
                status.loc[idx, "error"] = ""

            except Exception as e:

                status.loc[idx, "status"] = "failed"
                status.loc[idx, "error"] = repr(e)[:1000]

    save_checkpoint()

    gc.collect()

    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

completed = int(
    (status["status"] == "completed").sum()
)

failed = int(
    (status["status"] == "failed").sum()
)

pending_count = int(
    (status["status"] == "pending").sum()
)

print("\nCompleted:", completed)
print("Failed:", failed)
print("Pending:", pending_count)

if completed != N:

    print(
        "\nExtraction incomplete. "
        "Checkpoint saved; rerun this cell to resume."
    )

    raise RuntimeError(
        f"Only {completed}/{N} embeddings completed."
    )

assert np.isfinite(embeddings).all()

twin_raw = embeddings.astype(np.float32)

norms = np.linalg.norm(
    twin_raw,
    axis=1,
    keepdims=True
)

twin_normalized = twin_raw / np.maximum(
    norms,
    1e-12
)

assert twin_raw.shape == (N, DIM)
assert twin_normalized.shape == (N, DIM)
assert np.isfinite(twin_normalized).all()

assert np.allclose(
    np.linalg.norm(twin_normalized, axis=1),
    1.0,
    atol=1e-4
)

original_rows = pairs[
    "original_embedding_row"
].to_numpy(dtype=np.int64)

original_subset = np.asarray(
    original_embeddings[original_rows],
    dtype=np.float32
)

assert original_subset.shape == (N, DIM)

similarities = np.sum(
    original_subset * twin_normalized,
    axis=1
)

assert len(similarities) == N
assert np.isfinite(similarities).all()

pairs["cosine_similarity_siglip2"] = similarities

pairs["twin_embedding_row"] = np.arange(N)

def safe_save_array(array, path):

    if path.exists():

        existing = np.load(path, mmap_mode="r")

        if existing.shape != array.shape:

            raise RuntimeError(
                f"Existing output shape conflict: {path}"
            )

        print("Compatible existing array preserved:", path)

    else:

        tmp = path.with_suffix(".tmp.npy")

        np.save(tmp, array)

        os.replace(tmp, path)

        print("Saved:", path)

safe_save_array(twin_raw, TWIN_RAW_PATH)

safe_save_array(
    twin_normalized,
    TWIN_NORMALIZED_PATH
)

safe_save_array(
    original_subset,
    ORIGINAL_SUBSET_PATH
)

if not PAIR_MANIFEST_PATH.exists():

    pairs.to_csv(
        PAIR_MANIFEST_PATH,
        index=False
    )

else:

    print("Existing pair manifest preserved.")

similarity_df = pairs.sort_values(
    "cosine_similarity_siglip2"
).reset_index(drop=True)

if not SIMILARITY_PATH.exists():

    similarity_df.to_csv(
        SIMILARITY_PATH,
        index=False
    )

else:

    print("Existing similarity CSV preserved.")

sim = similarities.astype(np.float64)

summary = {
    "dataset": "GenImage",
    "experiment": "SDXL Img2Img twins",
    "embedding_model": MODEL_ID,
    "total_pairs": N,
    "embedding_dimension": DIM,
    "completed": completed,
    "failed": failed,
    "pending": pending_count,
    "overall_similarity": {
        "mean": float(sim.mean()),
        "median": float(np.median(sim)),
        "std": float(sim.std()),
        "min": float(sim.min()),
        "max": float(sim.max()),
        "q05": float(np.quantile(sim, 0.05)),
        "q25": float(np.quantile(sim, 0.25)),
        "q75": float(np.quantile(sim, 0.75)),
        "q95": float(np.quantile(sim, 0.95))
    },
    "class_similarity": {},
    "elapsed_seconds": round(time.time() - start_time, 2)
}

for cls in ["real", "fake"]:

    mask = pairs["class_name"].astype(str) == cls

    values = sim[mask.to_numpy()]

    summary["class_similarity"][cls] = {
        "count": int(mask.sum()),
        "mean": float(values.mean()),
        "median": float(np.median(values)),
        "std": float(values.std()),
        "min": float(values.min()),
        "max": float(values.max())
    }

SUMMARY_PATH.write_text(
    json.dumps(summary, indent=2)
)

print("\n" + "=" * 80)
print("SDXL SIGLIP2 EMBEDDINGS + SIMILARITY COMPLETE")
print("=" * 80)

print("\nEmbedding shapes:")
print("Twin raw:", twin_raw.shape)
print("Twin normalized:", twin_normalized.shape)
print("Original matched:", original_subset.shape)

print("\nOverall cosine similarity:")
print(json.dumps(
    summary["overall_similarity"],
    indent=2
))

print("\nClass-wise similarity:")
print(json.dumps(
    summary["class_similarity"],
    indent=2
))

print("\n10 lowest similarity pairs:")
display(
    similarity_df.head(10)[
        [
            "source_id",
            "class_name",
            "cosine_similarity_siglip2"
        ]
    ]
)

print("\n10 highest similarity pairs:")
display(
    similarity_df.tail(10)[
        [
            "source_id",
            "class_name",
            "cosine_similarity_siglip2"
        ]
    ]
)

print("\nArtifacts:")
print(TWIN_RAW_PATH)
print(TWIN_NORMALIZED_PATH)
print(ORIGINAL_SUBSET_PATH)
print(PAIR_MANIFEST_PATH)
print(SIMILARITY_PATH)
print(SUMMARY_PATH)

print("\nOriginal GenImage embeddings were reused.")
print("SDXL images were NOT regenerated.")
print("================================================")

In [ ]:
import pandas as pd
import numpy as np

path = "/content/drive/MyDrive/TwinCheck/data/copies/vision/genimage_sdxl_img2img_500/features/siglip2_so400m/sdxl_original_twin_cosine_similarity.csv"

df = pd.read_csv(path)

print("Total pairs:", len(df))
print("\nColumns:", df.columns.tolist())

sim_col = next(c for c in df.columns if "similarity" in c.lower())
class_col = next(c for c in df.columns if "class" in c.lower())

thresholds = [0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90, 0.92]

rows = []

for t in thresholds:
    selected = df[df[sim_col] >= t]

    rows.append({
        "threshold": t,
        "total_selected": len(selected),
        "real": (selected[class_col].astype(str).str.lower() == "real").sum(),
        "fake": (selected[class_col].astype(str).str.lower() == "fake").sum(),
        "discarded": len(df) - len(selected)
    })

result = pd.DataFrame(rows)

print("\nTHRESHOLD ANALYSIS")
display(result)

print("\nSimilarity distribution by class:")
display(df.groupby(class_col)[sim_col].describe())

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from PIL import Image
import json

BASE = Path("/content/drive/MyDrive/TwinCheck/data/copies/vision/genimage_sdxl_img2img_500")

SIM_PATH = BASE / "features/siglip2_so400m/sdxl_original_twin_cosine_similarity.csv"

OUT = BASE / "final_selected_similarity_070_jpeg85"
REAL_DIR = OUT / "real"
FAKE_DIR = OUT / "fake"

REAL_DIR.mkdir(parents=True, exist_ok=True)
FAKE_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(SIM_PATH)

SIM_COL = "cosine_similarity_siglip2"
THRESHOLD = 0.70

selected = df[df[SIM_COL] >= THRESHOLD].copy()
selected = selected.reset_index(drop=True)

print("Selected pairs:", len(selected))
print(selected["class_name"].value_counts())

final_paths = []
conversion_errors = []

for i, row in selected.iterrows():

    source = Path(row["sdxl_image_path"])

    class_name = str(row["class_name"]).lower()

    if class_name == "real":
        destination_dir = REAL_DIR
    elif class_name == "fake":
        destination_dir = FAKE_DIR
    else:
        conversion_errors.append({
            "source_id": row["source_id"],
            "error": f"Unknown class: {class_name}"
        })
        final_paths.append(None)
        continue

    destination = destination_dir / f'{row["source_id"]}.jpg'

    try:
        with Image.open(source) as im:
            im = im.convert("RGB")
            im.save(destination, format="JPEG", quality=85, optimize=True)

        final_paths.append(str(destination))

    except Exception as e:
        final_paths.append(None)
        conversion_errors.append({
            "source_id": row["source_id"],
            "error": str(e)
        })

selected["final_twin_path_jpeg85"] = final_paths

missing = []
corrupt = []
wrong_format = []
wrong_dimensions = []

for p in selected["final_twin_path_jpeg85"]:

    if p is None or not Path(p).exists():
        missing.append(p)
        continue

    try:
        with Image.open(p) as im:

            if im.format != "JPEG":
                wrong_format.append(p)

            if im.size != (512, 512):
                wrong_dimensions.append((p, im.size))

            im.verify()

    except Exception as e:
        corrupt.append((p, str(e)))

manifest_path = OUT / "final_sdxl_twin_manifest_similarity070_jpeg85.csv"
selected.to_csv(manifest_path, index=False)

summary = {
    "pipeline": "SDXL Img2Img GenImage twins",
    "similarity_model": "SigLIP2",
    "selection_threshold": THRESHOLD,
    "selected_pairs": len(selected),
    "real": int((selected["class_name"].str.lower() == "real").sum()),
    "fake": int((selected["class_name"].str.lower() == "fake").sum()),
    "discarded_pairs": int(len(df) - len(selected)),
    "jpeg_quality": 85,
    "expected_dimensions": [512, 512],
    "missing_files": len(missing),
    "corrupt_files": len(corrupt),
    "wrong_format": len(wrong_format),
    "wrong_dimensions": len(wrong_dimensions),
    "conversion_errors": len(conversion_errors),
    "manifest": str(manifest_path),
    "dataset_directory": str(OUT),
    "audit_status": (
        "PASS"
        if len(missing) == 0
        and len(corrupt) == 0
        and len(wrong_format) == 0
        and len(wrong_dimensions) == 0
        and len(conversion_errors) == 0
        else "FAIL"
    )
}

summary_path = OUT / "final_dataset_audit_summary.json"

with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2)

print("\n" + "="*65)
print("FINAL SDXL TWIN DATASET AUDIT")
print("="*65)

print("Selected pairs:", summary["selected_pairs"])
print("Real:", summary["real"])
print("Fake:", summary["fake"])
print("Discarded:", summary["discarded_pairs"])
print("JPEG quality:", summary["jpeg_quality"])

print("\nVerification:")
print("Missing:", summary["missing_files"])
print("Corrupt:", summary["corrupt_files"])
print("Wrong format:", summary["wrong_format"])
print("Wrong dimensions:", summary["wrong_dimensions"])
print("Conversion errors:", summary["conversion_errors"])

print("\nAUDIT STATUS:", summary["audit_status"])

print("\nManifest:", manifest_path)
print("Summary:", summary_path)
print("Dataset folder:", OUT)

print("="*65)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import zipfile
import os

ROOT = Path("/content/drive/MyDrive/TwinCheck")

ZIP_PATH = ROOT / "F2_GITHUB_READY_RESULTS.zip"

SEARCH_DIRS = [
    ROOT / "data/processed/vision",
    ROOT / "data/copies/vision",
    ROOT / "features/vision",
    ROOT / "audit",
]

ALLOWED_EXTENSIONS = {
    ".csv",
    ".json",
    ".txt",
    ".md",
    ".xlsx",
}

INCLUDE_KEYWORDS = [
    "manifest",
    "audit",
    "summary",
    "similarity",
    "cosine",
    "eligible",
    "final",
    "selected",
    "fingerprint",
    "verification",
    "integrity",
    "report",
    "statistics",
    "stats",
]

EXCLUDE_KEYWORDS = [
    "checkpoint",
    "cache",
    "temporary",
    "temp",
    "debug",
    "log",
    "backup",
    "old",
    "intermediate",
]

selected_files = set()

for folder in SEARCH_DIRS:

    if not folder.exists():
        continue

    for file in folder.rglob("*"):

        if not file.is_file():
            continue

        if file.suffix.lower() not in ALLOWED_EXTENSIONS:
            continue

        name_lower = file.name.lower()

        if any(keyword in name_lower for keyword in EXCLUDE_KEYWORDS):
            continue

        if any(keyword in name_lower for keyword in INCLUDE_KEYWORDS):
            selected_files.add(file)

selected_files = sorted(selected_files)

print("=" * 70)
print("F2 GITHUB-READY PACKAGE")
print("=" * 70)

print(f"Selected files: {len(selected_files)}")

with zipfile.ZipFile(
    ZIP_PATH,
    "w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=6
) as zipf:

    for file in selected_files:

        relative_path = file.relative_to(ROOT)

        zipf.write(file, relative_path)

with zipfile.ZipFile(ZIP_PATH, "r") as zipf:

    contents = zipf.namelist()

    bad_file = zipf.testzip()

size_mb = ZIP_PATH.stat().st_size / (1024 * 1024)

print("\n" + "=" * 70)
print("PACKAGE CONTENTS")
print("=" * 70)

for item in contents:
    print(item)

print("\n" + "=" * 70)
print("FINAL STATUS")
print("=" * 70)

print(f"Files packaged: {len(contents)}")
print(f"ZIP size: {size_mb:.2f} MB")
print(f"ZIP integrity: {'PASS' if bad_file is None else 'FAIL'}")
print(f"ZIP path: {ZIP_PATH}")

print("=" * 70)